# Care Finder: the right facility for a service, with evidence

**CMU Datathon, Fall 2026 — Virtue Foundation challenge.** One notebook holding the project's whole pipeline code, in run order.

**The problem.** A community health worker, an NGO coordinator or a district planner asks: *where is the best place to get this service (dialysis, emergency C-section, NICU, cardiac surgery, MRI, burn care...) from here?* The Virtue Foundation dataset describes 10,077 Indian facilities, but it was built by scraping websites and merging them with a language model, so every field is a claim, not a fact. No column says "offers dialysis": the evidence is spread over about 451,000 short sentences, departments are not services (2,153 facilities list nephrology), some records mix text from several hospitals, and the state field holds 254 values for India's 37 states and territories.

**WHY a facility is recommended.** Every result quotes the facility's own sentences that claim the service, says how strong they are (a bed or machine count, a plain statement, or only a hint), and carries a confidence tier: *Confirmed*, *Likely*, *Possible* or *Unverified*. Warnings say when the record may describe a different hospital, when the service is unusual for that kind of facility, or when the location is uncertain. Nothing is shown as fact without a quoted sentence behind it.

**HOW we decide.** A language model reads the text once, in advance: it types each facility (step 3), tags every distinct sentence with the services it proves and how strongly (step 4), and checks which sentences belong to a different organisation (step 5). Hand-written lookups then turn evidence, department backing and facility-type fit into a tier with plain SQL rules (step 6). At query time the app sorts by **tier first, then distance**: a confirmed hospital further away ranks above a closer unverified one, because sending a patient to a place that does not offer the service costs more than a longer trip. Blind web checks (step 8) test the tiers: every Confirmed facility we could check offers the service (30 of 30 across three services).

**Safety switches.** With `REBUILD = False` and `RUN_LLM = False` (the default) this notebook only **reads** the existing tables and shows counts and examples per step. The live app reads the same tables, so rebuilding is behind explicit switches.

### Pipeline: tables in and out per step

| step | reads | writes (all in `workspace.vf`) | LLM |
|---|---|---|---|
| 1 Clean base | share `facilities` | `facilities_clean`, `facility_sentences`, `distinct_sentences` | no |
| 2 Location | share `facilities`, `india_post_pincode_directory` | `haversine_km()`, `pin_point`, `city_point`, `facility_location` | no |
| 3 Facility type | `facilities_clean` | `facility_type` (+ `facility_type_single_raw/_status/_prompts/_runs`) | yes |
| 4 Sentence tagging | `distinct_sentences`, `facility_sentences`, `facility_type` | `sentence_tags` (+ `sentence_tags_raw`, `st_queue`, `st_type_sets`, `st_chunk_members`, `pa_prompt`, `retag_v2_*`, `sentence_tags_backup_v1`) | yes |
| 5 Record ownership audit | `facilities_clean`, `facility_sentences` | `record_audit`, `sentence_foreign_flag` (+ `ra_*`) | yes |
| 6 Lookups and verdict | step 1-5 tables | `service_department_map`, `service_type_fit`, `service_terms`, **`facility_service`**, `build_stats` | no |
| 7 NFHS district context | share `nfhs_5_district_health_indicators`, `india_post_pincode_directory`, `facilities_clean` | `nfhs_district`, `district_alias`, `pin_district`, `facility_district`, `service_indicator_map` | no |
| 8 Blind validation | `validation_dialysis`, `validation_wave3` (hand-checked) | none | no |
| 9 The app | `facility_service`, `facility_location`, `facilities_clean`, `pin_point` | none (read-only) | one call per typed request |

Model for every LLM step: `databricks-meta-llama-3-3-70b-instruct` through `ai_query`, temperature 0, every answer validated against an allowed set before it is stored.

In [0]:
# ---- Safety switches -------------------------------------------------------------------------
# With both False the notebook only READS existing tables. The live app reads these tables.
REBUILD = False   # True: re-create the workspace.vf tables step by step (overwrites what the app reads)
RUN_LLM = False   # True: allow ai_query / model-serving calls (steps 3, 4, 5 and the app matcher)

import json
import os
import re
import sys
import tempfile
import time
from pathlib import Path

SHARE = "databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset"  # read-only Delta Share
VF = "workspace.vf"                                                                 # our writable schema
MODEL = "databricks-meta-llama-3-3-70b-instruct"
ARAVIND = "b8a5401f-42f1-422a-8cd9-686a15b4cb76"   # demo record: Aravind Eye Hospital, Hyderabad
LOG_ROOT = Path(tempfile.gettempdir()) / "vf_logs"   # job logs (the scripts wrote these under reports/)
LOG_ROOT.mkdir(parents=True, exist_ok=True)
print(f"REBUILD={REBUILD} RUN_LLM={RUN_LLM}")

REBUILD=False RUN_LLM=False


**Execution adapter.** The project scripts ran on the SQL warehouse through `databricks-sql-connector` (`conn.cursor().execute(...)`). Here the same calls go to `spark.sql` through a small cursor-shaped adapter, so the copied code runs unchanged. The adapter is also the safety net: any statement that is not a read raises unless `REBUILD` is on, and any `ai_query` raises unless `RUN_LLM` is on.

In [0]:
_READ_START = re.compile(r"^\s*(SELECT|WITH|SHOW|DESCRIBE|DESC|EXPLAIN|VALUES)\b", re.I)


def _strip_leading_comments(stmt: str) -> str:
    s = stmt.lstrip()
    while s.startswith("--") or s.startswith("("):
        if s.startswith("("):
            s = s[1:].lstrip()
        else:
            s = s.split("\n", 1)[1].lstrip() if "\n" in s else ""
    return s


def guard(stmt: str):
    """Refuse writes unless REBUILD, and ai_query unless RUN_LLM."""
    if re.search(r"\bai_query\s*\(", stmt, re.I) and not RUN_LLM:
        raise PermissionError("ai_query blocked: RUN_LLM is False")
    if not _READ_START.match(_strip_leading_comments(stmt)) and not REBUILD:
        raise PermissionError("write blocked: REBUILD is False -> " + stmt.strip()[:80])


class _Cursor:
    """The subset of the DB-API cursor the scripts use, on top of spark.sql."""
    description = None

    def execute(self, stmt, params=None):
        guard(stmt)
        self._df = spark.sql(stmt, args=params) if params else spark.sql(stmt)
        self.description = [(f.name, None, None, None, None, None, None) for f in self._df.schema.fields]

    def fetchall(self):
        return [tuple(r) for r in self._df.collect()]

    def fetchone(self):
        rows = self._df.limit(1).collect()
        return tuple(rows[0]) if rows else None

    def close(self):
        pass

    def __enter__(self):
        return self

    def __exit__(self, *a):
        return False


class _Conn:
    def cursor(self):
        return _Cursor()

    def close(self):
        pass

    def __enter__(self):
        return self

    def __exit__(self, *a):
        return False


def connect():
    """Replaces databricks.sql.connect(server_hostname=..., http_path=..., access_token=token())."""
    return _Conn()


def sql_exec(stmt, params=None):
    c = _Cursor()
    c.execute(stmt, params)
    return c.fetchall()

**Shared names** (copied from the app's `config.py`, the single source for table names, location constants and readable service names; the verdict build in step 6 and the app in step 9 both use them).

In [0]:
SRC = SHARE
TGT = VF
T_FACILITIES = f"{SRC}.facilities"
T_PINCODE    = f"{SRC}.india_post_pincode_directory"
T_PIN_POINT  = f"{TGT}.pin_point"
T_CITY_POINT = f"{TGT}.city_point"
T_FAC_LOC    = f"{TGT}.facility_location"
F_HAVERSINE  = f"{TGT}.haversine_km"

# --- constants ------------------------------------------------------------
NEAR_KM             = 50.0
PIN_TIGHT_KM        = 25.0
PREFIX_OUTLIER_KM   = 100.0
MIN_CITY_FACILITIES = 5
LAT_MIN, LAT_MAX    = 6.0, 38.0
LON_MIN, LON_MAX    = 68.0, 98.0

def subs() -> dict:
    """Values available to sql/*.sql as {placeholders}."""
    return {k: v for k, v in globals().items() if k.isupper()}

# --- service search (contract table filled later by the verdict job) -------
T_FAC_CLEAN      = f"{TGT}.facilities_clean"
T_SERVICE        = f"{TGT}.facility_service"         # real verdicts; empty until filled
T_BUILD_STATS    = f"{TGT}.build_stats"              # funnel numbers, rewritten each build
TIER_ORDER       = ["Confirmed", "Likely", "Possible", "Unverified"]
# The 8 services whose tiers were checked first (keyword-tier services). Other
# services show a "results still loading" note while tagging is incomplete.
APP_SERVICES     = ["dialysis", "caesarean_section", "nicu", "cardiac_surgery", "mri",
                    "burn_care", "icu_critical_care", "trauma_emergency"]
N_DISTINCT_SENTENCES = 271_262   # workspace.vf.distinct_sentences rows (coverage denominator)
HIDDEN_SERVICES  = {"other_clinical"}
WEAK_LOCATION    = ("suspect", "approximate", "unverified")
# Shown first in the dropdown, ONLY under "Common searches" (not repeated in their group).
COMMON_SEARCHES  = ["dialysis", "caesarean_section", "nicu", "cardiac_surgery", "mri", "burn_care"]
DEFAULT_SERVICE  = "dialysis"

# Dropdown: group -> [(service label, readable name)], in display order.
# Every label in facility_service except HIDDEN_SERVICES is offered as "<Group> / <Name>";
# a label missing here falls into "Other specialties" under a generated name.
SERVICE_GROUPS = {
    "Emergency and critical care": [
        ("trauma_emergency", "Emergency and trauma"),
        ("icu_critical_care", "ICU / critical care"),
        ("ventilator_support", "Ventilator support"),
        ("burn_care", "Burn care"),
        ("stroke_care", "Stroke care"),
        ("ambulance", "Ambulance"),
        ("blood_bank", "Blood bank"),
    ],
    "Maternity and newborn": [
        ("caesarean_section", "C-section"),
        ("normal_delivery", "Normal delivery"),
        ("obstetrics_maternity", "Maternity / obstetrics"),
        ("high_risk_obstetrics", "High-risk pregnancy care"),
        ("nicu", "NICU (newborn intensive care)"),
        ("picu", "PICU (children's intensive care)"),
        ("pediatrics", "Paediatrics"),
        ("infertility_ivf", "Infertility / IVF"),
        ("gynecology", "Gynaecology (non-pregnancy)"),
    ],
    "Surgery": [
        ("general_surgery", "General surgery"),
        ("cardiac_surgery", "Cardiac surgery"),
        ("interventional_cardiology", "Interventional cardiology (angioplasty)"),
        ("neurosurgery", "Neurosurgery"),
        ("orthopedic_surgery", "Orthopaedic surgery"),
        ("joint_replacement", "Joint replacement"),
        ("spine_surgery", "Spine surgery"),
        ("eye_surgery", "Eye surgery"),
        ("transplant", "Organ transplant"),
        ("cosmetic_aesthetic", "Cosmetic / aesthetic surgery"),
        ("proctology", "Piles, fistula and fissure treatment"),
        ("vascular_surgery", "Vascular surgery and varicose veins"),
    ],
    "Imaging and diagnostics": [
        ("mri", "MRI scan"),
        ("ct_scan", "CT scan"),
        ("xray", "X-ray"),
        ("ultrasound", "Ultrasound"),
        ("mammography", "Mammography"),
        ("nuclear_imaging", "Nuclear imaging"),
        ("gi_endoscopy", "GI endoscopy"),
        ("laboratory_pathology", "Laboratory / pathology"),
        ("cardiac_diagnostics", "Heart tests (echo, ECG, TMT)"),
    ],
    "Cancer": [
        ("cancer_surgery", "Cancer surgery"),
        ("chemotherapy", "Chemotherapy"),
        ("radiotherapy", "Radiotherapy"),
    ],
    "Other specialties": [
        ("dialysis", "Dialysis"),
        ("nephrology_renal", "Nephrology (kidney care)"),
        ("dental", "Dental"),
        ("ent", "ENT (ear, nose, throat)"),
        ("ophthalmology_medical", "Eye care (medical)"),
        ("dermatology", "Dermatology"),
        ("gastroenterology", "Gastroenterology"),
        ("pulmonology_respiratory", "Pulmonology / respiratory"),
        ("mental_health", "Mental health"),
        ("physiotherapy_rehab", "Physiotherapy / rehabilitation"),
        ("prosthetics_orthotics", "Prosthetics / orthotics"),
        ("outpatient_opd", "Outpatient (OPD)"),
        ("urology", "Urology and kidney stone treatment"),
        ("diabetes_endocrinology", "Diabetes, thyroid and hormone care"),
        ("health_checkup_vaccination", "Health check-ups and vaccination"),
        ("neurology", "Neurology (non-surgical brain and nerve care)"),
        ("pain_management", "Pain management"),
        ("cardiology_medical", "Cardiology (heart care, no surgery)"),
    ],
    "Traditional medicine (AYUSH)": [
        ("ayush_traditional_medicine", "Ayurveda, homeopathy and other traditional medicine"),
    ],
}
# Readable facility types, with article, for sentences ("... is unusual for a dental clinic").
FACILITY_TYPE_NAME = {
    "multispecialty_hospital": "a multispecialty hospital",
    "dental": "a dental clinic",
    "general_hospital_or_nursing_home": "a general hospital or nursing home",
    "diagnostic_imaging_lab": "a diagnostic or imaging centre",
    "eye": "an eye hospital",
    "ayush_alternative": "an AYUSH (traditional medicine) centre",
    "maternity_women_children": "a maternity and children's hospital",
    "general_clinic_polyclinic": "a general clinic or polyclinic",
    "medical_college_teaching_hospital": "a medical college hospital",
    "other_single_specialty": "a single-specialty clinic",
    "ivf_fertility": "a fertility (IVF) clinic",
    "orthopedic_spine": "an orthopaedic and spine hospital",
    "skin_hair_cosmetic": "a skin, hair or cosmetic clinic",
    "ent_hearing_speech": "an ENT, hearing or speech clinic",
    "pediatric": "a children's hospital",
    "government_health_centre": "a government health centre",
    "cardiac": "a heart hospital",
    "cancer": "a cancer hospital",
    "kidney_urology": "a kidney and urology hospital",
    "mental_health_deaddiction": "a mental health or de-addiction centre",
    "physiotherapy_rehab": "a physiotherapy or rehabilitation centre",
    "neuro": "a neurology and neurosurgery hospital",
    "gastro_liver": "a gastro and liver hospital",
    "not_a_care_facility": "an organisation that is not a care facility",
    "home_care_or_ambulance": "a home-care or ambulance service",
    "unknown": "a facility of unknown type",
}


def facility_type_name(ft, with_article=True):
    n = FACILITY_TYPE_NAME.get(ft or "unknown", f"a {(ft or 'unknown').replace('_', ' ')} facility")
    return n if with_article else n.split(" ", 1)[1]


SERVICE_NAME  = {s: n for g in SERVICE_GROUPS.values() for s, n in g}
SERVICE_GROUP = {s: grp for grp, g in SERVICE_GROUPS.items() for s, _ in g}


def service_name(label: str) -> str:
    return SERVICE_NAME.get(label, label.replace("_", " ").capitalize())

## Step 1 — Clean base

**What.** Mechanical cleaning, no LLM. Removes the 11 exact duplicate rows (10,088 -> 10,077 facilities), flags 88 records whose columns shifted during scraping, splits `procedure` / `capability` / `equipment` into one row per sentence, marks placeholders (*"no procedures listed in the provided content"*, four conservative rules) and sentences repeated inside the same record, and stores each distinct sentence once so it is read by the model only once.

**Why.** Every later step works at sentence level; duplicates would fan out joins, shifted rows would put text in the wrong fields, and repeats would be counted twice as evidence.

**Reads** share `facilities`. **Writes** `facilities_clean` (10,077 rows), `facility_sentences` (451,110), `distinct_sentences` (271,262). Source: `scripts/build_clean_base.py`.

Measured (`reports/clean_base_build.md`): `is_shifted` 88, placeholders 1,000, repeats 26,422; postcode valid for 9,909 facilities.

In [0]:
# scripts/build_clean_base.py
SRC = "databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.facilities"
FC = "workspace.vf.facilities_clean"
FS = "workspace.vf.facility_sentences"
DS = "workspace.vf.distinct_sentences"

# --------------------------------------------------------------------------------------
# Reusable SQL expressions
# --------------------------------------------------------------------------------------

def missing_to_null(col: str) -> str:
    """Trim; '', 'null', 'none' (any case) -> NULL."""
    return f"CASE WHEN lower(trim({col})) IN ('', 'null', 'none') THEN NULL ELSE trim({col}) END"


def domain(col: str) -> str:
    """Host only: lowercase, strip scheme, strip leading www., stop at / : ? # whitespace.
    Result must look like a hostname (labels of [a-z0-9-] joined by dots), else NULL."""
    host = (
        f"regexp_extract(lower(trim({col})), "
        r"r'^(?:[a-z][a-z0-9+.\-]*://)?(?:www\.)?([^/:?#\s]+)', 1)"
    )
    return (
        f"CASE WHEN {col} IS NULL OR lower(trim({col})) IN ('', 'null', 'none') THEN NULL "
        rf"WHEN {host} RLIKE r'^[a-z0-9\-]+(\.[a-z0-9\-]+)+$' THEN {host} ELSE NULL END"
    )


def int_or_null(col: str) -> str:
    return f"try_cast(trim({col}) AS INT)"


def is_empty_scalar(col: str) -> str:
    return f"({col} IS NULL OR lower(trim({col})) IN ('', 'null', 'none'))"


UUID_RE = r"r'^[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}$'"

# --------------------------------------------------------------------------------------
# Placeholder rules (applied to sentence_key: lowercase, collapsed whitespace,
# trailing [whitespace . , ; : ! ? …] removed). Rule-based, conservative.
# --------------------------------------------------------------------------------------

P1_EXACT = [
    "none", "nil", "n/a", "na", "null", "nan", "unknown", "-", "--", "tbd",
    "not specified", "not available", "not applicable", "not mentioned", "not listed",
    "not provided", "not stated", "none listed", "none mentioned", "none specified",
    "none found", "not specified in content", "not specified in listing",
]

# P2: refers to the scraped source ("provided content", "on the page", ...) AND states absence.
# Evaluated on the key with parenthesised segments removed, so a real claim followed by
# "(not specified in content)" is NOT flagged.
P2_META = (
    r"r'\b(provided|supplied|given|source|input|scraped|available)( [a-z0-9&\x27.\-]+){0,4} "
    r"(content|text|page|pages|data|information|material|document|snippet|entry)\b"
    r"|\b(in|from|on) (the |this )?(content|text|input|listing|page|hospital page)\b"
    r"|\b(the|provided) content (does|did) not\b'"
)
P2_NEG = (
    r"r'\b(no|not|none|nor|unknown|without|omitted|unavailable|unspecified)\b|n\x27t\b'"
)

# P3: extractor note marking an item as absent: "liver transplant (not listed)",
# "neurophysiology? (not listed) — skip", "... -- exclude".
P3 = (
    r"r'\((not|no) (explicitly )?(listed|mentioned|specified|stated|found)\)\s*"
    r"((—|--|-)\s*(skip|exclude))?$|(—|--)\s*(skip|exclude)$'"
)

# P4: generic absence statement without a source reference:
# starts with "no"/"none" (optionally "there are/has/offers/the hospital has"), names a
# clinical-content noun AND a listing verb. Length-capped.
P4_START = r"r'^(there (are|is) |has |offers |the (hospital|facility) has )?(no|none) '"
P4_NOUN = (
    r"r'\b(procedures?|equipment|devices?|treatments?|surger(y|ies)|services?|"
    r"capabilit(y|ies)|details|information|interventions)\b'"
)
P4_VERB = (
    r"r'\b(listed|mentioned|specified|provided|described|documented|stated|detailed|"
    r"enumerated|identified)\b'"
)


def sql_list(items):
    return ", ".join("'" + i.replace("'", "''") + "'" for i in items)


def placeholder_rule(k: str) -> str:
    """Returns the first matching rule name ('P1'..'P4') or NULL."""
    # strip parenthesised segments, unless the whole key is one parenthesis
    k_noparen = (f"CASE WHEN {k} RLIKE r'^\\(.*\\)$' THEN {k} "
                 f"ELSE regexp_replace({k}, r'\\([^)]*\\)', '') END")
    return f"""CASE
      WHEN {k} IN ({sql_list(P1_EXACT)}) THEN 'P1_exact'
      WHEN {k_noparen} RLIKE {P2_META} AND {k_noparen} RLIKE {P2_NEG} THEN 'P2_source_absent'
      WHEN {k} RLIKE {P3} THEN 'P3_not_listed_note'
      WHEN length({k}) <= 150 AND {k} RLIKE {P4_START} AND {k} RLIKE {P4_NOUN}
           AND {k} RLIKE {P4_VERB} THEN 'P4_generic_absence'
      ELSE NULL END"""


# --------------------------------------------------------------------------------------
# Build statements
# --------------------------------------------------------------------------------------

BUILD = [
    # ---------------------------------------------------------------- facilities_clean
    f"""
CREATE OR REPLACE TABLE {FC} AS
WITH d AS (SELECT DISTINCT * FROM {SRC})
SELECT
  unique_id,
  {missing_to_null('name')}                       AS name,
  description,
  {missing_to_null('address_city')}               AS city,
  {missing_to_null('address_stateOrRegion')}      AS state,
  nullif(regexp_replace({missing_to_null('address_zipOrPostcode')}, r'\\s', ''), '') AS postcode,
  coalesce(regexp_replace({missing_to_null('address_zipOrPostcode')}, r'\\s', '')
           RLIKE r'^[1-9][0-9]{{5}}$', false)     AS postcode_valid,
  latitude,
  longitude,
  {int_or_null('numberDoctors')}                  AS numberDoctors,
  {int_or_null('capacity')}                       AS capacity,
  {int_or_null('yearEstablished')}                AS yearEstablished,
  array_sort(array_distinct(filter(
      transform(from_json(specialties, 'array<string>'), x -> trim(x)),
      x -> x IS NOT NULL AND x <> '')))           AS specialties,
  {domain('officialWebsite')}                     AS official_domain,
  {missing_to_null('officialPhone')}              AS official_phone,
  array_sort(array_distinct(filter(
      transform(from_json(source_urls, 'array<string>'), u -> {domain('u')}),
      x -> x IS NOT NULL)))                       AS source_domains,
  facilityTypeId,
  operatorTypeId,
  affiliationTypeIds,
  coalesce(organization_type, '') <> 'facility'   AS is_shifted,
  (latitude IS NULL OR longitude IS NULL)         AS coords_missing,
  coalesce(latitude NOT BETWEEN 6 AND 38 OR longitude NOT BETWEEN 68 AND 98, false)
                                                  AS coords_outside_india,
  coalesce(trim(address_country), '') <> 'India'  AS country_not_india,
  NOT coalesce(unique_id RLIKE {UUID_RE}, false)  AS id_not_uuid
FROM d
""",
    # ---------------------------------------------------------------- facility_sentences
    f"""
CREATE OR REPLACE TABLE {FS} AS
WITH d AS (SELECT DISTINCT unique_id, procedure, capability, equipment FROM {SRC}),
items AS (
  SELECT unique_id, 'procedure' AS source_column, 1 AS col_order, pos AS position, item
  FROM d LATERAL VIEW posexplode(from_json(procedure, 'array<string>')) e AS pos, item
  UNION ALL
  SELECT unique_id, 'capability', 2, pos, item
  FROM d LATERAL VIEW posexplode(from_json(capability, 'array<string>')) e AS pos, item
  UNION ALL
  SELECT unique_id, 'equipment', 3, pos, item
  FROM d LATERAL VIEW posexplode(from_json(equipment, 'array<string>')) e AS pos, item
),
keyed AS (
  SELECT unique_id, source_column, col_order, position,
         trim(item) AS sentence,
         regexp_replace(lower(regexp_replace(trim(item), r'\\s+', ' ')),
                        r'[\\s.,;:!?…]+$', '') AS sentence_key
  FROM items
  WHERE item IS NOT NULL AND trim(item) <> ''
),
flagged AS (
  SELECT *, {placeholder_rule('sentence_key')} AS placeholder_rule
  FROM keyed
  WHERE sentence_key <> ''
)
SELECT unique_id, source_column, position, sentence, sentence_key,
       placeholder_rule IS NOT NULL AS is_placeholder,
       placeholder_rule,
       row_number() OVER (PARTITION BY unique_id, sentence_key
                          ORDER BY col_order, position) > 1 AS is_repeat
FROM flagged
""",
    # ---------------------------------------------------------------- distinct_sentences
    f"""
CREATE OR REPLACE TABLE {DS} AS
WITH s AS (
  SELECT s.unique_id, s.sentence, s.sentence_key
  FROM {FS} s JOIN {FC} f USING (unique_id)
  WHERE NOT s.is_placeholder AND NOT f.is_shifted
),
forms AS (
  SELECT sentence_key, sentence, count(*) AS n,
         row_number() OVER (PARTITION BY sentence_key ORDER BY count(*) DESC, sentence) AS rk
  FROM s GROUP BY sentence_key, sentence
),
agg AS (
  SELECT sentence_key, count(DISTINCT unique_id) AS n_facilities, count(*) AS n_occurrences
  FROM s GROUP BY sentence_key
)
SELECT substr(sha2(a.sentence_key, 256), 1, 16) AS sentence_id,
       a.sentence_key,
       f.sentence AS example_sentence,
       a.n_facilities,
       a.n_occurrences
FROM agg a JOIN forms f ON f.sentence_key = a.sentence_key AND f.rk = 1
""",
]

# --------------------------------------------------------------------------------------
# Comments
# --------------------------------------------------------------------------------------

TABLE_COMMENTS = {
    FC: (
        "One row per facility (unique_id), mechanically cleaned from the read-only share "
        f"{SRC}. Built by scripts/build_clean_base.py: SELECT DISTINCT * removes the 11 exact "
        "duplicate rows (10,088 -> 10,077); name/city/state/postcode trimmed with '', 'null', "
        "'none' -> NULL; postcode spaces stripped; numberDoctors/capacity/yearEstablished "
        "try_cast to INT (NULL if unparseable, no range check); specialties parsed to a sorted "
        "distinct array; official_domain and source_domains are lowercase hosts without "
        "scheme/www. No content judgement, no LLM. Flags: is_shifted (organization_type <> "
        "'facility', NULL fails; the canonical column-shift test), coords_missing, "
        "coords_outside_india, country_not_india, id_not_uuid, postcode_valid. Filter "
        "downstream with NOT is_shifted."
    ),
    FS: (
        "One row per non-empty item of the procedure, capability and equipment JSON arrays, "
        f"from the deduplicated facilities (one row per unique_id, shifted rows included; "
        f"filter via {FC}.is_shifted). Built by scripts/build_clean_base.py. Items that are "
        "NULL, blank after trim, or whose sentence_key is empty are dropped. Flags: "
        "is_placeholder (rule-based, see placeholder_rule: P1_exact list, P2_source_absent = "
        "mentions the provided content/page AND states absence, P3_not_listed_note = "
        "'(not listed)'/'-- skip' extractor notes, P4_generic_absence = 'no ... procedures/"
        "equipment ... listed/mentioned'); is_repeat (same sentence_key already seen for the "
        "same unique_id, ordered procedure, capability, equipment then position; first "
        "occurrence unflagged)."
    ),
    DS: (
        "One row per distinct sentence_key over workspace.vf.facility_sentences, EXCLUDING "
        "placeholders (is_placeholder) and EXCLUDING sentences of shifted facilities "
        f"({FC}.is_shifted). Repeats within a facility are included in n_occurrences. "
        "sentence_id = first 16 hex chars of sha2(sentence_key, 256) (deterministic, stable "
        "across rebuilds). example_sentence = most frequent original (trimmed) form, ties "
        "broken alphabetically. Built by scripts/build_clean_base.py."
    ),
}

COLUMN_COMMENTS = {
    FC: {
        "unique_id": "Facility id from the source (unique in this table). Not always a UUID; see id_not_uuid.",
        "name": "Source name, trimmed; '', 'null', 'none' -> NULL.",
        "description": "Source description, unchanged.",
        "city": "Source address_city, trimmed; '', 'null', 'none' -> NULL. Unreliable; prefer coordinates.",
        "state": "Source address_stateOrRegion, trimmed; '', 'null', 'none' -> NULL. 254 raw spellings; prefer coordinates.",
        "postcode": "Source address_zipOrPostcode, trimmed, all whitespace removed; '', 'null', 'none' -> NULL.",
        "postcode_valid": "TRUE when postcode is 6 digits with first digit 1-9 (Indian PIN format). FALSE otherwise, including NULL.",
        "latitude": "Source latitude (DOUBLE), unchanged. See coords_missing / coords_outside_india.",
        "longitude": "Source longitude (DOUBLE), unchanged. See coords_missing / coords_outside_india.",
        "numberDoctors": "try_cast(trim(numberDoctors) AS INT); NULL when missing or unparseable. Not range-checked.",
        "capacity": "try_cast(trim(capacity) AS INT); NULL when missing or unparseable. Not range-checked.",
        "yearEstablished": "try_cast(trim(yearEstablished) AS INT); NULL when missing or unparseable. Not range-checked.",
        "specialties": "Source specialties JSON parsed to array<string>, items trimmed, blanks removed, distinct, sorted.",
        "official_domain": "Host of officialWebsite: lowercase, scheme and leading www. stripped, cut at / : ? #. NULL if missing ('', 'null', 'none') or not hostname-shaped.",
        "official_phone": "Source officialPhone, trimmed; '', 'null', 'none' (any case) -> NULL; otherwise the raw value (not parsed or validated; may hold several numbers).",
        "source_domains": "Distinct sorted hosts of the source_urls JSON array, same normalisation as official_domain.",
        "facilityTypeId": "Source facilityTypeId, unchanged.",
        "operatorTypeId": "Source operatorTypeId, unchanged.",
        "affiliationTypeIds": "Source affiliationTypeIds, unchanged.",
        "is_shifted": "Canonical column-shift flag: organization_type <> 'facility' (NULL counts as shifted). Exclude these rows downstream.",
        "coords_missing": "TRUE when latitude or longitude is NULL.",
        "coords_outside_india": "TRUE when coordinates are present but latitude not in [6,38] or longitude not in [68,98].",
        "country_not_india": "TRUE when trimmed address_country is not exactly 'India' (NULL counts as not India).",
        "id_not_uuid": "TRUE when unique_id is not a canonical 8-4-4-4-12 hex UUID.",
    },
    FS: {
        "unique_id": "Facility id; joins to workspace.vf.facilities_clean.unique_id.",
        "source_column": "Source array the item came from: procedure, capability or equipment.",
        "position": "0-based index of the item in its source JSON array (gaps where empty items were dropped).",
        "sentence": "The item text, trimmed (otherwise verbatim).",
        "sentence_key": "Normalised key: lowercase, whitespace runs collapsed to one space, trailing whitespace and . , ; : ! ? … removed.",
        "is_placeholder": "TRUE when the item is a 'nothing listed' placeholder per placeholder_rule. Not a facility claim.",
        "placeholder_rule": "First placeholder rule matched (P1_exact, P2_source_absent, P3_not_listed_note, P4_generic_absence) or NULL. Rules in scripts/build_clean_base.py.",
        "is_repeat": "TRUE when the same sentence_key already occurred for this unique_id earlier (order: procedure, capability, equipment, then position). First occurrence is FALSE.",
    },
    DS: {
        "sentence_id": "Stable id: first 16 hex chars of sha2(sentence_key, 256).",
        "sentence_key": "Normalised sentence key (see workspace.vf.facility_sentences.sentence_key). Unique in this table.",
        "example_sentence": "Most frequent original (trimmed) form of this key; ties broken alphabetically.",
        "n_facilities": "Number of distinct non-shifted facilities with at least one non-placeholder occurrence.",
        "n_occurrences": "Number of non-placeholder rows in facility_sentences for non-shifted facilities with this key (repeats included).",
    },
}


def comment_statements():
    out = []
    for t, c in TABLE_COMMENTS.items():
        out.append(f"COMMENT ON TABLE {t} IS '{c.replace(chr(39), chr(39) * 2)}'")
    for t, cols in COLUMN_COMMENTS.items():
        for col, c in cols.items():
            out.append(f"ALTER TABLE {t} ALTER COLUMN {col} COMMENT '{c.replace(chr(39), chr(39) * 2)}'")
    return out


# --------------------------------------------------------------------------------------
# Report queries
# --------------------------------------------------------------------------------------

REPORT = {
    "row_counts": f"""
SELECT (SELECT count(*) FROM {SRC}) AS src_rows,
       (SELECT count(*) FROM {FC}) AS facilities_clean_rows,
       (SELECT count(DISTINCT unique_id) FROM {FC}) AS facilities_clean_distinct_ids,
       (SELECT count(*) FROM {FS}) AS facility_sentences_rows,
       (SELECT count(*) FROM {DS}) AS distinct_sentences_rows,
       (SELECT count(DISTINCT sentence_id) FROM {DS}) AS distinct_sentence_ids""",
    "shift_tests": f"""
SELECT 'is_shifted' t, count_if(is_shifted) n, count_if(is_shifted) overlap, 0 missed_by_is_shifted FROM {FC}
UNION ALL SELECT 'coords_missing', count_if(coords_missing), count_if(coords_missing AND is_shifted), count_if(coords_missing AND NOT is_shifted) FROM {FC}
UNION ALL SELECT 'coords_outside_india', count_if(coords_outside_india), count_if(coords_outside_india AND is_shifted), count_if(coords_outside_india AND NOT is_shifted) FROM {FC}
UNION ALL SELECT 'country_not_india', count_if(country_not_india), count_if(country_not_india AND is_shifted), count_if(country_not_india AND NOT is_shifted) FROM {FC}
UNION ALL SELECT 'id_not_uuid', count_if(id_not_uuid), count_if(id_not_uuid AND is_shifted), count_if(id_not_uuid AND NOT is_shifted) FROM {FC}
UNION ALL SELECT 'any_extra_test', count_if(coords_missing OR coords_outside_india OR country_not_india OR id_not_uuid),
  count_if((coords_missing OR coords_outside_india OR country_not_india OR id_not_uuid) AND is_shifted),
  count_if((coords_missing OR coords_outside_india OR country_not_india OR id_not_uuid) AND NOT is_shifted) FROM {FC}""",
    "shift_missed_examples": f"""
SELECT f.unique_id, f.name, f.city, s.address_country, f.latitude, f.longitude, s.organization_type,
       f.coords_missing, f.coords_outside_india, f.country_not_india, f.id_not_uuid
FROM {FC} f JOIN (SELECT DISTINCT unique_id, address_country, organization_type FROM {SRC}) s USING (unique_id)
WHERE NOT f.is_shifted AND (f.coords_missing OR f.coords_outside_india OR f.country_not_india OR f.id_not_uuid)
ORDER BY f.coords_outside_india DESC, f.country_not_india DESC, f.id_not_uuid DESC, f.unique_id LIMIT 20""",
    "numeric_parse": f"""
WITH d AS (SELECT DISTINCT * FROM {SRC})
SELECT 'numberDoctors' f, count_if(NOT {is_empty_scalar('numberDoctors')}) non_empty,
       count_if(NOT {is_empty_scalar('numberDoctors')} AND {int_or_null('numberDoctors')} IS NULL) failed,
       count_if(lower(trim(numberDoctors)) = 'null') literal_null,
       min({int_or_null('numberDoctors')}) mn, max({int_or_null('numberDoctors')}) mx, count({int_or_null('numberDoctors')}) parsed FROM d
UNION ALL
SELECT 'capacity', count_if(NOT {is_empty_scalar('capacity')}),
       count_if(NOT {is_empty_scalar('capacity')} AND {int_or_null('capacity')} IS NULL),
       count_if(lower(trim(capacity)) = 'null'),
       min({int_or_null('capacity')}), max({int_or_null('capacity')}), count({int_or_null('capacity')}) FROM d
UNION ALL
SELECT 'yearEstablished', count_if(NOT {is_empty_scalar('yearEstablished')}),
       count_if(NOT {is_empty_scalar('yearEstablished')} AND {int_or_null('yearEstablished')} IS NULL),
       count_if(lower(trim(yearEstablished)) = 'null'),
       min({int_or_null('yearEstablished')}), max({int_or_null('yearEstablished')}), count({int_or_null('yearEstablished')}) FROM d""",
    "numeric_fail_by_shift": f"""
WITH d AS (SELECT DISTINCT * FROM {SRC})
SELECT coalesce(organization_type,'') <> 'facility' shifted,
  count_if(NOT {is_empty_scalar('numberDoctors')} AND {int_or_null('numberDoctors')} IS NULL) nd_fail,
  count_if(NOT {is_empty_scalar('capacity')} AND {int_or_null('capacity')} IS NULL) cap_fail,
  count_if(NOT {is_empty_scalar('yearEstablished')} AND {int_or_null('yearEstablished')} IS NULL) yr_fail
FROM d GROUP BY 1""",
    "field_stats": f"""
SELECT count_if(postcode_valid) postcode_valid, count_if(postcode IS NOT NULL AND NOT postcode_valid) postcode_invalid,
       count_if(postcode IS NULL) postcode_null,
       count_if(official_domain IS NOT NULL) has_official_domain,
       count_if(size(source_domains) > 0) has_source_domains,
       count_if(specialties IS NOT NULL AND size(specialties) > 0) has_specialties,
       count_if(name IS NULL) name_null, count_if(city IS NULL) city_null, count_if(state IS NULL) state_null
FROM {FC}""",
    "website_not_hostname": f"""
WITH d AS (SELECT DISTINCT * FROM {SRC})
SELECT count_if(NOT {is_empty_scalar('officialWebsite')}) website_non_empty,
       count_if(NOT {is_empty_scalar('officialWebsite')} AND {domain('officialWebsite')} IS NULL) website_unparsed
FROM d""",
    "items_total": f"""
WITH d AS (SELECT DISTINCT unique_id, procedure, capability, equipment FROM {SRC}),
a AS (SELECT 'procedure' c, from_json(procedure,'array<string>') a FROM d UNION ALL
      SELECT 'capability', from_json(capability,'array<string>') FROM d UNION ALL
      SELECT 'equipment', from_json(equipment,'array<string>') FROM d),
i AS (SELECT c, item FROM a LATERAL VIEW explode(a) e AS item)
SELECT c, count(*) total_items,
       count_if(item IS NULL OR trim(item) = '') dropped_blank,
       count_if(item IS NOT NULL AND trim(item) <> '' AND
                regexp_replace(lower(regexp_replace(trim(item), r'\\s+', ' ')), r'[\\s.,;:!?…]+$', '') = '') dropped_punct_only
FROM i GROUP BY ROLLUP(c) ORDER BY c""",
    "sentence_flags": f"""
SELECT coalesce(source_column,'ALL') source_column, count(*) rows_kept, count_if(is_placeholder) placeholder,
       count_if(is_repeat) repeat, count_if(is_placeholder AND is_repeat) both,
       count_if(f.is_shifted) from_shifted,
       count_if(NOT is_placeholder AND NOT is_repeat AND NOT f.is_shifted) usable
FROM {FS} s JOIN {FC} f USING (unique_id) GROUP BY ROLLUP(source_column) ORDER BY 1""",
    "placeholder_by_rule": f"""
SELECT placeholder_rule, count(*) n, count(DISTINCT sentence_key) distinct_keys
FROM {FS} WHERE is_placeholder GROUP BY 1 ORDER BY 1""",
    "top20_placeholders": f"""
SELECT sentence_key, placeholder_rule, count(*) n FROM {FS} WHERE is_placeholder
GROUP BY 1, 2 ORDER BY n DESC, sentence_key LIMIT 20""",
    "top30_sentences": f"""
SELECT example_sentence, n_facilities, n_occurrences FROM {DS}
ORDER BY n_facilities DESC, n_occurrences DESC, sentence_key LIMIT 30""",
    "zero_usable": f"""
SELECT count_if(NOT f.is_shifted) non_shifted_facilities,
       count_if(NOT f.is_shifted AND coalesce(u.n, 0) = 0) zero_usable_non_shifted,
       count_if(NOT f.is_shifted AND coalesce(a.n, 0) = 0) zero_items_at_all_non_shifted
FROM {FC} f
LEFT JOIN (SELECT unique_id, count(*) n FROM {FS} WHERE NOT is_placeholder AND NOT is_repeat GROUP BY 1) u USING (unique_id)
LEFT JOIN (SELECT unique_id, count(*) n FROM {FS} GROUP BY 1) a USING (unique_id)""",
    "check_occurrences": f"""
SELECT (SELECT sum(n_occurrences) FROM {DS}) sum_occurrences,
       (SELECT count(*) FROM {FS} s JOIN {FC} f USING (unique_id) WHERE NOT s.is_placeholder AND NOT f.is_shifted) expected""",
}


OFFICIAL_PHONE_COMMENT = "Source officialPhone, trimmed; '', 'null', 'none' (any case) -> NULL; otherwise the raw value (not parsed or validated; may hold several numbers)."


def add_phone(cur):
    """Add official_phone to the existing facilities_clean in place (no recreate; other
    columns and flags untouched): ADD COLUMN if missing, then MERGE from the source share."""
    cur.execute(f"DESCRIBE TABLE {FC}")
    if "official_phone" not in {r[0] for r in cur.fetchall()}:
        cur.execute(f"ALTER TABLE {FC} ADD COLUMN official_phone STRING")
    # the 11 duplicated ids are exact duplicate rows, so max() picks their single value
    cur.execute(f"""MERGE INTO {FC} t
USING (SELECT unique_id, max({missing_to_null('officialPhone')}) AS official_phone
       FROM {SRC} GROUP BY unique_id) s
ON t.unique_id = s.unique_id
WHEN MATCHED THEN UPDATE SET t.official_phone = s.official_phone""")
    cur.execute(f"ALTER TABLE {FC} ALTER COLUMN official_phone COMMENT '{OFFICIAL_PHONE_COMMENT.replace(chr(39), chr(92) + chr(39))}'")
    cur.execute(f"SELECT count(*), count(official_phone) FROM {FC}")
    print("facilities_clean rows, with official_phone:", cur.fetchall(), flush=True)


def clean_base_main(mode="build", report_names=None):
    report_only = mode == "report"
    with connect() as conn:
        cur = conn.cursor()
        if mode == "add_phone":
            add_phone(cur)
            return
        if not report_only:
            for stmt in BUILD:
                print("RUN:", stmt.strip().splitlines()[0], flush=True)
                cur.execute(stmt)
            for stmt in comment_statements():
                cur.execute(stmt)
            print("comments applied", flush=True)
        for name, q in REPORT.items():
            if report_names and name not in report_names:
                continue
            cur.execute(q)
            rows = cur.fetchall()
            cols = [d[0] for d in cur.description]
            print(f"\n== {name}")
            print(" | ".join(cols))
            for r in rows:
                print(" | ".join(str(x) for x in r))

In [0]:
if REBUILD:
    clean_base_main("build")   # CREATE OR REPLACE the three tables, comments, then the full report
else:
    clean_base_main("report", ["row_counts", "shift_tests", "sentence_flags", "placeholder_by_rule", "zero_usable"])


== row_counts
src_rows | facilities_clean_rows | facilities_clean_distinct_ids | facility_sentences_rows | distinct_sentences_rows | distinct_sentence_ids
10088 | 10077 | 10077 | 451110 | 271262 | 271262

== shift_tests
t | n | overlap | missed_by_is_shifted
is_shifted | 88 | 88 | 0
coords_missing | 118 | 88 | 30
coords_outside_india | 6 | 0 | 6
country_not_india | 88 | 88 | 0
id_not_uuid | 88 | 88 | 0
any_extra_test | 124 | 88 | 36

== sentence_flags
source_column | rows_kept | placeholder | repeat | both | from_shifted | usable
ALL | 451110 | 1000 | 26422 | 18 | 0 | 423706
capability | 231465 | 23 | 11695 | 1 | 0 | 219748
equipment | 55998 | 433 | 4558 | 16 | 0 | 51023
procedure | 163647 | 544 | 10169 | 1 | 0 | 152935

== placeholder_by_rule
placeholder_rule | n | distinct_keys
P1_exact | 22 | 7
P2_source_absent | 948 | 865
P3_not_listed_note | 7 | 7
P4_generic_absence | 23 | 23

== zero_usable
non_shifted_facilities | zero_usable_non_shifted | zero_items_at_all_non_shifted
9989 | 6

In [0]:
%sql
-- One record, split into sentences (the Aravind Eye Hospital record, used again in steps 5 and 6)
SELECT source_column, position, sentence, is_placeholder, is_repeat
FROM workspace.vf.facility_sentences
WHERE unique_id = 'b8a5401f-42f1-422a-8cd9-686a15b4cb76'
ORDER BY source_column DESC, position
LIMIT 8

source_column,position,sentence,is_placeholder,is_repeat
procedure,0,Dialysis treatment is provided via a 20-bed Dialysis Unit with 10 machines as part of the Pradhan Mantri National Dialysis Programme,false,false
procedure,1,CT scan and MRI imaging services are available for registered IGH patients on advice,false,false
procedure,2,Routine minor and major operating theatre services are available,false,false
procedure,3,"Provides Panchkarma therapies and OPD services at Charak Ayurvedic Kendra for joints (Osteoarthritis, Rheumatoid arthritis, Ankylosing spondylitis, PIVD cervical and lumbar), muscular disorders, neurological disorders (paralysis, migraine), and skin disorders (eczema, psoriasis).",false,false
procedure,4,Brain and spine surgical operations are performed in the Neurosurgery department,false,false
procedure,5,"Subtraction angiography (DSA) is used for brain, spine and other regions",false,false
procedure,6,Interventional embolization procedures for tumours using DSA,false,false
procedure,7,Hemodialysis is performed in three units as a regular service,false,false


## Step 2 — Place every facility on the map

**What.** Checks each facility's listed coordinates against two independent references: a **PIN-code centre** (median of the post offices of that PIN, after dropping offices more than 100 km from their 3-digit prefix centre) and a **city centre** (median of the other facilities in the same city, cities with >= 5 facilities). Ten ordered rules pick one point per facility and a confidence level with a plain-language note.

**Why.** Place names are unreliable (254 state spellings) and some coordinates are scrambled, so distance must come from a checked point, and an uncertain point must say so.

**Reads** share `facilities`, `india_post_pincode_directory`. **Writes** function `haversine_km`, `pin_point` (18,966 PINs), `city_point` (272 cities), `facility_location` (10,077). Source: `sql/00_schema.sql` .. `sql/03_facility_location.sql`, placeholders filled from the shared names above.

Measured: confidence high 5,551, good 3,221, approximate / suspect / unverified 1,215, cannot be placed 90.

In [0]:
LOCATION_SQL = {
    "00_schema": """
-- Target schema and the distance function every other file uses.
CREATE SCHEMA IF NOT EXISTS {TGT};

CREATE OR REPLACE FUNCTION {F_HAVERSINE}(
  lat1 DOUBLE, lon1 DOUBLE, lat2 DOUBLE, lon2 DOUBLE
) RETURNS DOUBLE
COMMENT 'Great-circle distance in km between two WGS84 points.'
RETURN 6371.0088 * 2 * asin(sqrt(
    pow(sin(radians(lat2 - lat1) / 2), 2)
  + cos(radians(lat1)) * cos(radians(lat2)) * pow(sin(radians(lon2 - lon1) / 2), 2)
));
""",
    "01_pin_point": """
-- One row per PIN code.
--   prefix point = median lat/lon of all in-box offices sharing a 3-digit prefix
--   offices more than {PREFIX_OUTLIER_KM} km from their prefix point are dropped
--   point       = median lat/lon of what remains
--   spread_km   = furthest remaining office from that point
--   trusted     = spread_km < {PIN_TIGHT_KM}
CREATE OR REPLACE TABLE {T_PIN_POINT} AS
WITH office AS (
  SELECT lpad(CAST(pincode AS STRING), 6, '0')             AS pin,
         substr(lpad(CAST(pincode AS STRING), 6, '0'), 1, 3) AS prefix,
         try_cast(latitude  AS DOUBLE)                     AS lat,
         try_cast(longitude AS DOUBLE)                     AS lon
  FROM {T_PINCODE}
),
in_box AS (
  SELECT * FROM office
  WHERE lat BETWEEN {LAT_MIN} AND {LAT_MAX}
    AND lon BETWEEN {LON_MIN} AND {LON_MAX}
),
prefix_point AS (
  SELECT prefix, median(lat) AS plat, median(lon) AS plon
  FROM in_box GROUP BY prefix
),
kept AS (
  SELECT b.pin, b.lat, b.lon
  FROM in_box b JOIN prefix_point p USING (prefix)
  WHERE {F_HAVERSINE}(b.lat, b.lon, p.plat, p.plon) <= {PREFIX_OUTLIER_KM}
),
point AS (
  SELECT pin, median(lat) AS lat, median(lon) AS lon, count(*) AS n_offices
  FROM kept GROUP BY pin
)
SELECT p.pin,
       p.lat,
       p.lon,
       p.n_offices,
       max({F_HAVERSINE}(k.lat, k.lon, p.lat, p.lon))                       AS spread_km,
       max({F_HAVERSINE}(k.lat, k.lon, p.lat, p.lon)) < {PIN_TIGHT_KM}      AS trusted
FROM point p JOIN kept k USING (pin)
GROUP BY p.pin, p.lat, p.lon, p.n_offices;
""",
    "02_city_point": """
-- One row per city name, from the facilities that already have good coordinates.
-- Only cities with at least {MIN_CITY_FACILITIES} such facilities are kept.
CREATE OR REPLACE TABLE {T_CITY_POINT} AS
WITH dedup AS (
  SELECT * FROM (
    SELECT unique_id, address_city, latitude, longitude,
           row_number() OVER (PARTITION BY unique_id ORDER BY name) AS rn
    FROM {T_FACILITIES}
  ) WHERE rn = 1
),
usable AS (
  SELECT lower(trim(address_city)) AS city_key, latitude AS lat, longitude AS lon
  FROM dedup
  WHERE address_city IS NOT NULL AND trim(address_city) <> ''
    AND latitude  BETWEEN {LAT_MIN} AND {LAT_MAX}
    AND longitude BETWEEN {LON_MIN} AND {LON_MAX}
)
SELECT city_key,
       median(lat) AS lat,
       median(lon) AS lon,
       count(*)    AS n_facilities
FROM usable
GROUP BY city_key
HAVING count(*) >= {MIN_CITY_FACILITIES};
""",
    "03_facility_location": """
-- One row per facility. Picks the point to show and says how much to trust it.
-- A check "passes" when the distance is under {NEAR_KM} km.
-- The PIN point is only used as a check when pin_point.trusted. An untrusted
-- PIN point is still used as a weaker check, with the threshold widened by that
-- PIN's own spread_km (rules 9 and 10).
CREATE OR REPLACE TABLE {T_FAC_LOC} AS
WITH dedup AS (
  SELECT * FROM (
    SELECT f.*, row_number() OVER (PARTITION BY unique_id ORDER BY name) AS rn
    FROM {T_FACILITIES} f
  ) WHERE rn = 1
),
base AS (
  SELECT
    unique_id AS facility_id,
    name,
    nullif(trim(concat_ws(', ',
      nullif(trim(address_line1), ''), nullif(trim(address_line2), ''),
      nullif(trim(address_line3), ''))), '')            AS address,
    address_city                                        AS city,
    address_stateOrRegion                               AS state,
    nullif(trim(address_zipOrPostcode), '')             AS postcode,
    latitude  AS raw_lat,
    longitude AS raw_lon,
    lower(trim(address_city))                           AS city_key,
    -- first 6 consecutive digits found in the postcode field
    regexp_extract(regexp_replace(coalesce(address_zipOrPostcode, ''), '[^0-9]', ''),
                   '^([0-9]{{6}})', 1)                  AS pin
  FROM dedup
),
joined AS (
  SELECT
    b.facility_id, b.name, b.address, b.city, b.state, b.postcode,
    b.raw_lat, b.raw_lon,
    CASE WHEN p.trusted     THEN p.lat END AS pin_lat,
    CASE WHEN p.trusted     THEN p.lon END AS pin_lon,
    CASE WHEN NOT p.trusted THEN p.lat END AS weak_pin_lat,
    CASE WHEN NOT p.trusted THEN p.lon END AS weak_pin_lon,
    CASE WHEN NOT p.trusted THEN p.spread_km END AS weak_pin_spread_km,
    c.lat AS city_lat,
    c.lon AS city_lon,
    -- latitude/longitude are NULL for most unplaceable rows, and NULL BETWEEN ...
    -- is NULL, not false. Without the coalesce, NOT raw_ok never fires and every
    -- such row falls through to the "nothing usable" rule.
    coalesce(b.raw_lat BETWEEN {LAT_MIN} AND {LAT_MAX}
         AND b.raw_lon BETWEEN {LON_MIN} AND {LON_MAX}, false) AS raw_ok
  FROM base b
  LEFT JOIN {T_PIN_POINT}  p ON b.pin = p.pin AND b.pin <> ''
  LEFT JOIN {T_CITY_POINT} c ON b.city_key = c.city_key
),
dist AS (
  SELECT j.*,
    CASE WHEN raw_ok AND pin_lat  IS NOT NULL THEN {F_HAVERSINE}(raw_lat, raw_lon, pin_lat,  pin_lon)  END AS d_pin_km,
    CASE WHEN raw_ok AND city_lat IS NOT NULL THEN {F_HAVERSINE}(raw_lat, raw_lon, city_lat, city_lon) END AS d_city_km,
    CASE WHEN raw_ok AND weak_pin_lat IS NOT NULL THEN {F_HAVERSINE}(raw_lat, raw_lon, weak_pin_lat, weak_pin_lon) END AS d_weak_pin_km,
    CASE WHEN pin_lat IS NOT NULL AND city_lat IS NOT NULL
         THEN {F_HAVERSINE}(pin_lat, pin_lon, city_lat, city_lon) END AS d_pin_city_km
  FROM joined j
),
flagged AS (
  SELECT d.*,
    (CASE WHEN pin_lat  IS NOT NULL THEN 1 ELSE 0 END)
  + (CASE WHEN city_lat IS NOT NULL THEN 1 ELSE 0 END)                       AS n_checks,
    (CASE WHEN d_pin_km  < {NEAR_KM} THEN 1 ELSE 0 END)
  + (CASE WHEN d_city_km < {NEAR_KM} THEN 1 ELSE 0 END)                      AS n_pass,
    coalesce(d_weak_pin_km < {NEAR_KM} + weak_pin_spread_km, false)          AS weak_pin_pass
  FROM dist d
),
ruled AS (
  SELECT f.*,
    CASE
      WHEN raw_ok AND n_checks = 2 AND n_pass = 2                              THEN 1
      WHEN raw_ok AND n_pass >= 1                                              THEN 2
      WHEN raw_ok AND n_checks = 2 AND n_pass = 0
           AND d_pin_city_km < {NEAR_KM}                                       THEN 3
      -- weak PIN rescue: only reachable when the PIN point exists but is untrusted
      WHEN raw_ok AND weak_pin_lat IS NOT NULL AND weak_pin_pass               THEN 9
      WHEN raw_ok AND n_checks >= 1 AND n_pass = 0                             THEN 4
      WHEN raw_ok AND n_checks = 0 AND weak_pin_lat IS NOT NULL                THEN 10
      WHEN raw_ok AND n_checks = 0                                             THEN 5
      WHEN NOT raw_ok AND pin_lat  IS NOT NULL                                 THEN 6
      WHEN NOT raw_ok AND city_lat IS NOT NULL                                 THEN 7
      ELSE 8
    END AS rule_id
  FROM flagged f
)
SELECT
  facility_id, name, address, city, state, postcode,
  raw_lat, raw_lon, pin_lat, pin_lon, city_lat, city_lon,
  weak_pin_lat, weak_pin_lon, weak_pin_spread_km,
  d_pin_km, d_city_km, d_weak_pin_km,
  CASE rule_id WHEN 3 THEN city_lat WHEN 6 THEN pin_lat WHEN 7 THEN city_lat
               WHEN 8 THEN CAST(NULL AS DOUBLE) ELSE raw_lat END AS final_lat,
  CASE rule_id WHEN 3 THEN city_lon WHEN 6 THEN pin_lon WHEN 7 THEN city_lon
               WHEN 8 THEN CAST(NULL AS DOUBLE) ELSE raw_lon END AS final_lon,
  CASE rule_id WHEN 3 THEN 'city_point' WHEN 6 THEN 'pin_point'
               WHEN 7 THEN 'city_point' WHEN 8 THEN 'none'
               ELSE 'coordinates' END                            AS location_source,
  CASE rule_id WHEN 1 THEN 'high'        WHEN 2 THEN 'good'
               WHEN 3 THEN 'approximate' WHEN 4 THEN 'suspect'
               WHEN 5 THEN 'unverified'  WHEN 6 THEN 'approximate'
               WHEN 7 THEN 'approximate' WHEN 9 THEN 'good'
               WHEN 10 THEN 'suspect'    ELSE 'none' END         AS location_confidence,
  CASE rule_id
    WHEN 1 THEN 'Listed coordinates agree with both the postcode and the city.'
    WHEN 2 THEN 'Listed coordinates agree with the postcode or the city.'
    WHEN 3 THEN 'Listed coordinates conflict with the postcode and city. Showing city centre.'
    WHEN 4 THEN 'Listed coordinates do not match the postcode or city on record. Position may be wrong.'
    WHEN 5 THEN 'Listed coordinates could not be cross-checked against any postcode or city.'
    WHEN 6 THEN 'No usable coordinates. Showing the centre of the postcode area.'
    WHEN 7 THEN 'No usable coordinates. Showing the city centre.'
    WHEN 9 THEN concat('Listed coordinates agree with the postcode area, which is wide (about ',
                       cast(round(weak_pin_spread_km) AS INT), ' km across), so this is a loose check.')
    WHEN 10 THEN concat('The only cross-check available was a wide postcode area (about ',
                        cast(round(weak_pin_spread_km) AS INT),
                        ' km across) and the coordinates fall outside even that. Position may be wrong.')
    ELSE 'No usable coordinates, postcode or city. This facility cannot be placed on the map.'
  END                                                            AS location_note,
  rule_id
FROM ruled;
""",
}


def run_sql_file(text: str):
    """Fill {placeholders} from the shared names, then run each statement in order."""
    for stmt in re.split(r";\s*(?:\n|$)", text.format(**subs())):
        if stmt.strip():
            sql_exec(stmt)


if REBUILD:
    for name in LOCATION_SQL:
        print("RUN", name)
        run_sql_file(LOCATION_SQL[name])

In [0]:
%sql
SELECT location_confidence, location_source, count(*) AS facilities
FROM workspace.vf.facility_location l
JOIN workspace.vf.facilities_clean c ON c.unique_id = l.facility_id
WHERE NOT c.is_shifted
GROUP BY 1, 2 ORDER BY facilities DESC

location_confidence,location_source,facilities
high,coordinates,5551
good,coordinates,3221
suspect,coordinates,666
approximate,city_point,449
unverified,coordinates,73
approximate,pin_point,26
none,none,3


In [0]:
%sql
-- Examples of the weaker levels, with the note the app shows
SELECT c.name, c.city, l.location_confidence, round(l.d_pin_km, 1) AS km_from_pin_centre,
       round(l.d_city_km, 1) AS km_from_city_centre, l.location_note
FROM workspace.vf.facility_location l
JOIN workspace.vf.facilities_clean c ON c.unique_id = l.facility_id
WHERE NOT c.is_shifted AND l.location_confidence IN ('suspect', 'approximate')
ORDER BY l.facility_id LIMIT 5

name,city,location_confidence,km_from_pin_centre,km_from_city_centre,location_note
Samved Orthopaedic Hospital,Navrangpura,suspect,275.5,null,Listed coordinates do not match the postcode or city on record. Position may be wrong.
Indira Dental Clinic,Vellore,suspect,null,236.2,Listed coordinates do not match the postcode or city on record. Position may be wrong.
Gokul Hospital,Jamnagar,suspect,58.8,87.3,Listed coordinates do not match the postcode or city on record. Position may be wrong.
"Sadhana Diabetes & Hormone’s Clinic , Dadar West , Mumbai",Mumbai,approximate,124.6,126.5,Listed coordinates conflict with the postcode and city. Showing city centre.
Government Chest Diseases and Tuberculosis Hospital,Jammu,approximate,149.6,152.9,Listed coordinates conflict with the postcode and city. Showing city centre.


## Step 3 — What kind of facility is it?

**What.** One `ai_query` call per facility assigns one of 26 types (eye hospital, dental clinic, maternity, multispecialty...), plus ownership and a self-reported confidence. Input is the name, the first 400 characters of the description and the specialties; name and description lead, specialties only break ties because they are over-assigned in this data. `unique_id` stays a SQL column and is never sent through the model. Each answer is parsed and every field checked against its allowed set; failures get one retry, then `unknown`.

**Why.** An eye hospital claiming a dialysis unit deserves suspicion; the verdict in step 6 needs to know what each facility is.

**Reads** `facilities_clean`. **Writes** `facility_type` (9,989 rows, one per non-shifted facility). Source: `scripts/facility_type_single.py`, prompt `ft_single_v1`.

Measured (`reports/facility_type_single.md`): 9,691 rows in the full run, 9,690 valid on the first call, 1 failed after retry; about 40 rows/s.

In [0]:
# scripts/facility_type_single.py
S = VF
SRC = f"{S}.facilities_clean"
PROMPT_VERSION = "ft_single_v1"
PROBE_N = 100

TYPES = [
    "multispecialty_hospital", "general_hospital_or_nursing_home",
    "medical_college_teaching_hospital", "government_health_centre",
    "maternity_women_children", "pediatric", "eye", "dental", "ent_hearing_speech",
    "skin_hair_cosmetic", "ivf_fertility", "kidney_urology", "cardiac", "cancer",
    "orthopedic_spine", "neuro", "gastro_liver", "diagnostic_imaging_lab",
    "physiotherapy_rehab", "mental_health_deaddiction", "ayush_alternative",
    "home_care_or_ambulance", "general_clinic_polyclinic", "other_single_specialty",
    "not_a_care_facility", "unknown",
]
OWNERSHIP = ["public", "private", "charitable_trust", "unknown"]
CONFIDENCE = ["high", "medium", "low"]

# ft_v3 text (as stored in facility_type_raw) with only these changes: one facility instead
# of a numbered list, no "n"/"echo" fields, a single JSON object instead of an array.
# The facility JSON is appended directly after "Facility:\n".
PROMPT = """You classify records of Indian healthcare facilities by FACILITY TYPE.

The facility below is given as JSON with "name",
"description" (first 400 characters) and "specialties" (a list of tags).

How to weigh the evidence:
- The NAME and the DESCRIPTION lead. Decide from them first.
- "specialties" is over-assigned and unreliable: generic tags such as
  internalMedicine and familyMedicine appear even on dental clinics and hearing
  clinics. Use specialties ONLY to break a tie when the name and description
  do not decide.

facility_type - exactly one of:
  multispecialty_hospital - hospital with many specialties ("multi speciality", "super speciality")
  general_hospital_or_nursing_home - general hospital or nursing home without one specialty focus
  medical_college_teaching_hospital - medical college, institute of medical sciences, teaching hospital
  government_health_centre - PHC, CHC, sub-centre, district/taluk/civil/government hospital, ESI dispensary
  maternity_women_children - maternity home, women's hospital, mother and child, obstetrics/gynaecology
  pediatric - children's hospital or child clinic
  eye
  dental
  ent_hearing_speech - ENT, hearing aids, audiology, speech therapy
  skin_hair_cosmetic - dermatology, hair, cosmetic or aesthetic clinic
  ivf_fertility
  kidney_urology - nephrology, dialysis centre, urology
  cardiac
  cancer
  orthopedic_spine
  neuro - neurology, neurosurgery
  gastro_liver
  diagnostic_imaging_lab - diagnostic centre, scan centre, imaging, pathology lab
  physiotherapy_rehab
  mental_health_deaddiction
  ayush_alternative - ayurveda, homeopathy, unani, siddha, naturopathy, yoga
  home_care_or_ambulance
  general_clinic_polyclinic - doctor's clinic or polyclinic without one specialty focus
  other_single_specialty - a single specialty not listed above
  not_a_care_facility - pharmacy, supplier, company, NGO office, school, veterinary, other non-provider
  unknown - not enough information to decide

is_single_specialty - true if the facility focuses on one specialty or service line,
  false for hospitals, colleges, government centres and general clinics.
ownership - public (government, municipal, ESI, railway, armed forces),
  private, charitable_trust (trust, society, mission, NGO, religious charity),
  or unknown when nothing indicates it.
confidence - high (stated plainly), medium (inferred), low (guess or conflicting evidence).
reason - at most 12 words, quoting the words in the name or description that decided it.

Output rules:
- Reply with ONLY a single JSON object for this facility, nothing added.
- Use ONLY the allowed values above. Never invent a value.
- The object: {"facility_type":"<value>",
  "is_single_specialty":true|false,"ownership":"<value>","confidence":"<value>",
  "reason":"<at most 12 words>"}

Facility:
"""


def run(stmt: str, params=None, fetch=False):
    with connect() as c:
        with c.cursor() as cur:
            cur.execute(stmt, params or [])
            return cur.fetchall() if fetch else None


def q(s: str) -> str:
    return "'" + str(s).replace("\\", "\\\\").replace("'", "\\'") + "'"


# --------------------------------------------------------------------------- validation
def parse(raw):
    return json.loads(raw[raw.index("{"): raw.rindex("}") + 1])


def validate(raw, err):
    if err:
        return False, f"endpoint_error: {str(err)[:150]}"
    if not raw:
        return False, "empty_response"
    try:
        o = parse(raw)
    except Exception as e:  # noqa: BLE001
        return False, f"unparseable: {type(e).__name__}"
    if not isinstance(o, dict):
        return False, "not an object"
    if o.get("facility_type") not in TYPES:
        return False, f"facility_type {o.get('facility_type')!r}"
    if not isinstance(o.get("is_single_specialty"), bool):
        return False, f"is_single_specialty {o.get('is_single_specialty')!r}"
    if o.get("ownership") not in OWNERSHIP:
        return False, f"ownership {o.get('ownership')!r}"
    if o.get("confidence") not in CONFIDENCE:
        return False, f"confidence {o.get('confidence')!r}"
    if not isinstance(o.get("reason"), str):
        return False, f"reason {o.get('reason')!r}"
    return True, ""


# --------------------------------------------------------------------------- setup
def setup():
    run(f"""CREATE TABLE IF NOT EXISTS {S}.facility_type_single_prompts (
        prompt_version STRING, model STRING, prompt STRING, created_ts TIMESTAMP)""")
    run(f"DELETE FROM {S}.facility_type_single_prompts WHERE prompt_version = ?", [PROMPT_VERSION])
    run(f"INSERT INTO {S}.facility_type_single_prompts VALUES (?, ?, ?, current_timestamp())",
        [PROMPT_VERSION, MODEL, PROMPT])
    run(f"""CREATE TABLE IF NOT EXISTS {S}.facility_type_single_raw (
        unique_id STRING, phase STRING, attempt INT, response STRING, error STRING,
        prompt_version STRING, ts TIMESTAMP)""")
    run(f"""CREATE TABLE IF NOT EXISTS {S}.facility_type_single_runs (
        phase STRING, unique_id STRING)""")
    run(f"""CREATE TABLE IF NOT EXISTS {S}.facility_type_single_status (
        unique_id STRING, phase STRING, accepted_attempt INT, failed BOOLEAN,
        problems STRING, prompt_version STRING)""")


def assign(phase: str):
    for t in ("facility_type_single_runs", "facility_type_single_status"):
        run(f"DELETE FROM {S}.{t} WHERE phase = ?", [phase])
    run(f"DELETE FROM {S}.facility_type_single_raw WHERE phase = ? AND prompt_version = ?",
        [phase, PROMPT_VERSION])
    if phase == "pilot":
        sel = f"""SELECT b.unique_id FROM {S}.facility_type_batches b
                  JOIN {SRC} c USING (unique_id)
                  WHERE b.phase = 'pilot' AND NOT c.is_shifted"""
    else:
        others = f"SELECT unique_id FROM {S}.facility_type_single_runs WHERE phase != '{phase}'"
        limit = f"ORDER BY md5(concat('probe', unique_id)) LIMIT {PROBE_N}" if phase == "probe" else ""
        sel = f"""SELECT unique_id FROM {SRC}
                  WHERE NOT is_shifted AND unique_id NOT IN ({others}) {limit}"""
    run(f"INSERT INTO {S}.facility_type_single_runs SELECT '{phase}', unique_id FROM ({sel})")
    return run(f"SELECT count(*) FROM {S}.facility_type_single_runs WHERE phase = ?",
               [phase], fetch=True)[0][0]


# --------------------------------------------------------------------------- calling
def call(phase: str, attempt: int, only_failed_of=None) -> float:
    """One SQL statement: ai_query over every facility of the phase (attempt 1) or over the
    given failed unique_ids (attempt 2). Returns wall-clock seconds."""
    where = ""
    if only_failed_of is not None:
        where = "AND c.unique_id IN (" + ",".join(q(u) for u in only_failed_of) + ")"
    stmt = f"""
    INSERT INTO {S}.facility_type_single_raw
    WITH inp AS (
      SELECT c.unique_id,
             concat(?, to_json(named_struct(
               'name', coalesce(trim(c.name), ''),
               'description', coalesce(substr(c.description, 1, 400), ''),
               'specialties', coalesce(c.specialties, array())))) AS prompt
      FROM {SRC} c JOIN {S}.facility_type_single_runs r
        ON r.unique_id = c.unique_id AND r.phase = ?
      WHERE NOT c.is_shifted {where}
    ), called AS (
      SELECT unique_id,
             ai_query('{MODEL}', prompt,
                      modelParameters => named_struct('temperature', 0.0, 'max_tokens', 400),
                      failOnError => false) AS r
      FROM inp
    )
    SELECT unique_id, ?, {attempt}, r.result, r.errorMessage, ?, current_timestamp()
    FROM called"""
    t = time.time()
    run(stmt, [PROMPT, phase, phase, PROMPT_VERSION])
    return round(time.time() - t, 1)


def check(phase: str, attempt: int):
    rows = run(f"""SELECT unique_id, response, error FROM {S}.facility_type_single_raw
                   WHERE phase = ? AND attempt = ? AND prompt_version = ?""",
               [phase, attempt, PROMPT_VERSION], fetch=True)
    ok, bad = [], {}
    for u, r, e in rows:
        good, prob = validate(r, e)
        (ok.append(u) if good else bad.__setitem__(u, prob))
    return ok, bad


def write_status(phase, ok1, bad1, ok2, bad2):
    vals = [f"({q(u)},'{phase}',1,false,'',{q(PROMPT_VERSION)})" for u in ok1]
    vals += [f"({q(u)},'{phase}',2,false,{q('a1: ' + bad1[u])},{q(PROMPT_VERSION)})" for u in ok2]
    vals += [f"({q(u)},'{phase}',NULL,true,{q('a1: ' + bad1[u] + ' | a2: ' + bad2.get(u, 'missing'))},"
             f"{q(PROMPT_VERSION)})" for u in bad1 if u not in ok2]
    for i in range(0, len(vals), 1000):
        run(f"INSERT INTO {S}.facility_type_single_status VALUES " + ",".join(vals[i:i + 1000]))


def run_phase(phase: str, retry=True):
    setup()
    n = assign(phase)
    out = {"phase": phase, "rows": n}
    out["a1_seconds"] = call(phase, 1)
    ok1, bad1 = check(phase, 1)
    out["a1_valid"], out["a1_failed"] = len(ok1), len(bad1)
    out["a1_missing"] = n - len(ok1) - len(bad1)
    out["a1_problems"] = bad1
    print(json.dumps({k: v for k, v in out.items() if k != "a1_problems"}), flush=True)
    ok2, bad2 = [], {}
    if retry and bad1:
        out["a2_seconds"] = call(phase, 2, list(bad1))
        ok2, bad2 = check(phase, 2)
        out["a2_valid"], out["a2_failed"] = len(ok2), len(bad2)
    write_status(phase, ok1, bad1, ok2, bad2)
    out["failed_after_retry"] = len([u for u in bad1 if u not in ok2])
    print(json.dumps(out, indent=1), flush=True)
    return out


# --------------------------------------------------------------------------- output
PARSED = f"""
    SELECT s.unique_id, s.phase, s.failed, s.accepted_attempt,
           from_json(regexp_extract(r.response, '(?s)(\\\\{{.*\\\\}})', 1),
             'struct<facility_type:string,is_single_specialty:boolean,ownership:string,
                     confidence:string,reason:string>') AS o, r.ts
    FROM {S}.facility_type_single_status s
    LEFT JOIN {S}.facility_type_single_raw r
      ON r.unique_id = s.unique_id AND r.phase = s.phase AND r.attempt = s.accepted_attempt
     AND r.prompt_version = s.prompt_version
    WHERE s.prompt_version = '{PROMPT_VERSION}'"""


def build_output():
    run(f"""
    CREATE OR REPLACE TABLE {S}.facility_type AS
    WITH p AS ({PARSED})
    SELECT c.unique_id, c.name,
           CASE WHEN p.failed THEN 'unknown' ELSE p.o.facility_type END AS facility_type,
           CASE WHEN p.failed THEN NULL ELSE p.o.is_single_specialty END AS is_single_specialty,
           CASE WHEN p.failed THEN 'unknown' ELSE p.o.ownership END AS ownership,
           CASE WHEN p.failed THEN 'low' ELSE p.o.confidence END AS confidence,
           CASE WHEN p.failed THEN NULL ELSE p.o.reason END AS reason,
           coalesce(p.failed, true) AS failed,
           '{PROMPT_VERSION}' AS prompt_version,
           p.ts AS run_ts
    FROM {SRC} c LEFT JOIN p USING (unique_id)
    WHERE NOT c.is_shifted""")
    comments = {
        "unique_id": "Facility key; joins to workspace.vf.facilities_clean.unique_id.",
        "name": "Facility name from facilities_clean (not model output).",
        "facility_type": "One of 26 labels from prompt ft_single_v1, decided from name + first 400 chars of description (specialties only as tiebreaker). 'unknown' when the model said so or the row failed validation.",
        "is_single_specialty": "Model output: true if the facility focuses on one specialty or service line. NULL when failed.",
        "ownership": "Model output: public, private, charitable_trust or unknown ('unknown' when failed).",
        "confidence": "Model self-reported confidence: high (stated plainly), medium (inferred), low (guess or conflicting). Set to low when failed.",
        "reason": "Model output, meant to be at most 12 words quoting the deciding words of name/description. Not enforced; can exceed 12 words. NULL when failed.",
        "failed": "true if both the first call and the one retry returned a response that did not parse or had a value outside the allowed sets.",
        "prompt_version": "Prompt version; exact text in workspace.vf.facility_type_single_prompts.",
        "run_ts": "Timestamp of the accepted ai_query response (NULL when failed).",
    }
    run(f"""COMMENT ON TABLE {S}.facility_type IS {q(
        "Facility type per non-shifted facility in workspace.vf.facilities_clean (one row per unique_id). "
        "Classified by ai_query (" + MODEL + ", temperature 0), one facility per call, prompt ft_single_v1; "
        "inputs: name, description truncated to 400 chars, distinct specialties. Labels are model claims, "
        "not verified facts: use confidence and failed. Raw responses: facility_type_single_raw; per-row "
        "validation: facility_type_single_status. Built by scripts/facility_type_single.py; report "
        "reports/facility_type_single.md.")}""")
    for col, txt in comments.items():
        run(f"ALTER TABLE {S}.facility_type ALTER COLUMN {col} COMMENT {q(txt)}")


def stats():
    for stmt in [
        f"SELECT count(*), count(DISTINCT unique_id), sum(cast(failed AS INT)) FROM {S}.facility_type",
        f"SELECT facility_type, count(*) n FROM {S}.facility_type GROUP BY 1 ORDER BY n DESC",
        f"SELECT confidence, count(*) n FROM {S}.facility_type GROUP BY 1 ORDER BY n DESC",
        f"SELECT ownership, count(*) n FROM {S}.facility_type GROUP BY 1 ORDER BY n DESC",
        f"SELECT is_single_specialty, count(*) n FROM {S}.facility_type GROUP BY 1 ORDER BY n DESC",
        f"SELECT count_if(size(split(trim(reason), '\\\\s+')) > 12) FROM {S}.facility_type",
        f"""SELECT name, facility_type, reason FROM {S}.facility_type WHERE confidence = 'low'
            AND NOT failed ORDER BY md5(concat('low', unique_id)) LIMIT 10""",
    ]:
        print("--", stmt.split("FROM")[0].strip()[:80])
        for r in run(stmt, fetch=True):
            print("  ", tuple(r))


def facility_type_main():
    """The 'full' command: every non-shifted facility, one retry, build facility_type, print stats."""
    t0 = time.time()
    run_phase("full")
    build_output()
    print("full total seconds", round(time.time() - t0, 1))
    stats()

In [0]:
print(PROMPT)
if REBUILD and RUN_LLM:
    facility_type_main()
else:
    stats()   # read-only summary of workspace.vf.facility_type

You classify records of Indian healthcare facilities by FACILITY TYPE.

The facility below is given as JSON with "name",
"description" (first 400 characters) and "specialties" (a list of tags).

How to weigh the evidence:
- The NAME and the DESCRIPTION lead. Decide from them first.
- "specialties" is over-assigned and unreliable: generic tags such as
  internalMedicine and familyMedicine appear even on dental clinics and hearing
  clinics. Use specialties ONLY to break a tie when the name and description
  do not decide.

facility_type - exactly one of:
  multispecialty_hospital - hospital with many specialties ("multi speciality", "super speciality")
  general_hospital_or_nursing_home - general hospital or nursing home without one specialty focus
  medical_college_teaching_hospital - medical college, institute of medical sciences, teaching hospital
  government_health_centre - PHC, CHC, sub-centre, district/taluk/civil/government hospital, ESI dispensary
  maternity_women_children - m

## Step 4 — Tag each sentence with the service it proves

**What.** For every distinct sentence the model answers three separate questions:
1. `is_claim` — does it assert that this facility offers a service? Biographies, fees, awards, research findings, buildings and training programmes are not claims.
2. `services` — which services, multi-label, from a fixed list (49 labels, grown to 60 with prompt `pa_single_v2` after studying the claims that fitted no label; anything else is `other_clinical` with a suggested name).
3. `strength` — `explicit` (concrete capacity: beds, machines, units), `stated` (named plainly) or `weak` (only implied).

Every returned label is checked against the allowed list; invalid answers get one retry. Sentences that occur only in dental clinics are labelled `dental` by rule, without the model. A keyword tier (dialysis, delivery, NICU, cardiac surgery, MRI, burns, ICU/emergency) is tagged first, then hospital sentences, then the rest.

**Why.** Services exist only in free text; a fixed list makes matching exact and inspectable, and the sentence is the unit that can be quoted and checked.

**Reads** `distinct_sentences`, `facility_sentences`, `facility_type`. **Writes** `sentence_tags` (one row per distinct sentence). Source: `scripts/pilot_a_tagger.py` (prompt), `scripts/tag_sentences.py` (full run), `scripts/retag_new_labels.py` (re-tag with 60 labels).

Measured: 269,731 of 271,262 distinct sentences tagged; 188,538 are real service claims. Dental-only rule rows: 32,147. Re-tag: 31,491 sentences, 259 kept their v1 tag after a failed retry.

In [0]:
# scripts/pilot_a_tagger.py: the 49 labels and the validated prompt (pa_v1 batch form, pa_single_v1 one-sentence form)
import textwrap

SERVICES_V1 = [
    "dialysis", "caesarean_section", "normal_delivery", "high_risk_obstetrics",
    "nicu", "picu", "icu_critical_care", "ventilator_support",
    "cardiac_surgery", "interventional_cardiology", "stroke_care",
    "mri", "ct_scan", "xray", "ultrasound", "mammography", "nuclear_imaging",
    "burn_care", "trauma_emergency", "ambulance",
    "chemotherapy", "radiotherapy", "cancer_surgery",
    "orthopedic_surgery", "joint_replacement", "neurosurgery", "spine_surgery",
    "general_surgery", "gi_endoscopy", "pulmonology_respiratory", "transplant",
    "blood_bank", "laboratory_pathology", "dental", "eye_surgery",
    "physiotherapy_rehab", "mental_health", "outpatient_opd",
    "infertility_ivf", "dermatology", "ent", "ophthalmology_medical",
    "pediatrics", "obstetrics_maternity", "cosmetic_aesthetic",
    "prosthetics_orthotics", "nephrology_renal", "gastroenterology",
    "other_clinical",
]
ALLOWED = set(SERVICES_V1)
STRENGTHS = {"explicit", "stated", "weak", "na"}

# Copied from scripts/classify_eval.py PROMPT with exactly two changes:
#  (1) the education/training paragraph now says is_claim false;
#  (2) other_clinical objects must carry "suggested_name" (<= 4 words).
PROMPT = textwrap.dedent(f"""\
    You label snippets scraped from Indian hospital websites.

    For EACH numbered snippet decide which clinical service the snippet is
    EVIDENCE THAT THE FACILITY OFFERS.

    Allowed labels: {", ".join(SERVICES_V1)}

    Set "is_claim": false when the snippet does not assert that this facility
    offers a service. Research findings, statistics, doctor biographies, fees,
    founding dates, buildings, accommodation and awards all get is_claim false,
    even when they mention clinical words. When is_claim is false, "services"
    MUST be [] and "strength" MUST be "na".

    Use "other_clinical" when it IS a service claim but no allowed label fits.
    Whenever "services" contains "other_clinical", the object MUST also contain
    "suggested_name": a short service name of at most 4 words. Other objects
    may omit "suggested_name" or set it to null.

    Note: lung function tests, spirometry, bronchoscopy, ventilation and sleep
    studies are all "pulmonology_respiratory". A general emergency department or
    casualty ward is "trauma_emergency", NOT any obstetric label.

    Also return "strength":
      "explicit" - names the service with concrete detail (beds, machines, units)
      "stated"   - names the service plainly
      "weak"     - only implies it

    A snippet may evidence SEVERAL services - return all of them.
    "imaging including USG, CT and MRI" -> ["ultrasound","ct_scan","mri"].

    Education, training, fellowship and degree programmes get is_claim false
    (services [] and strength "na"), even in a clinical subject.

    Cosmetic and aesthetic procedures are "cosmetic_aesthetic", never cancer_surgery.
    Breast implants, cupping, body contouring and hair removal are cosmetic_aesthetic.

    Orthodontics, aligners, braces and anything about teeth is "dental",
    NEVER orthopedic_surgery. Orthodontic and orthopaedic are different words.

    Use ONLY labels from the allowed list. Never invent a label.

    Strength calibration:
    - A bare list of department names is "weak" for each department, not "explicit".
    - "Has N specialists on staff" is "weak" - staffing is not a service claim detail.
    - "explicit" needs concrete capacity: beds, machines, units, theatres, counts.

    Rules:
    - Return EXACTLY one object per snippet, same order, nothing omitted.
    - Echo the first three words of the snippet in "echo" so alignment is checkable.
    - Reply with ONLY a JSON array of
      {{"i":<index>,"echo":"<first three words>","is_claim":<true|false>,
       "services":[<labels>],"strength":"explicit"|"stated"|"weak"|"na",
       "suggested_name":"<only when other_clinical>"}}
    - When is_claim is true, "services" must contain at least one allowed label.

    Snippets:
    """)

# One-sentence variant (pa_single_v1). Only adaptation: a single JSON object for one
# snippet instead of a numbered array; "i" and "echo" dropped. Everything else unchanged.
PROMPT_SINGLE_VERSION = "pa_single_v1"
_NEW_RULES = textwrap.dedent("""\
    Rules:
    - Reply with ONLY a JSON object
      {"is_claim":<true|false>,
       "services":[<labels>],"strength":"explicit"|"stated"|"weak"|"na",
       "suggested_name":"<only when other_clinical>"}
    - When is_claim is true, "services" must contain at least one allowed label.

    Snippet:
    """)
PROMPT_SINGLE = (PROMPT[: PROMPT.index("Rules:\n")]
                 .replace("For EACH numbered snippet decide", "For the snippet decide")
                 + _NEW_RULES)

print(PROMPT_SINGLE)

You label snippets scraped from Indian hospital websites.

For the snippet decide which clinical service the snippet is
EVIDENCE THAT THE FACILITY OFFERS.

Allowed labels: dialysis, caesarean_section, normal_delivery, high_risk_obstetrics, nicu, picu, icu_critical_care, ventilator_support, cardiac_surgery, interventional_cardiology, stroke_care, mri, ct_scan, xray, ultrasound, mammography, nuclear_imaging, burn_care, trauma_emergency, ambulance, chemotherapy, radiotherapy, cancer_surgery, orthopedic_surgery, joint_replacement, neurosurgery, spine_surgery, general_surgery, gi_endoscopy, pulmonology_respiratory, transplant, blood_bank, laboratory_pathology, dental, eye_surgery, physiotherapy_rehab, mental_health, outpatient_opd, infertility_ivf, dermatology, ent, ophthalmology_medical, pediatrics, obstetrics_maternity, cosmetic_aesthetic, prosthetics_orthotics, nephrology_renal, gastroenterology, other_clinical

Set "is_claim": false when the snippet does not assert that this facility
of

In [0]:
# scripts/tag_sentences.py (prompt pa_single_v1, one sentence per ai_query call)
import concurrent.futures as cf
import datetime as dt
from collections import Counter

LOGDIR = LOG_ROOT / "sentence_tagging"
CHUNK = 10_000
WRITE_BATCH = 2_000

KEYWORDS = {
    "dialysis": ["dialysis", "haemodialysis", "hemodialysis", "renal replacement", "nephrolog"],
    "caesarean_delivery": ["caesarean", "cesarean", "c-section", "lscs", "delivery", "labour",
                           "labor room", "obstetric", "maternity"],
    "nicu": ["nicu", "neonatal", "newborn", "neonat"],
    "cardiac_surgery": ["cardiac surgery", "cardiothoracic", "bypass", "cabg", "valve",
                        "open heart"],
    "mri": ["mri", "magnetic resonance"],
    "burn_care": ["burn"],
    "icu_emergency": ["icu", "intensive care", "critical care", "emergency", "casualty",
                      "trauma", "ventilator"],
}
# Match on sentence_key (already lowercase) with a leading word boundary only, so prefixes
# like "nephrolog"/"neonat"/"burn" still match "nephrology"/"neonates"/"burns" but "icu"
# does not match "particular".
KEYWORD_REGEX = r"\b(" + "|".join(
    k.replace("-", r"\-") for ks in KEYWORDS.values() for k in ks) + ")"
HOSPITAL_TYPES = ["multispecialty_hospital", "general_hospital_or_nursing_home",
                  "medical_college_teaching_hospital", "government_health_centre",
                  "maternity_women_children", "pediatric", "cardiac", "cancer",
                  "orthopedic_spine", "neuro", "gastro_liver", "kidney_urology"]


def now() -> str:
    return dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")


def log(kind: str, rec: dict):
    LOGDIR.mkdir(parents=True, exist_ok=True)
    rec = {"ts": now(), **rec}
    with open(LOGDIR / f"{kind}.jsonl", "a") as f:
        f.write(json.dumps(rec) + "\n")
    print(kind, json.dumps(rec), flush=True)


# ---------------------------------------------------------------- credit guard
# Standing rule (CLAUDE.md "Credits and quota"): quota/credit exhaustion is a pause, not a
# failure. Stop at once, write nothing for the affected chunk, exit cleanly.
CREDIT_RE = re.compile(r"quota|credit|billing|limit exceeded|REQUEST_LIMIT_EXCEEDED|"
                       r"RESOURCE_EXHAUSTED|PAYMENT|insufficient", re.I)
BULK_SHARE = 0.30


class CreditsExhausted(Exception):
    pass


def credit_check(where: str, errs: list):
    """errs: errorMessage per raw row (None when ok). Raise CreditsExhausted on (a) or (b)."""
    msgs = [e for e in errs if e]
    hit = next((e for e in msgs if CREDIT_RE.search(e)), None)
    if hit:
        raise CreditsExhausted(f"{where}: errorMessage matches credit pattern: {hit}")
    if errs and msgs:
        top, n = Counter(msgs).most_common(1)[0]
        if n > BULK_SHARE * len(errs):
            raise CreditsExhausted(f"{where}: {n}/{len(errs)} rows share errorMessage: {top}")


def run(label: str, query: str, params=None, fetch=True):
    """One statement on a fresh connection (fresh token every statement)."""
    t0 = time.time()
    try:
        with connect() as c:
            with c.cursor() as cur:
                cur.execute(query, params) if params is not None else cur.execute(query)
                rows = cur.fetchall() if fetch else None
    except Exception as e:  # noqa: BLE001  (c) warehouse/statement errors with credit text
        if CREDIT_RE.search(str(e)):
            raise CreditsExhausted(f"statement {label}: {e}") from e
        raise
    log("statements", {"label": label, "seconds": round(time.time() - t0, 2)})
    return rows


# ---------------------------------------------------------------- tables

def ensure_tables():
    run("create_sentence_tags", f"""
        CREATE TABLE IF NOT EXISTS {VF}.sentence_tags (
          sentence_id STRING, is_claim BOOLEAN, services ARRAY<STRING>, strength STRING,
          suggested_name STRING, prompt_version STRING, chunk_id STRING, failed BOOLEAN,
          fail_reason STRING, tagged_at TIMESTAMP)""", fetch=False)
    run("create_sentence_tags_raw", f"""
        CREATE TABLE IF NOT EXISTS {VF}.sentence_tags_raw (
          sentence_id STRING, chunk_id STRING, attempt INT, part INT,
          resp STRUCT<result: STRING, errorMessage: STRING>, created_at TIMESTAMP)""",
        fetch=False)
    run("create_st_chunk_members", f"""
        CREATE TABLE IF NOT EXISTS {VF}.st_chunk_members (
          chunk_id STRING, sentence_id STRING, part INT, created_at TIMESTAMP)""", fetch=False)


PROMPT_VERSION = PROMPT_SINGLE_VERSION  # set by --prompt (pa_single_v1 | pa_compact_v1)


def load_prompt() -> str:
    rows = run("read_prompt", f"SELECT prompt, model FROM {VF}.pa_prompt "
               f"WHERE prompt_version = '{PROMPT_VERSION}'")
    prompt, model = rows[0]
    assert model == MODEL
    if PROMPT_VERSION == PROMPT_SINGLE_VERSION:
        assert prompt == PROMPT_SINGLE, "pa_prompt differs from pilot_a_tagger"
    return prompt


# ---------------------------------------------------------------- step 1

def step1():
    ensure_tables()
    run("create_st_type_sets", f"""
        CREATE OR REPLACE TABLE {VF}.st_type_sets AS
        WITH occ AS (
          SELECT DISTINCT fs.sentence_key, coalesce(ft.facility_type, '(no_type_row)') AS ftype
          FROM {VF}.facility_sentences fs
          LEFT JOIN {VF}.facility_type ft USING (unique_id)
          WHERE NOT fs.is_placeholder)
        SELECT d.sentence_id, d.sentence_key, array_sort(collect_set(o.ftype)) AS types
        FROM {VF}.distinct_sentences d JOIN occ o USING (sentence_key)
        GROUP BY d.sentence_id, d.sentence_key""", fetch=False)
    tot = run("count_type_sets", f"""
        SELECT count(*), (SELECT count(*) FROM {VF}.distinct_sentences)
        FROM {VF}.st_type_sets""")[0]
    single = run("single_type_counts", f"""
        SELECT types[0], count(*) FROM {VF}.st_type_sets WHERE size(types) = 1
        GROUP BY 1 ORDER BY 2 DESC""")
    multi = run("multi_type_count",
                f"SELECT count(*) FROM {VF}.st_type_sets WHERE size(types) >= 2")[0][0]
    log("step1", {"type_sets_rows": tot[0], "distinct_sentences": tot[1],
                  "single_type": {t: n for t, n in single}, "two_plus_types": multi})

    run("insert_dental_rule", f"""
        INSERT INTO {VF}.sentence_tags
        SELECT s.sentence_id, CAST(NULL AS BOOLEAN), array('dental'), CAST(NULL AS STRING),
               CAST(NULL AS STRING), 'rule_dental_only', 'rule_dental_only', false,
               CAST(NULL AS STRING), current_timestamp()
        FROM {VF}.st_type_sets s
        WHERE s.types = array('dental')
          AND NOT EXISTS (SELECT 1 FROM {VF}.sentence_tags t WHERE t.sentence_id = s.sentence_id)
    """, fetch=False)

    hosp = ", ".join(f"'{t}'" for t in HOSPITAL_TYPES)
    run("create_st_queue", f"""
        CREATE OR REPLACE TABLE {VF}.st_queue AS
        SELECT d.sentence_id, d.example_sentence, d.n_facilities,
               CASE WHEN d.sentence_key RLIKE ? THEN 1
                    WHEN arrays_overlap(s.types, array({hosp})) THEN 2
                    ELSE 3 END AS priority,
               row_number() OVER (ORDER BY
                 CASE WHEN d.sentence_key RLIKE ? THEN 1
                      WHEN arrays_overlap(s.types, array({hosp})) THEN 2 ELSE 3 END,
                 d.n_facilities DESC, d.sentence_id) AS ord
        FROM {VF}.distinct_sentences d
        LEFT JOIN {VF}.st_type_sets s USING (sentence_id)
        WHERE s.types IS NULL OR s.types != array('dental')""",
        [KEYWORD_REGEX, KEYWORD_REGEX], fetch=False)
    q = run("queue_counts", f"""
        SELECT priority, count(*) FROM {VF}.st_queue GROUP BY 1 ORDER BY 1""")
    dental = run("dental_rows", f"SELECT count(*) FROM {VF}.sentence_tags "
                 "WHERE prompt_version = 'rule_dental_only'")[0][0]
    remaining = remaining_count()
    log("step1", {"dental_rule_rows": dental, "queue_by_priority": {p: n for p, n in q},
                  "remaining_to_tag": remaining, "keyword_regex": KEYWORD_REGEX})


def next_tier():
    """Lowest priority tier that still has untagged rows not yet assigned to a chunk."""
    return run("next_tier", f"""
        SELECT min(priority) FROM {VF}.st_queue q
        WHERE NOT EXISTS (SELECT 1 FROM {VF}.sentence_tags t WHERE t.sentence_id = q.sentence_id)
          AND NOT EXISTS (SELECT 1 FROM {VF}.st_chunk_members m WHERE m.sentence_id = q.sentence_id)
    """)[0][0]


def remaining_count() -> int:
    return run("remaining", f"""
        SELECT count(*) FROM {VF}.st_queue q
        WHERE NOT EXISTS (SELECT 1 FROM {VF}.sentence_tags t WHERE t.sentence_id = q.sentence_id)
    """)[0][0]


# ---------------------------------------------------------------- tagging core

def validate(res, err):
    """Return (obj, fail_reason, over_length)."""
    if err or res is None:
        return None, "ai_query_error", False
    try:
        o = json.loads(res[res.index("{"): res.rindex("}") + 1])
    except Exception:  # noqa: BLE001
        return None, "unparseable", False
    if not isinstance(o, dict):
        return None, "unparseable", False
    claim, svc, st = o.get("is_claim"), o.get("services"), o.get("strength")
    if not isinstance(claim, bool):
        return None, "is_claim_not_bool", False
    if not isinstance(svc, list):
        return None, "services_not_list", False
    if any(s not in ALLOWED for s in svc):
        return None, "unknown_label", False
    if st not in STRENGTHS:
        return None, "bad_strength", False
    if claim is False and (svc or st != "na"):
        return None, "inconsistent_false", False
    if claim is True and not svc:
        # Not a failure (user rule): unlabelled claim -> other_clinical, flagged.
        o["services"], o["_unlabelled"] = ["other_clinical"], True
        return o, None, False
    name = o.get("suggested_name")
    if "other_clinical" in svc:
        if not isinstance(name, str) or not name.strip():
            return None, "missing_suggested_name", False
        return o, None, len(name.split()) > 4
    return o, None, False


def create_chunk(chunk_id: str, n: int, parts: int):
    """Members = next n untagged queue rows in priority order, not already in a chunk."""
    existing = run("chunk_exists", f"SELECT count(*) FROM {VF}.st_chunk_members "
                   "WHERE chunk_id = ?", [chunk_id])[0][0]
    if existing:
        return existing
    tier = next_tier()
    run(f"{chunk_id}_members", f"""
        INSERT INTO {VF}.st_chunk_members
        SELECT ?, sentence_id, CAST(pmod(rn - 1, {parts}) AS INT), current_timestamp()
        FROM (
          SELECT q.sentence_id, row_number() OVER (ORDER BY q.ord) AS rn
          FROM {VF}.st_queue q
          WHERE NOT EXISTS (SELECT 1 FROM {VF}.sentence_tags t WHERE t.sentence_id = q.sentence_id)
            AND NOT EXISTS (SELECT 1 FROM {VF}.st_chunk_members m WHERE m.sentence_id = q.sentence_id)
            AND q.priority = {tier}
          ORDER BY q.ord LIMIT {n})""", [chunk_id], fetch=False)
    return run("chunk_count", f"SELECT count(*) FROM {VF}.st_chunk_members "
               "WHERE chunk_id = ?", [chunk_id])[0][0]


def ai_query_part(prompt: str, chunk_id: str, attempt: int, part, ids_filter: str):
    """One INSERT ... ai_query statement. part=None covers all parts."""
    part_sql = "" if part is None else f"AND m.part = {int(part)}"
    label = f"{chunk_id}_a{attempt}" + ("" if part is None else f"_p{part}")
    t0 = time.time()
    run(label, f"""
        INSERT INTO {VF}.sentence_tags_raw
        SELECT m.sentence_id, m.chunk_id, {attempt}, m.part,
               ai_query('{MODEL}', concat(?, q.example_sentence),
                        modelParameters => named_struct('max_tokens', 300, 'temperature', 0.0),
                        failOnError => false),
               current_timestamp()
        FROM {VF}.st_chunk_members m JOIN {VF}.st_queue q USING (sentence_id)
        WHERE m.chunk_id = ? {part_sql}
          AND NOT EXISTS (SELECT 1 FROM {VF}.sentence_tags t WHERE t.sentence_id = m.sentence_id)
          {ids_filter}""", [prompt, chunk_id], fetch=False)
    return time.time() - t0


def read_raw(chunk_id: str, attempt: int):
    rows = run(f"{chunk_id}_read_a{attempt}", f"""
        SELECT r.sentence_id, r.resp.result, r.resp.errorMessage
        FROM {VF}.sentence_tags_raw r
        WHERE r.chunk_id = ? AND r.attempt = {attempt}
          AND NOT EXISTS (SELECT 1 FROM {VF}.sentence_tags t WHERE t.sentence_id = r.sentence_id)
    """, [chunk_id])
    return {sid: (res, err) for sid, res, err in rows}


def write_tags(chunk_id: str, rows: list[dict]):
    for i in range(0, len(rows), WRITE_BATCH):
        payload = json.dumps(rows[i: i + WRITE_BATCH], ensure_ascii=False)
        run(f"{chunk_id}_write", f"""
            INSERT INTO {VF}.sentence_tags
            (sentence_id, is_claim, services, strength, suggested_name, prompt_version,
             chunk_id, failed, fail_reason, tagged_at, unlabelled_claim)
            SELECT sentence_id, is_claim, services, strength, suggested_name, prompt_version,
                   chunk_id, failed, fail_reason, current_timestamp(), unlabelled_claim
            FROM (SELECT inline(from_json(?, 'array<struct<sentence_id:string,
                  is_claim:boolean,services:array<string>,strength:string,
                  suggested_name:string,prompt_version:string,chunk_id:string,
                  failed:boolean,fail_reason:string,unlabelled_claim:boolean>>')))""",
                [payload], fetch=False)


def to_row(sid, chunk_id, o, fail):
    ok = fail is None
    svc = o.get("services") if ok else None
    return {"sentence_id": sid, "chunk_id": chunk_id, "prompt_version": PROMPT_VERSION,
            "is_claim": o.get("is_claim") if ok else None, "services": svc,
            "strength": o.get("strength") if ok else None,
            "suggested_name": (o.get("suggested_name") if ok and "other_clinical" in svc
                               and isinstance(o.get("suggested_name"), str) else None),
            "unlabelled_claim": bool(ok and o.get("_unlabelled")),
            "failed": not ok, "fail_reason": fail}


def run_chunk(prompt: str, chunk_id: str, n: int, mode: str, retry: bool = True):
    parts = 4 if mode == "conc4" else 1
    members = create_chunk(chunk_id, n, parts)
    t0 = time.time()
    # No self-read of sentence_tags_raw inside the INSERT (concurrent appends would conflict);
    # resumability: skip parts whose attempt-1 rows already exist (each INSERT is atomic).
    # A part counts as done only if it has error-free raw rows for all its members; otherwise
    # (e.g. after a credit stop) its raw rows are deleted and the part is queried again.
    stat = run("raw_parts", f"""
        SELECT m.part, count(*) AS n_mem,
               count(r.sentence_id) AS n_raw, count(r.resp.errorMessage) AS n_err
        FROM {VF}.st_chunk_members m
        LEFT JOIN {VF}.sentence_tags_raw r
          ON r.sentence_id = m.sentence_id AND r.chunk_id = m.chunk_id AND r.attempt = 1
        WHERE m.chunk_id = ? GROUP BY m.part""", [chunk_id])
    todo = []
    for p, n_mem, n_raw, n_err in stat:
        if n_raw and n_raw == n_mem and n_err == 0:
            continue
        if n_raw:
            run("raw_part_reset", f"DELETE FROM {VF}.sentence_tags_raw WHERE chunk_id = ? "
                f"AND attempt = 1 AND part = {int(p)}", [chunk_id], fetch=False)
        todo.append(p)
    if parts == 1:
        q_secs = ai_query_part(prompt, chunk_id, 1, None, "") if todo else 0.0
    else:
        with cf.ThreadPoolExecutor(parts) as ex:
            list(ex.map(lambda p: ai_query_part(prompt, chunk_id, 1, p, ""), todo))
        q_secs = time.time() - t0

    raw1 = read_raw(chunk_id, 1)
    credit_check(chunk_id, [e for _, e in raw1.values()])  # before any row is written
    first = {sid: validate(res, err) for sid, (res, err) in raw1.items()}
    failed1 = [sid for sid, (_, f, _) in first.items() if f]
    reasons1 = Counter(f for _, f, _ in first.values() if f)

    retry_secs, reasons2, final = 0.0, Counter(), dict(first)
    if failed1 and retry:
        ids = ", ".join(f"'{s}'" for s in failed1)
        retry_secs = ai_query_part(prompt, chunk_id, 2, None, f"AND m.sentence_id IN ({ids})")
        raw2 = read_raw(chunk_id, 2)
        for sid in failed1:
            if sid in raw2:
                final[sid] = validate(*raw2[sid])
        reasons2 = Counter(final[s][1] for s in failed1 if final[s][1])
    rows = [to_row(sid, chunk_id, o, f) for sid, (o, f, _) in final.items()]
    write_tags(chunk_id, rows)
    total = time.time() - t0
    n_rows = len(rows)
    stats = {
        "chunk_id": chunk_id, "mode": mode, "members": members, "rows_written": n_rows,
        "ai_query_seconds": round(q_secs, 1),
        "sentences_per_s": round(len(raw1) / q_secs, 2) if q_secs else None,
        "retry_rows": len(failed1), "retry_seconds": round(retry_secs, 1),
        "fail_first": dict(reasons1), "fail_after_retry": dict(reasons2),
        "failed_rows": sum(r["failed"] for r in rows),
        "over_length_names": sum(1 for o, f, ol in final.values() if ol and f is None),
        "chunk_total_seconds": round(total, 1),
    }
    log("chunks", stats)
    return stats


def step3(mode: str, max_chunks, until_tier=None):
    ensure_tables()
    prompt = load_prompt()
    done = run("chunk_ids", f"SELECT DISTINCT chunk_id FROM {VF}.st_chunk_members "
               "WHERE chunk_id LIKE 'c%'")
    nums = sorted(int(c[0][1:]) for c in done)
    # finish any half-done chunk first
    for k in nums:
        cid = f"c{k:03d}"
        left = run("chunk_left", f"""
            SELECT count(*) FROM {VF}.st_chunk_members m WHERE m.chunk_id = ?
              AND NOT EXISTS (SELECT 1 FROM {VF}.sentence_tags t
                              WHERE t.sentence_id = m.sentence_id)""", [cid])[0][0]
        if left:
            run_chunk(prompt, cid, CHUNK, mode, retry=False)
    k = (nums[-1] + 1) if nums else 1
    ran = 0
    while (max_chunks is None or ran < max_chunks):
        tier = next_tier()
        if tier is None or (until_tier is not None and tier > until_tier):
            break
        t0 = time.time()
        s = run_chunk(prompt, f"c{k:03d}", CHUNK, mode, retry=False)
        rem = remaining_count()
        rate = s["rows_written"] / (time.time() - t0)
        log("progress", {"chunk_id": s["chunk_id"], "tier": tier, "remaining": rem,
                         "end_to_end_rate": round(rate, 2),
                         "projected_hours_left": round(rem / rate / 3600, 2)})
        k += 1
        ran += 1


def final_retry():
    """All failed pa_single_v1 rows rerun once in ONE statement; replace their rows."""
    prompt = load_prompt()
    n = run("final_members", f"SELECT count(*) FROM {VF}.st_chunk_members "
            "WHERE chunk_id = 'final_retry'")[0][0]
    if not n:
        run("final_members_insert", f"""
            INSERT INTO {VF}.st_chunk_members
            SELECT 'final_retry', sentence_id, 0, current_timestamp()
            FROM {VF}.sentence_tags WHERE failed AND prompt_version = '{PROMPT_SINGLE_VERSION}'
        """, fetch=False)
    have, n_err = run("final_raw", f"SELECT count(*), count(resp.errorMessage) FROM "
                      f"{VF}.sentence_tags_raw WHERE chunk_id = 'final_retry'")[0]
    if have and n_err:  # e.g. an earlier credit stop: query again
        run("final_raw_reset", f"DELETE FROM {VF}.sentence_tags_raw "
            "WHERE chunk_id = 'final_retry'", fetch=False)
        have = 0
    t0 = time.time()
    if not have:
        run("final_retry_ai_query", f"""
            INSERT INTO {VF}.sentence_tags_raw
            SELECT m.sentence_id, 'final_retry', 2, 0,
                   ai_query('{MODEL}', concat(?, q.example_sentence),
                            modelParameters => named_struct('max_tokens', 300, 'temperature', 0.0),
                            failOnError => false),
                   current_timestamp()
            FROM {VF}.st_chunk_members m JOIN {VF}.st_queue q USING (sentence_id)
            WHERE m.chunk_id = 'final_retry'""", [prompt], fetch=False)
    secs = time.time() - t0
    raw = run("final_read", f"""SELECT sentence_id, resp.result, resp.errorMessage
        FROM {VF}.sentence_tags_raw WHERE chunk_id = 'final_retry'""")
    credit_check("final_retry", [e for _, _, e in raw])  # before any row is written
    res = {sid: validate(r, e) for sid, r, e in raw}
    # only replace rows that are still failed (idempotent if rerun)
    still = {sid for (sid,) in run("final_still", f"""SELECT t.sentence_id FROM {VF}.sentence_tags t
        JOIN {VF}.st_chunk_members m ON m.sentence_id = t.sentence_id AND m.chunk_id = 'final_retry'
        WHERE t.failed AND t.chunk_id != 'final_retry'""")}
    rows = [to_row(sid, "final_retry", o, f) for sid, (o, f, _) in res.items() if sid in still]
    if rows:
        run("final_delete_failed", f"""DELETE FROM {VF}.sentence_tags WHERE failed
            AND chunk_id != 'final_retry' AND sentence_id IN
            (SELECT sentence_id FROM {VF}.st_chunk_members WHERE chunk_id = 'final_retry')""",
            fetch=False)
        write_tags("final_retry", rows)
    log("chunks", {"chunk_id": "final_retry", "rows": len(res), "ai_query_seconds": round(secs, 1),
                   "replaced": len(rows), "still_failed": sum(r["failed"] for r in rows),
                   "fail_reasons": dict(Counter(f for _, f, _ in res.values() if f))})


def comment():
    tc = ("Service tags for every sentence in workspace.vf.distinct_sentences (key sentence_id). "
          "LLM rows: prompt_version pa_single_v1 (exact text in workspace.vf.pa_prompt), model "
          "databricks-meta-llama-3-3-70b-instruct, temperature 0, one sentence per ai_query "
          "call; raw responses in workspace.vf.sentence_tags_raw. Rule rows: prompt_version "
          "rule_dental_only = sentences that appear only in facilities typed dental in "
          "workspace.vf.facility_type; services = [dental], is_claim/strength NULL (not judged). "
          "unlabelled_claim = true marks is_claim true with no label (stored as other_clinical, "
          "not a failure). From chunk c007 ai_query uses max_tokens 300. "
          "failed = true when the response failed validation on the first call and on one retry "
          "(per chunk for tp_*, one final statement chunk_id final_retry for the rest). "
          "Rows are appended per chunk; failed rows are replaced by the final_retry pass. Resume with: python scripts/tag_sentences.py resume (skips sentence_ids "
          "already present, finishes half-done chunks from workspace.vf.st_chunk_members).")
    cols = {
        "sentence_id": "Key; joins to workspace.vf.distinct_sentences.sentence_id.",
        "is_claim": "Model: does the sentence assert this facility offers a service. NULL for "
                    "rule rows and failed rows.",
        "services": "Model labels from the 49-label list in pa_single_v1 ([] when is_claim false). "
                    "Rule rows: [dental]. NULL when failed.",
        "strength": "Model: explicit | stated | weak | na. NULL for rule rows and failed rows.",
        "suggested_name": "Model's short name when services contains other_clinical (asked for "
                          "<= 4 words, not enforced). NULL otherwise.",
        "prompt_version": "pa_single_v1 (LLM) or rule_dental_only (rule).",
        "chunk_id": "Run chunk: rule_dental_only, tp_single, tp_conc4 (throughput test), c001... (step 3, one tier per chunk), final_retry.",
        "failed": "true if the response failed validation on the first call and the one retry "
                  "(final_retry pass).",
        "fail_reason": "Validation failure of the final attempt (unparseable, unknown_label, "
                       "claim_without_label, ...). NULL when not failed.",
        "tagged_at": "Insert timestamp.",
        "unlabelled_claim": "true when the model said is_claim true but returned no label; stored "
                            "as services [other_clinical], strength as returned, failed false "
                            "(user rule, applied from chunk c007 and converted in place for "
                            "earlier rows). false for all other rows.",
    }
    run("table_comment", f"COMMENT ON TABLE {VF}.sentence_tags IS ?", [tc], fetch=False)
    for c, txt in cols.items():
        run(f"col_{c}", f"ALTER TABLE {VF}.sentence_tags ALTER COLUMN {c} COMMENT ?", [txt],
            fetch=False)


def credits_stop(err: Exception):
    try:
        done = run("credit_done", f"SELECT count(*) FROM {VF}.sentence_tags "
                   f"WHERE prompt_version = '{PROMPT_SINGLE_VERSION}'")[0][0]
        left = remaining_count()
    except Exception as e:  # noqa: BLE001
        done = left = f"unknown ({type(e).__name__})"
    msg = (f"CREDITS EXHAUSTED {now()}\nerror: {err}\nLLM rows done: {done}\n"
           f"sentences left: {left}\nNothing was written for the affected chunk. "
           "Resume after top-up: python scripts/tag_sentences.py resume --mode conc4\n")
    print(msg, flush=True)
    LOGDIR.mkdir(parents=True, exist_ok=True)
    (LOGDIR / "CREDITS_EXHAUSTED.txt").write_text(msg)


def seed_prompt():
    """pa_prompt holds the exact prompt text per version (first written by pilot_a_tagger.py)."""
    run("mk_pa_prompt", f"""CREATE TABLE IF NOT EXISTS {VF}.pa_prompt (prompt_version STRING, model STRING,
        temperature DOUBLE, prompt STRING, created_at TIMESTAMP)""", fetch=False)
    have = run("pa_prompt_have", f"SELECT count(*) FROM {VF}.pa_prompt WHERE prompt_version = ?",
               [PROMPT_SINGLE_VERSION])[0][0]
    if not have:
        run("pa_prompt_insert", f"INSERT INTO {VF}.pa_prompt VALUES (?, ?, 0.0, ?, current_timestamp())",
            [PROMPT_SINGLE_VERSION, MODEL, PROMPT_SINGLE], fetch=False)


def tag_main(cmd, mode="conc4", max_chunks=None, until=None):
    try:
        _tag_main(cmd, mode, max_chunks, until)
    except CreditsExhausted as e:
        credits_stop(e)


def _tag_main(cmd, mode, max_chunks, until):
    if cmd == "step1":
        seed_prompt()
        step1()
    elif cmd == "step3":
        step3(mode, max_chunks, until)
    elif cmd == "resume":
        step3(mode, max_chunks)
        if next_tier() is None:
            final_retry()
            comment()
            log("progress", {"done": True, "remaining": remaining_count()})
    elif cmd == "final_retry":
        final_retry()
    elif cmd == "comment":
        comment()
    else:
        raise ValueError(cmd)

In [0]:
if REBUILD and RUN_LLM:
    tag_main("step1")    # type sets, dental-only rule rows, priority queue
    tag_main("resume")   # all tiers in chunks of 10,000 (4 concurrent statements), final retry, comments

In [0]:
# scripts/retag_new_labels.py: re-tag other_clinical / unlabelled claims (and a cardiac set) with 60 labels
# (NEW_SERVICES_V2 is defined in scripts/build_lookups.py; copied here because step 6 comes later)
NEW_SERVICES_V2 = [
    "urology", "diabetes_endocrinology", "health_checkup_vaccination", "cardiac_diagnostics",
    "neurology", "ayush_traditional_medicine", "gynecology", "proctology", "pain_management",
    "vascular_surgery", "cardiology_medical",
]
V1, V2 = "pa_single_v1", "pa_single_v2"
SEED = "retag_v2_seed_1"
TEST_N = 500
PARTS = 4
GATE = 0.95
ALLOWED_V2 = set(SERVICES_V1) | set(NEW_SERVICES_V2)
SCOPE = ("NOT coalesce(failed, false) AND (array_contains(services, 'other_clinical') "
         "OR coalesce(unlabelled_claim, false))")

CARDIAC_TEST_RE = r"(?i)\b(echo|echocardiography|ecg|ekg|holter|tmt|treadmill|stress test|doppler)\b"
CARDIAC_SCOPE = (f"NOT coalesce(failed, false) AND sentence_id IN (SELECT t.sentence_id FROM {VF}.sentence_tags t "
                 f"JOIN {VF}.distinct_sentences d USING (sentence_id) WHERE "
                 "(arrays_overlap(t.services, array('cardiac_surgery', 'interventional_cardiology')) "
                 "AND d.example_sentence RLIKE ?) "
                 "OR (array_contains(t.services, 'cardiac_surgery') AND t.strength = 'weak'))")

LOGDIR = LOG_ROOT / "retag_v2"
ALLOWED = ALLOWED_V2             # validate() reads this global
PROMPT_VERSION = V2              # to_row() stamps this

LABEL_NOTES = """\
Note: urology consultations, kidney and urinary stone treatment (lithotripsy, PCNL, URS,
RIRS), prostate treatment (TURP, HoLEP), cystoscopy, uroflowmetry, circumcision and
andrology are "urology", NOT dialysis or nephrology_renal.
Note: diabetes clinics and diabetes management, insulin pumps, diabetic foot care, thyroid
and hormone (endocrine) treatment are "diabetes_endocrinology". A bare blood sugar test in a
lab list is "laboratory_pathology".
Note: health check-up packages, vaccination and immunisation, cancer screening programmes
and visa or employment medical examinations are "health_checkup_vaccination". A single lab
test is "laboratory_pathology"; antenatal check-ups are "obstetrics_maternity".
Note: echocardiography / 2D echo, treadmill or stress test (TMT), Holter monitoring, EECP and
cardiac Doppler tests are "cardiac_diagnostics". ECG mentioned alone is NOT
cardiac_diagnostics. Angioplasty, angiography and cath lab are "interventional_cardiology".
Note: general heart care - cardiology departments, cardiologist consultations and heart
clinics where no surgery, catheter procedure or heart test is named - is "cardiology_medical".
Open-heart, bypass and valve surgery are "cardiac_surgery"; angiography, angioplasty, stents
and pacemakers are "interventional_cardiology"; echo, ECG, TMT and Holter tests are
"cardiac_diagnostics".
Note: non-surgical brain and nerve care - neurology consultations, epilepsy, headache and
migraine, vertigo, Parkinson's, dementia, neuropathy, and EEG / EMG / nerve conduction
offered as a test service - is "neurology". A bare "EEG machine" line is NOT neurology.
Brain and spine operations are "neurosurgery"; stroke units are "stroke_care".
Note: Ayurveda (including Panchakarma, Shirodhara, Kshar Sutra), homeopathy, naturopathy,
Unani, Siddha and yoga therapy are "ayush_traditional_medicine". An Ayurvedic or homeopathic
treatment for a condition gets ONLY "ayush_traditional_medicine".
Note: non-pregnancy gynaecology - hysteroscopy, colposcopy, Pap smear, PCOS/PCOD, fibroids,
endometriosis, hysterectomy, menopause, contraception, tubectomy, MTP - is "gynecology".
Pregnancy, antenatal care and delivery keep the obstetric labels; IVF is "infertility_ivf".
Note: piles / haemorrhoids, anal fistula, fissure, pilonidal sinus and laser proctology are
"proctology". An AV fistula for dialysis access is "vascular_surgery", NOT proctology.
Note: varicose vein treatment (laser, sclerotherapy, surgery), peripheral arterial surgery
and AV fistula creation for dialysis access are "vascular_surgery". Angioplasty and stents of
the heart are "interventional_cardiology".
Note: pain clinics, nerve blocks and interventional pain procedures for back, joint or
chronic pain are "pain_management".
Note: general medicine, internal medicine, fever and infection treatment are
"outpatient_opd".
"""


def build_prompt_v2(v1: str) -> str:
    old = "nephrology_renal, gastroenterology, other_clinical\n"
    assert v1.count(old) == 1
    new = ("nephrology_renal, gastroenterology, " + ", ".join(NEW_SERVICES_V2)
           + ", other_clinical\n\n" + LABEL_NOTES)
    return v1.replace(old, new)


def retag_load_prompt(version=V2) -> str:
    rows = run("read_prompt", f"SELECT prompt, model FROM {VF}.pa_prompt WHERE prompt_version = ?",
               [version])
    assert rows and rows[0][1] == MODEL, f"{version} missing or wrong model"
    return rows[0][0]


def cmd_prompt():
    v1 = retag_load_prompt(V1)
    v2 = build_prompt_v2(v1)
    have = run("v2_exists", f"SELECT prompt FROM {VF}.pa_prompt WHERE prompt_version = ?", [V2])
    if have and have[0][0] == v2:
        print("pa_single_v2 already stored, identical")
        return
    if have:  # label list extended before any v2 row was written
        used = run("v2_used", f"SELECT count(*) FROM {VF}.sentence_tags WHERE prompt_version = ?", [V2])[0][0]
        assert not used, "pa_single_v2 already used in sentence_tags; make a new version"
        run("v2_delete", f"DELETE FROM {VF}.pa_prompt WHERE prompt_version = ?", [V2], fetch=False)
    run("v2_insert", f"INSERT INTO {VF}.pa_prompt (prompt_version, model, temperature, prompt, created_at) "
        "VALUES (?, ?, 0.0, ?, current_timestamp())", [V2, MODEL, v2], fetch=False)
    print(f"stored pa_single_v2 ({len(v2)} chars)")


def cmd_backup():
    exists = run("bk_exists", "SELECT count(*) FROM workspace.information_schema.tables "
                 "WHERE table_schema = 'vf' AND table_name = 'sentence_tags_backup_v1'")[0][0]
    if not exists:
        run("bk_create", f"""CREATE TABLE {VF}.sentence_tags_backup_v1
            COMMENT 'v1 (pa_single_v1) rows of workspace.vf.sentence_tags that were re-tagged with pa_single_v2 by scripts/retag_new_labels.py: not failed AND (services contains other_clinical OR unlabelled_claim). Snapshot before the MERGE.'
            AS SELECT * FROM {VF}.sentence_tags WHERE {SCOPE}""", fetch=False)
    # cardiac set (added later): append rows not yet in the backup, still in v1 state
    run("bk_cardiac", f"""INSERT INTO {VF}.sentence_tags_backup_v1
        SELECT * FROM {VF}.sentence_tags WHERE prompt_version = '{V1}' AND {CARDIAC_SCOPE}
          AND sentence_id NOT IN (SELECT sentence_id FROM {VF}.sentence_tags_backup_v1)""",
        [CARDIAC_TEST_RE], fetch=False)
    n_c = run("bk_cardiac_n", f"SELECT count(*) FROM {VF}.sentence_tags_backup_v1 WHERE {CARDIAC_SCOPE}",
              [CARDIAC_TEST_RE])[0][0]
    print("cardiac set rows in backup:", n_c)
    n = run("bk_count", f"SELECT count(*), count(DISTINCT sentence_id) FROM {VF}.sentence_tags_backup_v1")[0]
    print("backup rows:", tuple(n))
    return n[0]


def retag_ensure_tables():
    run("mk_members", f"""CREATE TABLE IF NOT EXISTS {VF}.retag_v2_members (
        chunk_id STRING, sentence_id STRING, part INT, created_at TIMESTAMP)""", fetch=False)
    run("mk_raw", f"""CREATE TABLE IF NOT EXISTS {VF}.retag_v2_raw (
        sentence_id STRING, chunk_id STRING, attempt INT, part INT,
        resp STRUCT<result: STRING, errorMessage: STRING>, created_at TIMESTAMP)""", fetch=False)
    for t in ("retag_v2_test", "retag_v2_results"):
        run(f"mk_{t}", f"""CREATE TABLE IF NOT EXISTS {VF}.{t} (
            sentence_id STRING, is_claim BOOLEAN, services ARRAY<STRING>, strength STRING,
            suggested_name STRING, prompt_version STRING, chunk_id STRING, failed BOOLEAN,
            fail_reason STRING, unlabelled_claim BOOLEAN, tagged_at TIMESTAMP)""", fetch=False)


def add_members(chunk_id: str, n: int, order_sql: str):
    have = run("mem_have", f"SELECT count(*) FROM {VF}.retag_v2_members WHERE chunk_id = ?",
               [chunk_id])[0][0]
    if have:
        return have
    run(f"{chunk_id}_members", f"""
        INSERT INTO {VF}.retag_v2_members
        SELECT ?, sentence_id, CAST(pmod(rn - 1, {PARTS}) AS INT), current_timestamp()
        FROM (SELECT b.sentence_id, row_number() OVER (ORDER BY {order_sql}) rn
              FROM {VF}.sentence_tags_backup_v1 b
              WHERE NOT EXISTS (SELECT 1 FROM {VF}.retag_v2_members m WHERE m.sentence_id = b.sentence_id)
              ORDER BY {order_sql} LIMIT {n})""", [chunk_id], fetch=False)
    return run("mem_count", f"SELECT count(*) FROM {VF}.retag_v2_members WHERE chunk_id = ?",
               [chunk_id])[0][0]


def query_part(prompt, chunk_id, attempt, part, extra=""):
    part_sql = "" if part is None else f"AND m.part = {int(part)}"
    run(f"{chunk_id}_a{attempt}_p{part}", f"""
        INSERT INTO {VF}.retag_v2_raw
        SELECT m.sentence_id, m.chunk_id, {attempt}, m.part,
               ai_query('{MODEL}', concat(?, q.example_sentence),
                        modelParameters => named_struct('max_tokens', 300, 'temperature', 0.0),
                        failOnError => false),
               current_timestamp()
        FROM {VF}.retag_v2_members m JOIN {VF}.st_queue q USING (sentence_id)
        WHERE m.chunk_id = ? {part_sql} {extra}""", [prompt, chunk_id], fetch=False)


def query_chunk(prompt, chunk_id):
    """Attempt 1 for every part that lacks complete, error-free raw rows (resumable)."""
    stat = run("raw_parts", f"""
        SELECT m.part, count(*), count(r.sentence_id), count(r.resp.errorMessage)
        FROM {VF}.retag_v2_members m
        LEFT JOIN {VF}.retag_v2_raw r
          ON r.sentence_id = m.sentence_id AND r.chunk_id = m.chunk_id AND r.attempt = 1
        WHERE m.chunk_id = ? GROUP BY m.part""", [chunk_id])
    todo = []
    for p, n_mem, n_raw, n_err in stat:
        if n_raw == n_mem and n_err == 0:
            continue
        if n_raw:
            run("raw_reset", f"DELETE FROM {VF}.retag_v2_raw WHERE chunk_id = ? AND attempt = 1 "
                f"AND part = {int(p)}", [chunk_id], fetch=False)
        todo.append(p)
    t0 = time.time()
    if todo:
        with cf.ThreadPoolExecutor(PARTS) as ex:
            list(ex.map(lambda p: query_part(prompt, chunk_id, 1, p), todo))
    return time.time() - t0, len(todo)


def retag_read_raw(chunk_id, attempt):
    rows = run(f"{chunk_id}_read{attempt}", f"""SELECT sentence_id, resp.result, resp.errorMessage
        FROM {VF}.retag_v2_raw WHERE chunk_id = ? AND attempt = {attempt}""", [chunk_id])
    return {sid: (res, err) for sid, res, err in rows}


def write_rows(table, rows):
    for i in range(0, len(rows), WRITE_BATCH):
        payload = json.dumps(rows[i: i + WRITE_BATCH], ensure_ascii=False)
        run(f"write_{table}", f"""
            INSERT INTO {VF}.{table}
            SELECT sentence_id, is_claim, services, strength, suggested_name, prompt_version,
                   chunk_id, failed, fail_reason, unlabelled_claim, current_timestamp()
            FROM (SELECT inline(from_json(?, 'array<struct<sentence_id:string,
                  is_claim:boolean,services:array<string>,strength:string,
                  suggested_name:string,prompt_version:string,chunk_id:string,
                  failed:boolean,fail_reason:string,unlabelled_claim:boolean>>')))""",
            [payload], fetch=False)


def cmd_test():
    retag_ensure_tables()
    prompt = retag_load_prompt()
    n = add_members("v2_test", TEST_N, f"xxhash64(concat('{SEED}', b.sentence_id))")
    secs, _ = query_chunk(prompt, "v2_test")
    raw = retag_read_raw("v2_test", 1)
    credit_check("v2_test", [e for _, e in raw.values()])
    res = {sid: validate(r, e) for sid, (r, e) in raw.items()}
    rows = [to_row(sid, "v2_test", o, f) for sid, (o, f, _) in res.items()]
    done = run("test_have", f"SELECT count(*) FROM {VF}.retag_v2_test")[0][0]
    if not done:
        write_rows("retag_v2_test", rows)
    valid = sum(not r["failed"] for r in rows)
    share = valid / len(rows)
    svc = Counter(s for r in rows if not r["failed"] for s in r["services"])
    stats = {"members": n, "responses": len(rows), "valid": valid, "valid_share": round(share, 4),
             "gate": GATE, "passed": share >= GATE, "seconds": round(secs, 1),
             "fail_reasons": dict(Counter(r["fail_reason"] for r in rows if r["failed"])),
             "is_claim_false": sum(1 for r in rows if not r["failed"] and r["is_claim"] is False),
             "unlabelled_claim": sum(r["unlabelled_claim"] for r in rows),
             "labels": dict(svc.most_common())}
    log("test", stats)
    return stats


def cmd_full():
    retag_ensure_tables()
    prompt = retag_load_prompt()
    test = run("test_done", f"SELECT count(*) FROM {VF}.retag_v2_test")[0][0]
    assert test, "run the test first"
    total = run("bk_n", f"SELECT count(*) FROM {VF}.sentence_tags_backup_v1")[0][0]
    k = 1
    while True:
        cid = f"v2_c{k:02d}"
        left = run("unassigned", f"""SELECT count(*) FROM {VF}.sentence_tags_backup_v1 b
            WHERE NOT EXISTS (SELECT 1 FROM {VF}.retag_v2_members m WHERE m.sentence_id = b.sentence_id)""")[0][0]
        have = run("mem_have", f"SELECT count(*) FROM {VF}.retag_v2_members WHERE chunk_id = ?", [cid])[0][0]
        if not have and not left:
            break
        n = add_members(cid, CHUNK, "b.sentence_id")
        secs, nparts = query_chunk(prompt, cid)
        raw = retag_read_raw(cid, 1)
        credit_check(cid, [e for _, e in raw.values()])
        log("chunks", {"chunk_id": cid, "members": n, "raw": len(raw), "parts_queried": nparts,
                       "seconds": round(secs, 1),
                       "per_s": round(len(raw) / secs, 1) if secs > 1 else None,
                       "invalid_first": sum(1 for r, e in raw.values() if validate(r, e)[1])})
        k += 1
    # one retry pass for every invalid attempt-1 response (test chunk included), one statement
    chunks = [c for (c,) in run("chunks", f"SELECT DISTINCT chunk_id FROM {VF}.retag_v2_members ORDER BY 1")]
    first = {}
    for c in chunks:
        for sid, (r, e) in retag_read_raw(c, 1).items():
            first[sid] = (c, validate(r, e))
    bad = [sid for sid, (_, v) in first.items() if v[1]]
    have2 = run("raw2", f"SELECT count(*), count(resp.errorMessage) FROM {VF}.retag_v2_raw WHERE attempt = 2")[0]
    if bad and (not have2[0] or have2[1]):
        run("raw2_reset", f"DELETE FROM {VF}.retag_v2_raw WHERE attempt = 2", fetch=False)
        run("retry_members", f"DELETE FROM {VF}.retag_v2_members WHERE chunk_id = 'v2_retry'", fetch=False)
        vals = ",".join(f"('{s}')" for s in bad)
        run("retry_members_ins", f"""INSERT INTO {VF}.retag_v2_members
            SELECT 'v2_retry', s, 0, current_timestamp() FROM VALUES {vals} AS v(s)""", fetch=False)
        t0 = time.time()
        query_part(prompt, "v2_retry", 2, None)
        log("chunks", {"chunk_id": "v2_retry", "rows": len(bad), "seconds": round(time.time() - t0, 1)})
    raw2 = retag_read_raw("v2_retry", 2) if bad else {}
    credit_check("v2_retry", [e for _, e in raw2.values()])
    rows = []
    for sid, (c, (o, f, _)) in first.items():
        if f and sid in raw2:
            o, f, _ = validate(*raw2[sid])
            c = "v2_retry" if not f else c
        rows.append(to_row(sid, c, o, f))
    run("results_reset", f"TRUNCATE TABLE {VF}.retag_v2_results", fetch=False)
    write_rows("retag_v2_results", rows)
    log("full", {"backup_rows": total, "results": len(rows), "invalid_first": len(bad),
                 "failed_after_retry": sum(r["failed"] for r in rows),
                 "fail_reasons": dict(Counter(r["fail_reason"] for r in rows if r["failed"]))})
    cmd_merge()


def cmd_merge():
    run("merge", f"""
        MERGE INTO {VF}.sentence_tags t
        USING (SELECT * FROM {VF}.retag_v2_results WHERE NOT failed) s
        ON t.sentence_id = s.sentence_id
        WHEN MATCHED AND t.prompt_version = '{V1}' THEN UPDATE SET
          is_claim = s.is_claim, services = s.services, strength = s.strength,
          suggested_name = s.suggested_name, prompt_version = '{V2}', chunk_id = s.chunk_id,
          failed = false, fail_reason = NULL, tagged_at = current_timestamp(),
          unlabelled_claim = s.unlabelled_claim""", fetch=False)
    r = run("merge_check", f"""SELECT prompt_version, count(*) FROM {VF}.sentence_tags
        WHERE sentence_id IN (SELECT sentence_id FROM {VF}.sentence_tags_backup_v1) GROUP BY 1""")
    log("merge", {"by_prompt_version": {p: n for p, n in r}})
    run("comment_pv", f"ALTER TABLE {VF}.sentence_tags ALTER COLUMN prompt_version COMMENT ?",
        ["pa_single_v1 (LLM, 49 labels), pa_single_v2 (LLM, 58 labels: re-tag of v1 rows that were "
         "other_clinical or unlabelled_claim, scripts/retag_new_labels.py; v1 rows in "
         "sentence_tags_backup_v1), or rule_dental_only (rule)."], fetch=False)
    run("comment_svc", f"ALTER TABLE {VF}.sentence_tags ALTER COLUMN services COMMENT ?",
        ["Model labels: 49-label list for pa_single_v1 rows, 58-label list for pa_single_v2 rows "
         "(exact prompts in workspace.vf.pa_prompt); [] when is_claim false. Rule rows: [dental]. "
         "NULL when failed."], fetch=False)


def retag_main(cmd):
    fn = {"prompt": cmd_prompt, "backup": cmd_backup, "test": cmd_test, "full": cmd_full,
          "merge": cmd_merge}[cmd]
    try:
        fn()
    except CreditsExhausted as e:
        msg = f"CREDITS EXHAUSTED, TOP UP NEEDED {now()}\nerror: {e}\nResume: retag_main({cmd!r})\n"
        print(msg, flush=True)
        LOGDIR.mkdir(parents=True, exist_ok=True)
        (LOGDIR / "CREDITS_EXHAUSTED.txt").write_text(msg)

In [0]:
if REBUILD and RUN_LLM:
    for cmd in ("prompt", "backup", "test", "full"):   # full ends with the MERGE into sentence_tags
        retag_main(cmd)

# Read-only check: the prompts stored in pa_prompt are exactly the ones this code builds.
stored = dict(sql_exec(f"SELECT prompt_version, prompt FROM {VF}.pa_prompt "
                       "WHERE prompt_version IN ('pa_single_v1', 'pa_single_v2')"))
print("pa_single_v1 stored == PROMPT_SINGLE:", stored.get("pa_single_v1") == PROMPT_SINGLE)
print("pa_single_v2 stored == build_prompt_v2(pa_single_v1):",
      stored.get("pa_single_v2") == build_prompt_v2(stored.get("pa_single_v1", "")))
print(f"labels: {len(SERVICES_V1)} in v1, {len(ALLOWED_V2)} in v2 (incl. other_clinical)")

pa_single_v1 stored == PROMPT_SINGLE: True
pa_single_v2 stored == build_prompt_v2(pa_single_v1): True
labels: 49 in v1, 60 in v2 (incl. other_clinical)


In [0]:
%sql
SELECT prompt_version, count(*) AS sentences, count_if(failed) AS failed,
       count_if(is_claim) AS claims, count_if(is_claim = false) AS not_claims
FROM workspace.vf.sentence_tags GROUP BY 1 ORDER BY 1

prompt_version,sentences,failed,claims,not_claims
pa_single_v1,207883,1531,126192,80160
pa_single_v2,31232,0,30199,1033
rule_dental_only,32147,0,0,0


In [0]:
%sql
-- Distinct sentences per service label (non-failed claims)
SELECT service, count(*) AS sentences,
       count_if(strength = 'explicit') AS explicit, count_if(strength = 'stated') AS stated, count_if(strength = 'weak') AS weak
FROM workspace.vf.sentence_tags LATERAL VIEW explode(services) e AS service
WHERE NOT coalesce(failed, false) AND (is_claim OR prompt_version = 'rule_dental_only')
GROUP BY 1 ORDER BY sentences DESC LIMIT 20

service,sentences,explicit,stated,weak
dental,35906,525,2770,464
laboratory_pathology,13602,3444,9354,804
general_surgery,11242,3683,5987,1572
eye_surgery,11124,3312,7617,195
orthopedic_surgery,9150,2204,5300,1646
other_clinical,9150,2063,4498,2420
obstetrics_maternity,8852,1022,6414,1416
outpatient_opd,7727,376,5572,1779
infertility_ivf,7472,1192,6156,124
ultrasound,6345,2658,3108,579


In [0]:
%sql
-- What the tags look like, for sentences containing 'dialysis'
SELECT d.example_sentence, t.is_claim, t.services, t.strength
FROM workspace.vf.sentence_tags t JOIN workspace.vf.distinct_sentences d USING (sentence_id)
WHERE d.sentence_key LIKE '%dialysis%' AND NOT coalesce(t.failed, false)
ORDER BY d.n_facilities DESC LIMIT 10

example_sentence,is_claim,services,strength
Dialysis,true,List(dialysis),stated
Provides dialysis treatment,true,List(dialysis),stated
Dialysis unit,true,List(dialysis),explicit
Dialysis services,true,List(dialysis),stated
Provides dialysis services,true,List(dialysis),stated
Dialysis treatment,true,List(dialysis),stated
Advanced Kidney Dialysis,true,List(dialysis),stated
Hemodialysis,true,List(dialysis),stated
Peritoneal dialysis,true,List(dialysis),stated
Dialysis machines,true,List(dialysis),explicit


## Step 5 — Does the record's text belong to this facility?

**What.** One `ai_query` call per facility reads all its usable sentences, numbered, against its own name and city, and lists the sentences that describe a different organisation. Renamed cities, localities inside the city, and the hospital's own branches count as the same place; a similarly named hospital in another city counts as foreign. The answer is parsed (ranges like `12-19` expanded, numbers range-checked); failures get one retry at temperature 0.3. Credit or quota errors stop the job without writing those rows as failed.

**Why.** Some records absorbed text from other hospitals. A plain text search recommends the Aravind Eye Hospital record in Hyderabad for dialysis, because it carries a dialysis unit copied from Delhi government hospitals. Flagged sentences are removed from the evidence in step 6.

**Reads** `facilities_clean`, `facility_sentences`. **Writes** `record_audit` (9,577 facilities with >= 5 usable sentences), `sentence_foreign_flag` (422,686 sentence rows). Source: `scripts/record_audit.py`, prompt and parser from `scripts/pilot_b_audit.py` (prompt `pb_v1`).

Measured (`reports/record_audit.md`): 2 failed audits of 9,577; 814 records end up with a mixed-record warning; for the Aravind record, 79 of 149 sentences flagged.

In [0]:
# scripts/pilot_b_audit.py: prompt pb_v1 and the validating parser (with fix (a) for bare ranges)
from datetime import datetime, timezone

PROMPT_VERSION = "pb_v1"
MAX_TOKENS = 800  # earlier audit used 400; raised because foreign_sentences adds output

PROMPT_HEAD = """You audit ONE record from a scraped database of Indian healthcare facilities.
Records were built by merging web snippets, and some records absorbed sentences that actually describe a DIFFERENT organisation (another hospital, clinic or government facility, often in another city).

RECORD
Name: """
PROMPT_TAIL = """

TASK
Judge every numbered sentence against the record's OWN name and city.
- A sentence is foreign if it names, or clearly describes, a different organisation or a different location.
- A branch, unit, parent or sister unit of the same group counts as the same organisation (consistent).
- Old and new names of one city are the same place (e.g. Bijapur and Vijayapura, Berhampur and Brahmapur), and a locality, taluk or district that contains the record's city is the same place.
- A different hospital with a similar or identical name in another city is foreign.
- A generic sentence that names no organisation and could fit this facility counts as consistent.
verdict: "consistent" if (almost) no sentences are foreign; "partly_foreign" if some but not most are foreign; "mostly_foreign" if more than half are foreign.

Return ONLY a JSON object, no other text:
{"verdict": "consistent" | "partly_foreign" | "mostly_foreign",
 "foreign_share": <number from 0 to 1: rough share of sentences that describe a different organisation>,
 "foreign_sentences": [<numbers of the sentences that describe a different organisation; integers, or ranges as strings like "12-19"; [] if none>],
 "other_names": [<up to 5 OTHER facility names found in the sentences; [] if none>],
 "reason": "<at most 15 words>"}"""
SENT_HEADER = "\\n\\nNUMBERED SENTENCES (from the record\\'s procedure, capability and equipment fields)\\n"


def lit(s: str) -> str:
    return "'" + s.replace("\\", "\\\\").replace("'", "\\'") + "'"


def full_prompt_text():
    return (PROMPT_HEAD + "{name}\nCity: {city}\nState: {state}\nDescription: {description[:1200]}"
            "\n\nNUMBERED SENTENCES (from the record's procedure, capability and equipment fields)\n"
            "1. {sentence 1}\n2. {sentence 2}\n..." + PROMPT_TAIL)


AUDIT_VERDICTS = {"consistent", "partly_foreign", "mostly_foreign"}
RANGE_RE = re.compile(r"^\s*(\d+)\s*-\s*(\d+)\s*$")


# fix (a), user decision after stage-0 run 1: quote bare a-b tokens before json parsing
BARE_RANGE_RE = re.compile(r'([\[,]\s*)(\d+)\s*-\s*(\d+)(\s*)(?=[,\]])')
DIGITS_RE = re.compile(r"^\s*(\d+)\s*$")


def normalise(raw):
    return BARE_RANGE_RE.sub(r'\1"\2-\3"\4', raw)


def parse(raw, n):
    if raw is None:
        return None, "null response"
    try:
        o = json.loads(normalise(raw[raw.index("{"): raw.rindex("}") + 1]))
    except Exception as e:  # noqa: BLE001
        return None, f"unparseable: {str(e)[:60]}"
    if not isinstance(o, dict):
        return None, "not an object"
    v = o.get("verdict")
    if v not in AUDIT_VERDICTS:
        return None, f"bad verdict {v!r}"
    fs = o.get("foreign_share")
    if isinstance(fs, str):
        try:
            fs = float(fs)
        except ValueError:
            return None, "foreign_share not number"
    if isinstance(fs, bool) or not isinstance(fs, (int, float)) or not 0 <= fs <= 1:
        return None, f"bad foreign_share {fs!r}"
    on = o.get("other_names")
    if not isinstance(on, list) or len(on) > 5 or not all(isinstance(x, str) for x in on):
        return None, "bad other_names"
    fl = o.get("foreign_sentences")
    if not isinstance(fl, list):
        return None, "foreign_sentences not a list"
    nums = set()
    for x in fl:
        if isinstance(x, int) and not isinstance(x, bool):
            if not 1 <= x <= n:
                return None, "foreign_sentences out of range"
            nums.add(x)
        elif isinstance(x, str) and DIGITS_RE.match(x):  # lenient: "7" -> 7
            k = int(x)
            if not 1 <= k <= n:
                return None, "foreign_sentences out of range"
            nums.add(k)
        elif isinstance(x, str) and RANGE_RE.match(x):
            a, b = map(int, RANGE_RE.match(x).groups())
            if not (1 <= a <= b <= n):
                return None, "foreign_sentences out of range"
            nums.update(range(a, b + 1))
        else:
            return None, "foreign_sentences bad item"
    r = o.get("reason")
    if not isinstance(r, str):
        return None, "bad reason"
    return {"verdict": v, "foreign_share": float(fs), "other_names": on, "reason": r,
            "foreign_sentences_raw": fl, "flagged": sorted(nums)}, None


# scripts/record_audit.py
FC, FS = f"{VF}.facilities_clean", f"{VF}.facility_sentences"
SEED = "ra2026"
N_STAGE1_CHUNKS = 7  # ~9,477 / 7 = ~1,354 records per chunk
RETRY_TEMP = 0.3     # same as Pilot B's single retry
RATE_LIMIT = 0.03
LOG = str(LOG_ROOT / "ra_timings.jsonl")

# Credits/quota standing rule (CLAUDE.md): stop at once, do not store those rows as failed, report.
import re  # noqa: E402
CREDIT_RE = re.compile(r"quota|credit|billing|limit exceeded|limit_exceeded|resource_exhausted|payment|insufficient|"
                       r"warehouse.{0,40}(start|stopped|not running)", re.I)
CREDIT_SQL = ("lower(coalesce(fail_reason,'')) RLIKE 'quota|credit|billing|limit exceeded|limit_exceeded|"
              "resource_exhausted|payment|insufficient'")
BULK_SHARE = 0.30


class CreditsExhausted(Exception):
    pass


def run(q, fetch=False, label=None):
    t0 = time.time()
    s0 = datetime.now(timezone.utc).isoformat(timespec="seconds")
    try:
        with connect() as c, c.cursor() as cur:
            cur.execute(q)
            out = None
            if fetch:
                cols = [d[0] for d in cur.description]
                out = [dict(zip(cols, r)) for r in cur.fetchall()]
    except CreditsExhausted:
        raise
    except Exception as e:  # noqa: BLE001
        if CREDIT_RE.search(str(e)):
            raise CreditsExhausted(f"statement error ({label}): {str(e)[:500]}") from e
        raise
    finally:
        rec = {"label": label or q.strip().split("\n")[0][:80], "start": s0,
               "end": datetime.now(timezone.utc).isoformat(timespec="seconds"),
               "seconds": round(time.time() - t0, 2)}
        with open(LOG, "a") as f:
            f.write(json.dumps(rec) + "\n")
        print(f"[{rec['start']} -> {rec['end']}] {rec['seconds']:8.1f}s  {rec['label']}", flush=True)
    return out


# ------------------------------------------------------------------------------------ prepare
def prepare():
    run(f"""
CREATE OR REPLACE TABLE {VF}.ra_sentences AS
WITH u AS (
  SELECT s.unique_id, s.source_column, s.position, s.sentence
  FROM {FS} s JOIN {FC} f USING (unique_id)
  WHERE NOT f.is_shifted AND s.is_placeholder = false AND s.is_repeat = false),
pop AS (SELECT unique_id FROM u GROUP BY unique_id HAVING count(*) >= 5)
SELECT u.unique_id, u.source_column, u.position, u.sentence,
  CAST(row_number() OVER (PARTITION BY u.unique_id ORDER BY
    CASE u.source_column WHEN 'procedure' THEN 1 WHEN 'capability' THEN 2 ELSE 3 END, u.position) AS INT)
    AS sentence_no
FROM u JOIN pop USING (unique_id)""", label="create ra_sentences")

    # stage 0 = Aravind + 99 seeded-random others; stage 1 = the rest in 7 seeded chunks
    run(f"""
CREATE OR REPLACE TABLE {VF}.ra_inputs AS
WITH agg AS (
  SELECT unique_id, count(*) n_usable,
    array_join(transform(array_sort(collect_list(struct(sentence_no, sentence))),
                         x -> concat(cast(x.sentence_no AS string), '. ', x.sentence)), '\\n') numbered
  FROM {VF}.ra_sentences GROUP BY unique_id),
ord AS (
  SELECT unique_id, row_number() OVER (ORDER BY CASE WHEN unique_id = '{ARAVIND}' THEN 0 ELSE 1 END,
                                        md5(concat('{SEED}', unique_id))) rn
  FROM agg),
ch AS (
  SELECT unique_id, rn,
    CASE WHEN rn <= 100 THEN 's0'
         ELSE concat('s1_c', lpad(cast(ntile({N_STAGE1_CHUNKS}) OVER (ORDER BY rn) AS string), 2, '0')) END tmp
  FROM ord)
SELECT a.unique_id, CAST(a.n_usable AS INT) n_usable,
  CASE WHEN ch.rn <= 100 THEN 's0' ELSE ch.tmp END AS chunk_id,
  concat({lit(PROMPT_HEAD)}, coalesce(f.name,'(none)'),
    '\\nCity: ', coalesce(f.city,'(none)'),
    '\\nState: ', coalesce(f.state,'(none)'),
    '\\nDescription: ', coalesce(substr(f.description,1,1200),'(none)'),
    '{SENT_HEADER}', a.numbered,
    {lit(PROMPT_TAIL)}) AS prompt
FROM agg a JOIN ch USING (unique_id) JOIN {FC} f USING (unique_id)""", label="create ra_inputs")
    # ntile above runs over all rows incl. stage 0; recompute stage-1 chunks over stage-1 rows only
    run(f"""
CREATE OR REPLACE TABLE {VF}.ra_inputs AS
WITH o AS (SELECT *, row_number() OVER (ORDER BY CASE WHEN unique_id = '{ARAVIND}' THEN 0 ELSE 1 END,
                                        md5(concat('{SEED}', unique_id))) rn FROM {VF}.ra_inputs),
s1 AS (SELECT unique_id, ntile({N_STAGE1_CHUNKS}) OVER (ORDER BY rn) k FROM o WHERE rn > 100)
SELECT o.unique_id, o.n_usable,
  CASE WHEN o.rn <= 100 THEN 's0' ELSE concat('s1_c', lpad(cast(s1.k AS string), 2, '0')) END chunk_id,
  o.prompt
FROM o LEFT JOIN s1 USING (unique_id)""", label="rechunk ra_inputs")
    print(run(f"""SELECT chunk_id, count(*) n, min(n_usable) mn, max(n_usable) mx, max(length(prompt)) maxlen,
      sum(CASE WHEN unique_id = '{ARAVIND}' THEN 1 ELSE 0 END) aravind
      FROM {VF}.ra_inputs GROUP BY 1 ORDER BY 1""", True, "summary ra_inputs"))
    # prompt identity check vs Pilot B's stored prompt for the same records
    print(run(f"""SELECT count(*) n_common, sum(CASE WHEN i.prompt = p.prompt THEN 1 ELSE 0 END) n_identical
      FROM {VF}.ra_inputs i JOIN {VF}.pb_records p USING (unique_id)""", True, "prompt identity vs pb_records"))

    run(f"""CREATE TABLE IF NOT EXISTS {VF}.ra_raw (unique_id STRING, chunk_id STRING, attempt INT,
      temperature DOUBLE, raw STRING, err STRING, called_at TIMESTAMP)""", label="create ra_raw")
    run(f"""CREATE TABLE IF NOT EXISTS {VF}.ra_parsed (unique_id STRING, attempt INT,
      foreign_sentences STRING, flagged ARRAY<INT>)""", label="create ra_parsed")
    run(f"""CREATE TABLE IF NOT EXISTS {VF}.record_audit (unique_id STRING, verdict STRING,
      foreign_share DOUBLE, other_names ARRAY<STRING>, reason STRING, n_usable INT, n_flagged INT,
      flagged_share DOUBLE, failed BOOLEAN, fail_reason STRING, attempts INT, chunk_id STRING,
      prompt_version STRING, audited_at TIMESTAMP)""", label="create record_audit")
    run(f"""CREATE TABLE IF NOT EXISTS {VF}.sentence_foreign_flag (unique_id STRING, sentence_no INT,
      source_column STRING, position INT, is_foreign BOOLEAN)""", label="create sentence_foreign_flag")


# ------------------------------------------------------------------------------------ core
def call(where, chunk_label, attempt, temp, n):
    t0 = time.time()
    run(f"""
INSERT INTO {VF}.ra_raw
SELECT unique_id, '{chunk_label}', {attempt}, {temp}, r.result, r.errorMessage, current_timestamp() FROM (
  SELECT unique_id, ai_query('{MODEL}', prompt,
      modelParameters => named_struct('max_tokens', {MAX_TOKENS}, 'temperature', {temp}),
      failOnError => false) AS r
  FROM {VF}.ra_inputs i WHERE {where})""", label=f"{chunk_label} ai_query attempt{attempt} ({n} rows)")
    return round(time.time() - t0, 2)


def screen_credit(rows, chunk, attempt):
    """Return error text if the rows show credit/quota exhaustion; also drop those raw rows so a resume recalls them."""
    errs = [r["err"] for r in rows if r["err"]]
    bad = {e for e in errs if CREDIT_RE.search(e)}
    if errs:
        top, cnt = max(((e, errs.count(e)) for e in set(errs)), key=lambda x: x[1])
        if rows and cnt / len(rows) > BULK_SHARE:
            bad.add(top)
    if not bad:
        return None
    ids = ",".join(lit(r["unique_id"]) for r in rows if r["err"] in bad)
    run(f"DELETE FROM {VF}.ra_raw WHERE attempt = {attempt} AND unique_id IN ({ids})", label="drop credit-error raw rows")
    rows[:] = [r for r in rows if r["err"] not in bad]
    return sorted(bad)[0]


def store(rows, attempt):
    """rows: dicts with unique_id, chunk_id, n_usable, raw, err. Parse, then write audit + flags."""
    audit, parsed, reasons = [], [], {}
    for r in rows:
        uid, n = r["unique_id"], r["n_usable"]
        res, e = parse(r["raw"], n) if not r["err"] else (None, f"endpoint error: {r['err'][:100]}")
        if res:
            k = len(res["flagged"])
            names = "array(" + ",".join(lit(x) for x in res["other_names"]) + ")" if res["other_names"] \
                else "CAST(array() AS ARRAY<STRING>)"
            audit.append(f"({lit(uid)},{lit(res['verdict'])},{res['foreign_share']},{names},{lit(res['reason'])},"
                         f"{n},{k},{k / n},false,NULL,{attempt},{lit(r['chunk_id'])})")
            parsed.append(f"({lit(uid)},{attempt},{lit(json.dumps(res['foreign_sentences_raw']))},"
                          f"array({','.join(map(str, res['flagged']))}))" if res["flagged"] else
                          f"({lit(uid)},{attempt},{lit(json.dumps(res['foreign_sentences_raw']))},"
                          f"CAST(array() AS ARRAY<INT>))")
        else:
            reasons[e.split(":")[0]] = reasons.get(e.split(":")[0], 0) + 1
            audit.append(f"({lit(uid)},NULL,NULL,NULL,NULL,{n},NULL,NULL,true,{lit(e)},{attempt},"
                         f"{lit(r['chunk_id'])})")
    if not audit:
        return 0, reasons
    ids = ",".join(lit(r["unique_id"]) for r in rows)
    if parsed:
        run(f"DELETE FROM {VF}.ra_parsed WHERE attempt = {attempt} AND unique_id IN ({ids})", label="clear ra_parsed")
        run(f"INSERT INTO {VF}.ra_parsed VALUES " + ",".join(parsed), label=f"insert ra_parsed ({len(parsed)})")
    run(f"DELETE FROM {VF}.sentence_foreign_flag WHERE unique_id IN ({ids})", label="clear flags")
    run(f"DELETE FROM {VF}.record_audit WHERE unique_id IN ({ids})", label="clear record_audit")
    run(f"INSERT INTO {VF}.record_audit SELECT *, '{PROMPT_VERSION}', current_timestamp() FROM VALUES "
        + ",".join(audit), label=f"insert record_audit ({len(audit)})")
    run(f"""
INSERT INTO {VF}.sentence_foreign_flag
SELECT s.unique_id, s.sentence_no, s.source_column, s.position,
  CASE WHEN a.failed THEN NULL ELSE array_contains(p.flagged, s.sentence_no) END
FROM {VF}.ra_sentences s JOIN {VF}.record_audit a USING (unique_id)
LEFT JOIN {VF}.ra_parsed p ON p.unique_id = s.unique_id AND p.attempt = a.attempts
WHERE s.unique_id IN ({ids})""", label="insert sentence_foreign_flag")
    return len(parsed), reasons


def run_chunk(chunk):
    pend = f"i.chunk_id = '{chunk}' AND i.unique_id NOT IN (SELECT unique_id FROM {VF}.record_audit)"
    n = run(f"""SELECT count(*) n, sum(CASE WHEN EXISTS (SELECT 1 FROM {VF}.ra_raw r WHERE r.unique_id = i.unique_id
      AND r.attempt = 1) THEN 1 ELSE 0 END) have FROM {VF}.ra_inputs i WHERE {pend}""", True, f"{chunk} pending")[0]
    wall = None
    todo = n["n"] - (n["have"] or 0)
    if todo:  # resume: only call the model for rows with no stored attempt-1 response
        wall = call(pend + f" AND i.unique_id NOT IN (SELECT unique_id FROM {VF}.ra_raw WHERE attempt = 1)",
                    chunk, 1, 0.0, todo)
    rows = run(f"""SELECT i.unique_id, i.chunk_id, i.n_usable, r.raw, r.err FROM {VF}.ra_inputs i
      JOIN (SELECT * FROM {VF}.ra_raw WHERE attempt = 1 QUALIFY row_number() OVER
            (PARTITION BY unique_id ORDER BY called_at DESC) = 1) r USING (unique_id) WHERE {pend}""",
               True, f"{chunk} read raw")
    credit_err = screen_credit(rows, chunk, 1)
    valid, reasons = store(rows, 1)  # non-credit rows of this chunk are kept
    if credit_err:
        raise CreditsExhausted(f"ai_query errorMessage in chunk {chunk}: {credit_err}")
    rec = {"chunk": chunk, "rows": len(rows), "called": todo, "ai_query_seconds": wall, "valid": valid,
           "failed": len(rows) - valid, "reasons": reasons,
           "s_per_record": round(wall / todo, 3) if wall and todo else None}
    with open(LOG_ROOT / "ra_chunks.jsonl", "a") as f:
        f.write(json.dumps(rec) + "\n")
    print("CHUNK", rec, flush=True)
    return rec


def rates():
    a = run(f"""SELECT count(*) n, sum(CASE WHEN failed THEN 1 ELSE 0 END) f FROM {VF}.record_audit
      WHERE NOT (failed AND {CREDIT_SQL})""", True, "audit failed rate")[0]
    t = run(f"""SELECT count(*) n, sum(CASE WHEN failed THEN 1 ELSE 0 END) f FROM {VF}.sentence_tags
      WHERE prompt_version = 'pa_single_v1' AND NOT (failed AND {CREDIT_SQL})
        AND NOT (failed AND coalesce(fail_reason, '') IN ('claim_without_label', 'unknown_label'))""",
            True, "tagging failed rate (content failures excluded)")[0]
    ar = (a["f"] or 0) / a["n"] if a["n"] else 0.0
    tr = (t["f"] or 0) / t["n"] if t["n"] else 0.0
    return ar, a, tr, t


def stage0():
    t0 = time.time()
    rec = run_chunk("s0")
    secs = round(time.time() - t0, 1)
    vc = run(f"SELECT verdict, count(*) n FROM {VF}.record_audit WHERE chunk_id = 's0' GROUP BY 1", True, "s0 verdicts")
    av = run(f"SELECT verdict, n_flagged, n_usable, failed, fail_reason FROM {VF}.record_audit "
             f"WHERE unique_id = '{ARAVIND}'", True, "aravind")
    ar, a, tr, t = rates()
    print(json.dumps({"stage0": rec, "seconds_total": secs, "verdicts": vc, "aravind": av,
                      "tagging_failed": t, "tagging_rate": tr}, default=str, indent=1))


def stage1():
    chunks = [r["chunk_id"] for r in run(f"""SELECT DISTINCT chunk_id FROM {VF}.ra_inputs
      WHERE chunk_id LIKE 's1_%' ORDER BY 1""", True, "list chunks")]
    t0 = time.time()
    for ch in chunks:
        ar, a, tr, t = rates()
        print(f"before {ch}: audit failed {a['f']}/{a['n']} = {ar:.4f}; tagging failed {t['f']}/{t['n']} = {tr:.4f}",
              flush=True)
        if ar > RATE_LIMIT or tr > RATE_LIMIT:
            print(f"PAUSE before {ch}: audit rate {ar:.4f}, tagging rate {tr:.4f}", flush=True)
            return
        run_chunk(ch)
    print(f"STAGE1 DONE in {time.time() - t0:.0f}s", flush=True)


def retry():
    rows = run(f"SELECT count(*) n FROM {VF}.record_audit WHERE failed AND attempts = 1", True, "retry count")[0]["n"]
    if rows:
        wall = call(f"i.unique_id IN (SELECT unique_id FROM {VF}.record_audit WHERE failed AND attempts = 1)",
                    "retry", 2, RETRY_TEMP, rows)
        got = run(f"""SELECT i.unique_id, a.chunk_id, i.n_usable, r.raw, r.err FROM {VF}.ra_inputs i
          JOIN {VF}.record_audit a USING (unique_id)
          JOIN (SELECT * FROM {VF}.ra_raw WHERE attempt = 2 QUALIFY row_number() OVER
                (PARTITION BY unique_id ORDER BY called_at DESC) = 1) r USING (unique_id)
          WHERE a.failed AND a.attempts = 1""", True, "retry read raw")
        credit_err = screen_credit(got, "retry", 2)
        valid, reasons = store(got, 2)
        if credit_err:
            raise CreditsExhausted(f"ai_query errorMessage in retry pass: {credit_err}")
        rec = {"chunk": "retry", "rows": len(got), "ai_query_seconds": wall, "valid": valid,
               "failed": len(got) - valid, "reasons": reasons}
        with open(LOG_ROOT / "ra_chunks.jsonl", "a") as f:
            f.write(json.dumps(rec) + "\n")
        print("RETRY", rec, flush=True)


def comments():
    C = {
        "record_audit": ("One row per audited facility (non-shifted, >= 5 usable sentences). LLM judgement of whether the "
                         "record's usable sentences describe a different organisation (record merging). Prompt pb_v1 "
                         "(text in workspace.vf.pb_prompt), model databricks-meta-llama-3-3-70b-instruct, temperature 0, "
                         "one retry at 0.3. Script scripts/record_audit.py.", {
            "unique_id": "Facility key; joins facilities_clean.unique_id.",
            "verdict": "consistent | partly_foreign | mostly_foreign; NULL if failed.",
            "foreign_share": "Model's own rough share (0-1) of sentences describing another organisation.",
            "other_names": "Up to 5 other facility names the model found in the sentences.",
            "reason": "Model's short reason (<= 15 words).",
            "n_usable": "Usable sentences sent (is_placeholder false AND is_repeat false), numbered 1..n_usable.",
            "n_flagged": "Number of sentences listed as foreign (ranges expanded).",
            "flagged_share": "n_flagged / n_usable; computed, not model-stated.",
            "failed": "True if the response failed validation on all attempts.",
            "fail_reason": "Validation or endpoint error of the last attempt.",
            "attempts": "1 = first call, 2 = after the single retry.",
            "chunk_id": "s0 (stage 0, 100 records) or s1_cNN (stage 1 chunk).",
            "prompt_version": "Prompt version (pb_v1).",
            "audited_at": "Time the row was written."}),
        "sentence_foreign_flag": ("One row per usable sentence of each audited facility; joins facility_sentences on "
                                  "unique_id + source_column + position. is_foreign from record_audit's sentence list.", {
            "unique_id": "Facility key.",
            "sentence_no": "Number shown to the model (order: procedure, capability, equipment, then position).",
            "source_column": "procedure | capability | equipment.",
            "position": "Position within the source column (facility_sentences.position).",
            "is_foreign": "True if the model listed the sentence as describing another organisation; NULL if the audit failed."}),
    }
    for t, (tc, cols) in C.items():
        run(f"COMMENT ON TABLE {VF}.{t} IS {lit(tc)}", label=f"comment {t}")
        for col, cc in cols.items():
            run(f"ALTER TABLE {VF}.{t} ALTER COLUMN {col} COMMENT {lit(cc)}", label=f"comment {t}.{col}")


def summary():
    q = lambda s, l: run(s, True, l)  # noqa: E731
    print(q(f"""SELECT count(*) n, sum(CASE WHEN failed THEN 1 ELSE 0 END) failed,
      sum(CASE WHEN attempts = 2 THEN 1 ELSE 0 END) retried FROM {VF}.record_audit""", "totals"))
    print(q(f"""SELECT verdict, count(*) n, round(count(*) / sum(count(*)) OVER (), 4) share,
      round(avg(flagged_share), 4) mean_flagged_share, round(avg(foreign_share), 4) mean_foreign_share
      FROM {VF}.record_audit GROUP BY 1 ORDER BY 1""", "verdicts"))
    print(q(f"SELECT fail_reason, count(*) n FROM {VF}.record_audit WHERE failed GROUP BY 1", "fail reasons"))
    print(q(f"""SELECT sum(CASE WHEN n_flagged > 0 THEN 1 ELSE 0 END) fac_with_flag, sum(n_flagged) flagged_total
      FROM {VF}.record_audit""", "flag totals"))
    print(q(f"""SELECT count(*) rows, sum(CASE WHEN is_foreign THEN 1 ELSE 0 END) foreign_rows,
      sum(CASE WHEN is_foreign IS NULL THEN 1 ELSE 0 END) null_rows, count(DISTINCT unique_id) fac
      FROM {VF}.sentence_foreign_flag""", "flag table"))
    print(q(f"SELECT * FROM {VF}.record_audit WHERE unique_id = '{ARAVIND}'", "aravind"))


def progress():
    d = run(f"SELECT count(*) n FROM {VF}.record_audit", True, "progress")[0]["n"]
    t = run(f"SELECT count(*) n FROM {VF}.ra_inputs", True, "population")[0]["n"]
    return d, t - d


def record_audit_main(cmd):
    try:
        {"prepare": prepare, "stage0": stage0, "stage1": stage1, "retry": retry, "comments": comments,
         "summary": summary}[cmd]()
    except CreditsExhausted as e:
        done, left = progress()
        print(f"CREDITS EXHAUSTED, TOP UP NEEDED\n{e}\njob: record_audit {cmd}; rows done {done}, "
              f"rows left {left}", flush=True)

In [0]:
print(full_prompt_text())
if REBUILD and RUN_LLM:
    for cmd in ("prepare", "stage0", "stage1", "retry", "comments"):
        record_audit_main(cmd)
record_audit_main("summary")   # read-only: totals, verdicts, flag counts, the Aravind row

You audit ONE record from a scraped database of Indian healthcare facilities.
Records were built by merging web snippets, and some records absorbed sentences that actually describe a DIFFERENT organisation (another hospital, clinic or government facility, often in another city).

RECORD
Name: {name}
City: {city}
State: {state}
Description: {description[:1200]}

NUMBERED SENTENCES (from the record's procedure, capability and equipment fields)
1. {sentence 1}
2. {sentence 2}
...

TASK
Judge every numbered sentence against the record's OWN name and city.
- A sentence is foreign if it names, or clearly describes, a different organisation or a different location.
- A branch, unit, parent or sister unit of the same group counts as the same organisation (consistent).
- Old and new names of one city are the same place (e.g. Bijapur and Vijayapura, Berhampur and Brahmapur), and a locality, taluk or district that contains the record's city is the same place.
- A different hospital with a similar

In [0]:
%sql
-- Aravind Eye Hospital, Hyderabad: sentences the audit flagged as describing another organisation
SELECT s.source_column, s.sentence, f.is_foreign
FROM workspace.vf.sentence_foreign_flag f
JOIN workspace.vf.facility_sentences s USING (unique_id, source_column, position)
WHERE f.unique_id = 'b8a5401f-42f1-422a-8cd9-686a15b4cb76' AND (s.sentence ILIKE '%dialysis%' OR s.sentence ILIKE '%eye%' OR s.sentence ILIKE '%cataract%')
ORDER BY f.is_foreign DESC, s.source_column, s.position LIMIT 10

source_column,sentence,is_foreign
equipment,Dialysis machines: 10,true
equipment,Dialysis Operating Theatre in the OT Complex,true
equipment,Dialysis Centre,true
capability,Eye care hospital,false
capability,Aravind Eye Hospital Pvt. Ltd. is listed in the Hyderabad hospital directory as Aravind Eye Hospital,false
capability,Dialysis unit of 20 beds under Pradhan Mantri National Dialysis Programme,false
procedure,Dialysis treatment is provided via a 20-bed Dialysis Unit with 10 machines as part of the Pradhan Mantri National Dialysis Programme,false
procedure,Hemodialysis is performed in three units as a regular service,false
procedure,Peritoneal dialysis is available around the clock,false
procedure,A dedicated Dialysis Operating Theatre is commissioned in the OT Complex,false


## Step 6 — Lookups and the verdict

**What.** Three hand-written lookups, then one SQL statement that turns evidence into a tier for every (facility, service) pair. No LLM.

- `service_department_map`: which specialties back each service (`nephrology` -> dialysis, *strong*) and which other services imply it (`normal_delivery` -> C-section is possible).
- `service_type_fit`: 60 services x 26 facility types -> `expected` / `plausible` / `unexpected` (dialysis is unexpected at an eye hospital), each with the rule that decided it.
- `service_terms`: words that must name the service for Confirmed, plus lists of dedicated-unit and named-procedure terms.

**Tier rules** (first match wins; sentences flagged foreign in step 5 are removed first):

| tier | rule |
|---|---|
| **Unverified** | record mostly foreign (or partly foreign with >= 50% of sentences flagged), **or** the service is unexpected for the facility type, **or** only weak sentences |
| **Possible** (department only) | every claim sentence is only a department name or an operating-theatre line, with no service term |
| **Confirmed** | >= 1 detailed sentence (a number next to a unit word such as beds or machines, a dedicated unit, or for surgical services a named procedure or a procedure count), that names the service, **and** (a backing department, **or** >= 2 explicit/stated sentences, **or** the service is expected for the type) |
| **Likely** | >= 1 explicit or stated sentence |
| **Possible** (inferred) | no direct claim, but a backing department and a claim for a closely related service, hospital types only |

Hedged (*"planned"*, *"coming soon"*), negated, extractor-commentary and *"at partner hospitals"* sentences count as weak and are never quoted for Confirmed or Likely. Each row keeps up to 3 quoted sentences and plain-text warnings.

**Reads** steps 1-5. **Writes** the lookups, `facility_service` (the contract table the app reads) and `build_stats`. Source: `scripts/build_lookups.py`, `scripts/build_facility_service.py`.

Measured: 12,128 Confirmed, 55,352 Likely, 8,005 Possible, 22,760 Unverified facility-service pairs across 60 services and 9,989 facilities.

In [0]:
# scripts/build_lookups.py (hand-written lookups: department backing, type fit, term lists)
MIN_FACILITIES = 20

SERVICES = [
    "dialysis", "caesarean_section", "normal_delivery", "high_risk_obstetrics",
    "nicu", "picu", "icu_critical_care", "ventilator_support",
    "cardiac_surgery", "interventional_cardiology", "stroke_care",
    "mri", "ct_scan", "xray", "ultrasound", "mammography", "nuclear_imaging",
    "burn_care", "trauma_emergency", "ambulance",
    "chemotherapy", "radiotherapy", "cancer_surgery",
    "orthopedic_surgery", "joint_replacement", "neurosurgery", "spine_surgery",
    "general_surgery", "gi_endoscopy", "pulmonology_respiratory", "transplant",
    "blood_bank", "laboratory_pathology", "dental", "eye_surgery",
    "physiotherapy_rehab", "mental_health", "outpatient_opd",
    "infertility_ivf", "dermatology", "ent", "ophthalmology_medical",
    "pediatrics", "obstetrics_maternity", "cosmetic_aesthetic",
    "prosthetics_orthotics", "nephrology_renal", "gastroenterology",
    "other_clinical",
]  # same list as scripts/classify_eval.py (49 labels, prompt pa_single_v1)
# 11 labels added with prompt pa_single_v2 (reports/new_service_labels_proposal.md;
# re-tag in scripts/retag_new_labels.py). General / internal medicine stays outpatient_opd.
NEW_SERVICES_V2 = [
    "urology", "diabetes_endocrinology", "health_checkup_vaccination", "cardiac_diagnostics",
    "neurology", "ayush_traditional_medicine", "gynecology", "proctology", "pain_management",
    "vascular_surgery", "cardiology_medical",
]
SERVICES = SERVICES + NEW_SERVICES_V2

FACILITY_TYPES = [
    "multispecialty_hospital", "general_hospital_or_nursing_home",
    "medical_college_teaching_hospital", "government_health_centre",
    "maternity_women_children", "pediatric", "eye", "dental", "ent_hearing_speech",
    "skin_hair_cosmetic", "ivf_fertility", "kidney_urology", "cardiac", "cancer",
    "orthopedic_spine", "neuro", "gastro_liver", "diagnostic_imaging_lab",
    "physiotherapy_rehab", "mental_health_deaddiction", "ayush_alternative",
    "home_care_or_ambulance", "general_clinic_polyclinic", "other_single_specialty",
    "not_a_care_facility", "unknown",
]

# ---------------------------------------------------------------------------
# service_department_map source
# S = strong (the specialty is this service's own department / subspecialty)
# P = supporting (related department; makes the service more likely, proves nothing)
# Department-only values (internalMedicine, familyMedicine, generalMedicine,
# preventiveMedicine, anesthesia) are over-assigned in this data and are never strong.
# ---------------------------------------------------------------------------
DEF_NOTE = {"S": "specialty is this service's own department or subspecialty",
            "P": "related department; raises likelihood, does not prove the service"}
STRENGTH = {"S": "strong", "P": "supporting"}

OBS = ["gynecologyAndObstetrics", "obstetricsAndGynaecology", "obstetricsAndGynecology",
       "obstetricsAndMaternityCare"]
RADIOLOGY = ["radiology", "diagnosticRadiology", "neuroradiology", "interventionalRadiology",
             "cardiothoracicRadiology", "pediatricRadiology"]
NO_MACHINE = "radiology department does not prove this machine exists"

SPEC = {
    "dialysis": [("nephrology", "S", "nephrology department usually runs haemodialysis; still a department, not proof of a unit"),
                 ("pediatricNephrology", "S", None), ("renalTransplantationUrology", "P", None),
                 ("transplantSurgery", "P", "renal transplant programmes need dialysis"), ("urology", "P", None)],
    "caesarean_section": [(v, "S", "obstetrics department; C-section is core obstetric surgery") for v in OBS]
                         + [("maternalFetalMedicineOrPerinatology", "S", None),
                            ("neonatologyPerinatalMedicine", "P", None), ("anesthesia", "P", "needed for C-section; department-only, over-assigned"),
                            ("anesthesiology", "P", None), ("gynecology", "P", "gynecology alone may exclude obstetrics")],
    "normal_delivery": [(v, "S", None) for v in OBS]
                       + [("maternalFetalMedicineOrPerinatology", "S", None), ("maternalAndChildHealth", "P", None),
                          ("gynecology", "P", "gynecology alone may exclude obstetrics"),
                          ("familyPlanningAndComplexContraception", "P", None)],
    "high_risk_obstetrics": [("maternalFetalMedicineOrPerinatology", "S", None),
                             ("neonatologyPerinatalMedicine", "P", None), ("gynecologyAndObstetrics", "P", None),
                             ("obstetricsAndMaternityCare", "P", None), ("criticalCareMedicine", "P", None)],
    "nicu": [("neonatologyPerinatalMedicine", "S", "neonatology implies newborn care; NICU beds still need text evidence"),
             ("pediatricCriticalCareMedicine", "P", None), ("pediatrics", "P", None),
             ("maternalFetalMedicineOrPerinatology", "P", None)],
    "picu": [("pediatricCriticalCareMedicine", "S", None), ("pediatricEmergencyMedicine", "P", None),
             ("pediatrics", "P", None), ("neonatologyPerinatalMedicine", "P", None)],
    "icu_critical_care": [("criticalCareMedicine", "S", None), ("pediatricCriticalCareMedicine", "P", None),
                          ("anesthesia", "P", "department-only, over-assigned"), ("anesthesiology", "P", None),
                          ("emergencyMedicine", "P", None), ("pulmonology", "P", None)],
    "ventilator_support": [("criticalCareMedicine", "S", None), ("pediatricCriticalCareMedicine", "S", None),
                           ("neonatologyPerinatalMedicine", "P", None), ("pulmonology", "P", None),
                           ("anesthesia", "P", "department-only, over-assigned"), ("sleepMedicine", "P", "non-invasive ventilation")],
    "cardiac_surgery": [("cardiacSurgery", "S", None), ("cardiothoracicSurgery", "S", None),
                        ("cardiology", "P", "cardiology is medical; does not imply surgery"),
                        ("pediatricCardiology", "P", None), ("vascularSurgery", "P", None)],
    "interventional_cardiology": [("interventionalCardiology", "S", None),
                                  ("cardiology", "P", "many cardiology departments have no cath lab"),
                                  ("pediatricCardiology", "P", None), ("interventionalRadiology", "P", None)],
    "stroke_care": [("vascularNeurology", "S", None), ("cerebrovascularNeurosurgery", "S", None),
                    ("neurology", "P", None), ("neurosurgery", "P", None), ("neuroradiology", "P", None),
                    ("interventionalRadiology", "P", None), ("emergencyMedicine", "P", None)],
    "mri": [(v, "P", NO_MACHINE) for v in RADIOLOGY],
    "ct_scan": [(v, "P", NO_MACHINE) for v in RADIOLOGY],
    "xray": [("radiology", "S", "x-ray is the baseline modality of any radiology department"),
             ("diagnosticRadiology", "S", None), ("cardiothoracicRadiology", "P", None),
             ("pediatricRadiology", "P", None), ("orthopedicSurgery", "P", None)],
    "ultrasound": [("radiology", "S", "ultrasound is near-universal in radiology departments"),
                   ("diagnosticRadiology", "S", None), ("maternalFetalMedicineOrPerinatology", "P", None),
                   ("gynecologyAndObstetrics", "P", None), ("reproductiveEndocrinologyAndInfertility", "P", None),
                   ("cardiology", "P", "echocardiography")],
    "mammography": [("breastImaging", "S", None), ("breastSurgery", "P", None), ("radiology", "P", NO_MACHINE),
                    ("diagnosticRadiology", "P", NO_MACHINE), ("surgicalOncology", "P", None)],
    "nuclear_imaging": [("nuclearMedicineAndMolecularImaging", "S", None), ("nuclearMedicine", "S", None),
                        ("medicalOncology", "P", None), ("radiationOncology", "P", None)],
    "burn_care": [("burnAndTraumaPlasticSurgery", "S", None), ("plasticSurgery", "P", None),
                  ("reconstructivePlasticSurgery", "P", None),
                  ("woundHealingAndDermatologicRegenerativeMedicine", "P", None),
                  ("underseaAndHyperbaricMedicine", "P", None)],
    "trauma_emergency": [("emergencyMedicine", "S", None), ("pediatricEmergencyMedicine", "S", None),
                         ("orthopedicTraumaAndFractureSurgery", "S", None), ("burnAndTraumaPlasticSurgery", "P", None),
                         ("criticalCareMedicine", "P", None), ("generalSurgery", "P", None)],
    "ambulance": [("emergencyMedicine", "P", "no specialty names ambulance service")],
    "chemotherapy": [("medicalOncology", "S", None), ("oncology", "S", None), ("pediatricHematologyOncology", "S", None),
                     ("hematology", "P", None), ("haematology", "P", None), ("gynecologicalOncology", "P", None),
                     ("gynecologicOncology", "P", None), ("neuroOncology", "P", None), ("urologicOncology", "P", None)],
    "radiotherapy": [("radiationOncology", "S", None), ("oncology", "P", None), ("medicalOncology", "P", None)],
    "cancer_surgery": [("surgicalOncology", "S", None), ("gynecologicalOncology", "S", None),
                       ("gynecologicOncology", "S", None), ("urologicOncology", "S", None),
                       ("orthopedicOncology", "S", None), ("neuroOncology", "P", None), ("ocularOncology", "P", None),
                       ("breastSurgery", "P", None), ("headAndNeckSurgeryOtolaryngology", "P", None),
                       ("hepatopancreatobiliarySurgery", "P", None), ("oncology", "P", None)],
    "orthopedic_surgery": [(v, "S", None) for v in
                           ["orthopedicSurgery", "orthopedics", "orthopaedics", "orthopedicTraumaAndFractureSurgery",
                            "shoulderAndElbowOrthopedicSurgery", "footAndAnkleOrthopedicSurgery",
                            "pediatricOrthopedicSurgery", "jointReconstructionSurgery"]]
                          + [(v, "P", None) for v in
                             ["handOrUpperExtremityAndPeripheralNerveSurgery", "handAndUpperExtremitiesSurgery",
                              "handOrUpperExtremitiesSurgery", "orthopedicSportsMedicine"]],
    "joint_replacement": [("jointReconstructionSurgery", "S", None), ("jointReplacementSurgery", "S", None),
                          ("orthopedicSurgery", "P", None), ("orthopedics", "P", None), ("orthopaedics", "P", None),
                          ("shoulderAndElbowOrthopedicSurgery", "P", None)],
    "neurosurgery": [("neurosurgery", "S", None), ("spineNeurosurgery", "S", None), ("skullBaseNeurosurgery", "S", None),
                     ("cerebrovascularNeurosurgery", "S", None), ("pediatricNeurosurgery", "S", None),
                     ("neurology", "P", "neurology is medical; does not imply surgery")],
    "spine_surgery": [("spineNeurosurgery", "S", None), ("orthopedicSpineSurgery", "S", None),
                      ("neurosurgery", "P", None), ("orthopedicSurgery", "P", None), ("spinalCordInjuryMedicine", "P", None)],
    "general_surgery": [(v, "S", None) for v in
                        ["generalSurgery", "colorectalSurgery", "bariatricSurgery", "hepatopancreatobiliarySurgery",
                         "upperGIAndForegutSurgery", "gastrointestinalSurgery", "surgicalGastroenterology",
                         "endocrineSurgery", "pediatricSurgery", "paediatricSurgery"]]
                       + [(v, "P", None) for v in
                          ["minimallyInvasiveSurgeryAndEndourology", "breastSurgery", "vascularSurgery",
                           "generalThoracicSurgery"]]
                       + [("anesthesia", "P", "department-only, over-assigned")],
    "gi_endoscopy": [("gastroenterology", "S", "endoscopy is the core gastroenterology procedure"),
                     ("surgicalGastroenterology", "S", None), ("hepatology", "P", None),
                     ("pediatricGastroenterology", "P", None), ("upperGIAndForegutSurgery", "P", None),
                     ("colorectalSurgery", "P", None)],
    "pulmonology_respiratory": [("pulmonology", "S", None), ("pulmonaryMedicine", "S", None),
                                ("pediatricPulmonology", "S", None), ("sleepMedicine", "P", None),
                                ("sleepMedicineOtolaryngology", "P", None), ("allergyAndImmunology", "P", None),
                                ("cardiothoracicSurgery", "P", None), ("generalThoracicSurgery", "P", None)],
    "transplant": [("transplantSurgery", "S", None), ("renalTransplantationUrology", "S", None),
                   ("boneMarrowTransplant", "S", None), ("hepatopancreatobiliarySurgery", "P", None),
                   ("nephrology", "P", None), ("hepatology", "P", None), ("hematology", "P", None)],
    "blood_bank": [("hematology", "P", "no specialty names a blood bank"), ("haematology", "P", None),
                   ("pathology", "P", None), ("clinicalPathology", "P", None)],
    "laboratory_pathology": [(v, "S", None) for v in
                             ["pathology", "clinicalPathology", "anatomicPathology", "histopathology",
                              "microbiology", "biochemistry"]]
                            + [("oralPathology", "P", None), ("hematology", "P", None)],
    "dental": [(v, "S", None) for v in
               ["dentistry", "generalDentistry", "endodontics", "orthodontics", "prosthodontics", "periodontics",
                "pediatricDentistry", "pedodontics", "aestheticDentistry", "cosmeticDentistry",
                "oralAndMaxillofacialSurgery", "dentoalveolarSurgery", "dentalImplants", "implantology",
                "restorativeDentistry", "laserDentistry", "preventiveDentistry", "oralMedicine"]]
              + [(v, "P", None) for v in
                 ["dentalAnesthesia", "oralPathology", "orthognathicSurgery",
                  "craniofacialAndCleftOralMaxillofacialSurgery", "cosmeticMaxillofacialSurgery"]],
    "eye_surgery": [(v, "S", None) for v in
                    ["cataractAndAnteriorSegmentSurgery", "retinaAndVitreoretinalOphthalmology",
                     "corneaOphthalmology", "glaucomaOphthalmology", "refractiveSurgeryOphthalmology",
                     "oculoplasticsAndReconstructiveOrbitalSurgery"]]
                   + [("ophthalmology", "P", "general ophthalmology; cataract surgery common but not certain")]
                   + [(v, "P", None) for v in
                      ["pediatricsAndStrabismusOphthalmology", "pediatricOphthalmology", "ocularOncology",
                       "eyeTraumaAndEmergencyEyeCare"]],
    "ophthalmology_medical": [(v, "S", None) for v in
                              ["ophthalmology", "neuroOphthalmology", "uveitisOphthalmology", "pediatricOphthalmology",
                               "pediatricsAndStrabismusOphthalmology"]]
                             + [(v, "P", None) for v in
                                ["glaucomaOphthalmology", "retinaAndVitreoretinalOphthalmology", "corneaOphthalmology"]],
    "physiotherapy_rehab": [(v, "S", None) for v in
                            ["physicalMedicineAndRehabilitation", "physiotherapy", "sportsMedicinePMR",
                             "pediatricRehabilitationMedicine", "occupationalAndVocationalRehabilitation"]]
                           + [(v, "P", None) for v in
                              ["neuromuscularMedicinePMR", "painMedicinePMR", "spinalCordInjuryMedicine",
                               "amputeeAndProstheticsAndOrthoticsRehabilitation", "orthopedicSportsMedicine"]],
    "mental_health": [(v, "S", None) for v in
                      ["psychiatry", "addictionPsychiatry", "childAndAdolescentPsychiatry", "geriatricPsychiatry",
                       "neuropsychiatry", "addictionMedicinePM"]]
                     + [(v, "P", None) for v in
                        ["psychology", "clinicalPsychology", "developmental–behavioralPediatrics",
                         "pediatricNeurodevelopmentalDisabilities"]],
    "outpatient_opd": [(v, "P", "OPD is a care format; any clinical specialty implies some OPD, none is strong backing")
                       for v in ["internalMedicine", "familyMedicine", "generalMedicine", "geriatricsInternalMedicine",
                                 "adolescentMedicine", "preventiveMedicine",
                                 "chronicDiseasePreventionAndLifestyleMedicine"]],
    "infertility_ivf": [("reproductiveEndocrinologyAndInfertility", "S", None),
                        ("andrologyAndMaleFertility", "S", None), ("andrology", "P", None),
                        ("gynecologyAndObstetrics", "P", None)],
    "dermatology": [(v, "S", None) for v in
                    ["dermatology", "cosmeticDermatology", "pediatricDermatology", "proceduralDermatology",
                     "hairAndNailDisorders"]]
                   + [("cosmetology", "P", None), ("woundHealingAndDermatologicRegenerativeMedicine", "P", None)],
    "ent": [(v, "S", None) for v in
            ["otolaryngology", "ent", "headAndNeckSurgeryOtolaryngology", "pediatricOtolaryngology", "laryngology",
             "rhinology", "neurotologyAndOtology"]]
           + [("sleepMedicineOtolaryngology", "P", None), ("audiology", "P", None)],
    "pediatrics": [(v, "S", None) for v in
                   ["pediatrics", "neonatologyPerinatalMedicine", "pediatricCriticalCareMedicine",
                    "pediatricEmergencyMedicine"]]
                  + [(v, "P", "pediatric subspecialty") for v in
                     ["pediatricCardiology", "pediatricSurgery", "adolescentMedicine",
                      "developmental–behavioralPediatrics", "pediatricEndocrinology", "pediatricGastroenterology",
                      "pediatricNephrology", "pediatricNeurology", "childNeurology", "pediatricPulmonology",
                      "pediatricHematologyOncology", "pediatricInfectiousDiseases", "maternalAndChildHealth"]],
    "obstetrics_maternity": [(v, "S", None) for v in OBS + ["maternalFetalMedicineOrPerinatology"]]
                            + [(v, "P", None) for v in
                               ["maternalAndChildHealth", "gynecology", "familyPlanningAndComplexContraception",
                                "menopauseAndMidlifeHealth", "urogynecologyAndReconstructivePelvisSurgery"]],
    "cosmetic_aesthetic": [(v, "S", None) for v in
                           ["aestheticAndCosmeticSurgery", "cosmeticSurgery", "cosmeticDermatology", "cosmetology"]]
                          + [(v, "P", None) for v in
                             ["plasticSurgery", "reconstructivePlasticSurgery", "cosmeticMaxillofacialSurgery",
                              "aestheticDentistry", "cosmeticDentistry", "hairAndNailDisorders"]],
    "prosthetics_orthotics": [("amputeeAndProstheticsAndOrthoticsRehabilitation", "S", None),
                              ("podiatry", "P", None), ("podiatricDiabeticLimbSalvageAndWoundCare", "P", None),
                              ("physicalMedicineAndRehabilitation", "P", None), ("orthopedicSurgery", "P", None)],
    "nephrology_renal": [("nephrology", "S", None), ("pediatricNephrology", "S", None),
                         ("renalTransplantationUrology", "P", None), ("urology", "P", None)],
    "gastroenterology": [("gastroenterology", "S", None), ("hepatology", "S", None),
                         ("pediatricGastroenterology", "S", None), ("surgicalGastroenterology", "S", None),
                         ("hepatopancreatobiliarySurgery", "P", None), ("gastrointestinalSurgery", "P", None)],
    "other_clinical": [],
    # --- pa_single_v2 labels ---
    "urology": [("urology", "S", None), ("pediatricUrology", "S", None),
                ("minimallyInvasiveSurgeryAndEndourology", "P", "endourology is mostly stone/prostate work, but the term also covers general laparoscopy"),
                ("urologicOncology", "P", "cancer subspecialty; cancer surgery has its own label"),
                ("andrologyAndMaleFertility", "P", None), ("andrology", "P", None),
                ("renalTransplantationUrology", "P", None),
                ("urogynecologyAndReconstructivePelvisSurgery", "P", None), ("nephrology", "P", None)],
    "diabetes_endocrinology": [("endocrinologyAndDiabetesAndMetabolism", "S", None), ("diabetology", "S", None),
                               ("endocrinology", "S", None), ("pediatricEndocrinology", "S", None),
                               ("endocrineSurgery", "P", "thyroid surgery; implies thyroid care"),
                               ("podiatricDiabeticLimbSalvageAndWoundCare", "P", "diabetic foot care"),
                               ("podiatry", "P", None),
                               ("chronicDiseasePreventionAndLifestyleMedicine", "P", None),
                               ("internalMedicine", "P", "department-only, over-assigned")],
    "health_checkup_vaccination": [("preventiveMedicine", "P", "no specialty names check-up packages or vaccination; department-only, over-assigned"),
                                   ("chronicDiseasePreventionAndLifestyleMedicine", "P", None),
                                   ("familyMedicine", "P", "department-only, over-assigned")],
    "cardiac_diagnostics": [("cardiology", "S", "cardiology department runs echo, TMT and consultations; still a department, not proof of a test"),
                            ("pediatricCardiology", "S", None), ("interventionalCardiology", "P", None),
                            ("internalMedicine", "P", "department-only, over-assigned")],
    "neurology": [("neurology", "S", None), ("headacheMedicine", "S", None), ("epilepsyNeurology", "S", None),
                  ("clinicalNeurophysiology", "S", None), ("pediatricNeurology", "S", None),
                  ("childNeurology", "S", None), ("cognitiveAndBehavioralNeurology", "S", None),
                  ("brainInjuryNeurology", "P", None), ("vascularNeurology", "P", "stroke has its own label"),
                  ("neurosurgery", "P", "neurosurgery is surgical; does not imply a neurology clinic"),
                  ("sleepMedicine", "P", None)],
    "ayush_traditional_medicine": [],  # no AYUSH specialty term reaches 20 facilities
    "gynecology": [(v, "S", "obstetrics and gynaecology department covers non-pregnancy gynaecology") for v in OBS
                   if v != "obstetricsAndMaternityCare"]
                  + [("gynecology", "S", None), ("gynecologicalSurgery", "S", None),
                     ("menopauseAndMidlifeHealth", "S", None),
                     ("urogynecologyAndReconstructivePelvisSurgery", "S", None),
                     ("obstetricsAndMaternityCare", "P", "maternity-only term"),
                     ("familyPlanningAndComplexContraception", "P", None),
                     ("reproductiveEndocrinologyAndInfertility", "P", "fertility has its own label"),
                     ("gynecologicalOncology", "P", "cancer subspecialty"), ("gynecologicOncology", "P", None)],
    "proctology": [("colorectalSurgery", "S", None), ("generalSurgery", "P", "general surgeons often treat piles; not proof"),
                   ("surgicalGastroenterology", "P", None), ("gastroenterology", "P", None)],
    "pain_management": [("painMedicinePMR", "S", None), ("painManagement", "S", None),
                        ("anesthesiology", "P", "pain clinics are often run by anaesthetists"),
                        ("anesthesia", "P", "department-only, over-assigned"),
                        ("physicalMedicineAndRehabilitation", "P", None)],
    "vascular_surgery": [("vascularSurgery", "S", None),
                         ("cardiothoracicSurgery", "P", None), ("interventionalRadiology", "P", None),
                         ("generalSurgery", "P", "general surgeons treat varicose veins; not proof")],
    "cardiology_medical": [("cardiology", "S", "cardiology department = general heart care"),
                           ("pediatricCardiology", "S", None), ("interventionalCardiology", "P", None),
                           ("internalMedicine", "P", "department-only, over-assigned")],
}

# implied_by_service: SERVICE <- [(other service whose evidence implies SERVICE is possible, strength, note)]
IMPLIED = {
    "dialysis": [("nephrology_renal", "S", "nephrology services usually include dialysis"),
                 ("transplant", "P", "renal transplant needs dialysis support"),
                 ("icu_critical_care", "P", "ICUs often run CRRT / bedside dialysis")],
    "caesarean_section": [("obstetrics_maternity", "S", "obstetric units perform C-sections"),
                          ("high_risk_obstetrics", "S", None), ("normal_delivery", "P", None), ("nicu", "P", None)],
    "normal_delivery": [("obstetrics_maternity", "S", None), ("caesarean_section", "S", None),
                        ("high_risk_obstetrics", "S", None)],
    "high_risk_obstetrics": [("nicu", "P", None), ("caesarean_section", "P", None), ("icu_critical_care", "P", None)],
    "nicu": [("picu", "P", None), ("high_risk_obstetrics", "P", "high-risk deliveries need newborn intensive care")],
    "picu": [("nicu", "P", None), ("icu_critical_care", "P", None)],
    "icu_critical_care": [("ventilator_support", "S", None), ("cardiac_surgery", "S", "post-op cardiac ICU required"),
                          ("transplant", "S", None), ("neurosurgery", "P", None), ("trauma_emergency", "P", None),
                          ("picu", "P", None), ("nicu", "P", None)],
    "ventilator_support": [("icu_critical_care", "S", None), ("nicu", "S", None), ("picu", "S", None)],
    "cardiac_surgery": [("interventional_cardiology", "P", None)],
    "interventional_cardiology": [("cardiac_surgery", "S", "cardiac surgical centres almost always have a cath lab")],
    "stroke_care": [("neurosurgery", "P", None), ("mri", "P", None), ("ct_scan", "P", None),
                    ("icu_critical_care", "P", None)],
    "mri": [("neurosurgery", "P", None), ("stroke_care", "P", None), ("spine_surgery", "P", None)],
    "ct_scan": [("trauma_emergency", "P", None), ("neurosurgery", "P", None), ("stroke_care", "P", None),
                ("mri", "P", "MRI sites usually also have CT")],
    "xray": [("orthopedic_surgery", "P", None), ("trauma_emergency", "P", None), ("ct_scan", "P", None)],
    "ultrasound": [("obstetrics_maternity", "P", None), ("infertility_ivf", "P", None)],
    "mammography": [("cancer_surgery", "P", None)],
    "nuclear_imaging": [("chemotherapy", "P", None), ("radiotherapy", "P", None)],
    "burn_care": [("trauma_emergency", "P", None), ("icu_critical_care", "P", None)],
    "trauma_emergency": [("icu_critical_care", "P", None), ("ambulance", "P", None), ("burn_care", "P", None)],
    "ambulance": [("trauma_emergency", "P", None), ("icu_critical_care", "P", None)],
    "chemotherapy": [("radiotherapy", "P", None), ("cancer_surgery", "P", None)],
    "radiotherapy": [("chemotherapy", "P", None), ("nuclear_imaging", "P", None)],
    "cancer_surgery": [("chemotherapy", "P", None), ("radiotherapy", "P", None)],
    "orthopedic_surgery": [("joint_replacement", "S", "joint replacement is orthopaedic surgery"),
                           ("spine_surgery", "P", None), ("trauma_emergency", "P", None)],
    "joint_replacement": [("orthopedic_surgery", "P", None)],
    "neurosurgery": [("spine_surgery", "P", None), ("stroke_care", "P", None)],
    "spine_surgery": [("neurosurgery", "P", None), ("orthopedic_surgery", "P", None)],
    "general_surgery": [("cancer_surgery", "P", None), ("gi_endoscopy", "P", None)],
    "gi_endoscopy": [("gastroenterology", "S", None)],
    "pulmonology_respiratory": [("ventilator_support", "P", None)],
    "transplant": [],
    "blood_bank": [("cardiac_surgery", "P", None), ("transplant", "P", None), ("trauma_emergency", "P", None),
                   ("caesarean_section", "P", None), ("chemotherapy", "P", None)],
    "laboratory_pathology": [("blood_bank", "P", None)],
    "dental": [],
    "eye_surgery": [("ophthalmology_medical", "P", None)],
    "physiotherapy_rehab": [("orthopedic_surgery", "P", None), ("joint_replacement", "P", None),
                            ("stroke_care", "P", None)],
    "mental_health": [],
    "outpatient_opd": [],
    "infertility_ivf": [],
    "dermatology": [("cosmetic_aesthetic", "P", None)],
    "ent": [],
    "ophthalmology_medical": [("eye_surgery", "S", None)],
    "pediatrics": [("nicu", "P", None), ("picu", "P", None)],
    "obstetrics_maternity": [("normal_delivery", "S", None), ("caesarean_section", "S", None),
                             ("high_risk_obstetrics", "S", None)],
    "cosmetic_aesthetic": [],
    "prosthetics_orthotics": [],
    "nephrology_renal": [("dialysis", "S", "a dialysis unit is a nephrology service"), ("transplant", "P", None)],
    "gastroenterology": [("gi_endoscopy", "S", None)],
    "other_clinical": [],
    # --- pa_single_v2 labels ---
    "urology": [("nephrology_renal", "P", None), ("transplant", "P", None)],
    "diabetes_endocrinology": [],
    "health_checkup_vaccination": [("laboratory_pathology", "P", None), ("pediatrics", "P", "child immunisation")],
    # no implied links: a cardiac_surgery / interventional claim + cardiology department made
    # ~960 Possible rows from department lists; removed on review
    "cardiac_diagnostics": [],
    "neurology": [("stroke_care", "P", None), ("neurosurgery", "P", None)],
    "ayush_traditional_medicine": [],
    "gynecology": [("obstetrics_maternity", "P", None), ("infertility_ivf", "P", None)],
    "proctology": [("general_surgery", "P", None)],
    "pain_management": [("physiotherapy_rehab", "P", None), ("orthopedic_surgery", "P", None)],
    "vascular_surgery": [("dialysis", "P", "dialysis access (AV fistula) is vascular surgery")],
    "cardiology_medical": [],  # implied links removed on review (same reason as cardiac_diagnostics)
}

# ---------------------------------------------------------------------------
# service_type_fit source
# ---------------------------------------------------------------------------
ALL = set(SERVICES)
T = {}


def spec(default, expected=(), plausible=(), unexpected=(), why=""):
    return {"default": default, "expected": set(expected), "plausible": set(plausible),
            "unexpected": set(unexpected), "why": why}


HOSP_CORE = ["icu_critical_care", "ventilator_support", "trauma_emergency", "ambulance", "ct_scan", "xray",
             "ultrasound", "laboratory_pathology", "outpatient_opd", "general_surgery", "orthopedic_surgery",
             "obstetrics_maternity", "normal_delivery", "caesarean_section", "pediatrics", "blood_bank",
             "physiotherapy_rehab", "gi_endoscopy", "pulmonology_respiratory", "gastroenterology", "nephrology_renal",
             "dialysis"]
TYPE_SPEC = {
    "multispecialty_hospital": spec("plausible", HOSP_CORE + ["mri", "interventional_cardiology"],
                                    why="broad hospital"),
    "medical_college_teaching_hospital": spec("plausible", HOSP_CORE + [
        "mri", "nicu", "high_risk_obstetrics", "mental_health", "dental", "ent", "ophthalmology_medical",
        "eye_surgery", "dermatology", "neurosurgery"], why="teaching hospital with mandated departments"),
    "government_health_centre": spec("plausible",
                                     ["outpatient_opd", "normal_delivery", "obstetrics_maternity", "pediatrics",
                                      "laboratory_pathology"],
                                     unexpected=["cardiac_surgery", "transplant", "radiotherapy", "nuclear_imaging",
                                                 "infertility_ivf", "cosmetic_aesthetic"],
                                     why="PHC/CHC up to district hospital; tertiary services unexpected"),
    "general_hospital_or_nursing_home": spec("plausible",
                                             ["outpatient_opd", "laboratory_pathology", "xray", "ultrasound",
                                              "general_surgery", "normal_delivery", "caesarean_section",
                                              "obstetrics_maternity", "pediatrics"],
                                             unexpected=["transplant", "radiotherapy", "nuclear_imaging"],
                                             why="general hospital / nursing home"),
    "maternity_women_children": spec("unexpected",
                                     ["normal_delivery", "caesarean_section", "high_risk_obstetrics",
                                      "obstetrics_maternity", "nicu", "pediatrics", "ultrasound", "outpatient_opd",
                                      "laboratory_pathology"],
                                     ["picu", "infertility_ivf", "mammography", "icu_critical_care",
                                      "ventilator_support", "blood_bank", "ambulance", "general_surgery",
                                      "cancer_surgery", "physiotherapy_rehab", "trauma_emergency",
                                      "cosmetic_aesthetic"], why="maternity / women and children"),
    "pediatric": spec("unexpected", ["pediatrics", "nicu", "picu", "outpatient_opd", "laboratory_pathology"],
                      ["ventilator_support", "icu_critical_care", "ultrasound", "general_surgery", "ambulance",
                       "trauma_emergency", "physiotherapy_rehab", "mental_health", "dental", "ent",
                       "ophthalmology_medical", "dermatology", "pulmonology_respiratory", "gastroenterology",
                       "nephrology_renal", "blood_bank", "ct_scan", "mri", "orthopedic_surgery", "neurosurgery",
                       "cardiac_surgery", "interventional_cardiology", "dialysis", "chemotherapy"],
                      why="children's hospital/clinic"),
    "eye": spec("unexpected", ["eye_surgery", "ophthalmology_medical", "outpatient_opd"],
                ["ultrasound", "cosmetic_aesthetic", "cancer_surgery"], why="eye facility (B-scan, oculoplastics, ocular oncology plausible)"),
    "dental": spec("unexpected", ["dental", "outpatient_opd"], ["cosmetic_aesthetic"], why="dental facility"),
    "ent_hearing_speech": spec("unexpected", ["ent", "outpatient_opd"],
                               ["cosmetic_aesthetic", "cancer_surgery", "pulmonology_respiratory",
                                "physiotherapy_rehab"], why="ENT / hearing / speech"),
    "skin_hair_cosmetic": spec("unexpected", ["dermatology", "cosmetic_aesthetic", "outpatient_opd"], ["dental"],
                               why="skin / hair / cosmetic"),
    "ivf_fertility": spec("unexpected", ["infertility_ivf", "outpatient_opd", "ultrasound", "laboratory_pathology"],
                          ["obstetrics_maternity", "high_risk_obstetrics", "normal_delivery", "caesarean_section",
                           "general_surgery", "nicu"], why="IVF / fertility"),
    "kidney_urology": spec("unexpected",
                           ["nephrology_renal", "dialysis", "outpatient_opd", "laboratory_pathology", "ultrasound"],
                           ["transplant", "general_surgery", "ct_scan", "icu_critical_care", "cancer_surgery",
                            "infertility_ivf", "ambulance"], why="kidney / urology"),
    "cardiac": spec("unexpected", ["cardiac_surgery", "interventional_cardiology", "icu_critical_care",
                                   "outpatient_opd", "laboratory_pathology", "ultrasound", "xray"],
                    ["ventilator_support", "ct_scan", "mri", "nuclear_imaging", "trauma_emergency", "ambulance",
                     "blood_bank", "stroke_care", "transplant", "physiotherapy_rehab", "pulmonology_respiratory"],
                    why="cardiac centre"),
    "cancer": spec("unexpected", ["chemotherapy", "radiotherapy", "cancer_surgery", "outpatient_opd",
                                  "laboratory_pathology", "ct_scan"],
                   ["mri", "nuclear_imaging", "mammography", "ultrasound", "icu_critical_care",
                    "ventilator_support", "blood_bank", "transplant", "gi_endoscopy", "physiotherapy_rehab",
                    "general_surgery"], why="cancer centre"),
    "orthopedic_spine": spec("unexpected", ["orthopedic_surgery", "joint_replacement", "spine_surgery",
                                            "physiotherapy_rehab", "xray", "outpatient_opd"],
                             ["trauma_emergency", "mri", "ct_scan", "laboratory_pathology", "prosthetics_orthotics",
                              "neurosurgery", "icu_critical_care", "ambulance", "ultrasound", "cancer_surgery"],
                             why="orthopaedic / spine"),
    "neuro": spec("unexpected", ["neurosurgery", "stroke_care", "outpatient_opd", "mri", "ct_scan"],
                  ["spine_surgery", "icu_critical_care", "ventilator_support", "trauma_emergency",
                   "physiotherapy_rehab", "mental_health", "laboratory_pathology", "ambulance", "cancer_surgery"],
                  why="neuro centre"),
    "gastro_liver": spec("unexpected", ["gastroenterology", "gi_endoscopy", "outpatient_opd", "laboratory_pathology",
                                        "ultrasound"],
                         ["general_surgery", "cancer_surgery", "transplant", "ct_scan", "mri", "icu_critical_care",
                          "blood_bank", "chemotherapy"], why="gastro / liver"),
    "diagnostic_imaging_lab": spec("unexpected", ["mri", "ct_scan", "xray", "ultrasound", "mammography",
                                                  "laboratory_pathology"],
                                   ["nuclear_imaging", "outpatient_opd"],
                                   why="imaging / lab only; no treatment or surgery"),
    "physiotherapy_rehab": spec("unexpected", ["physiotherapy_rehab", "outpatient_opd"], ["prosthetics_orthotics"],
                                why="physiotherapy / rehab"),
    "mental_health_deaddiction": spec("unexpected", ["mental_health", "outpatient_opd"], [],
                                      why="mental health / de-addiction"),
    "ayush_alternative": spec("unexpected", ["outpatient_opd"], ["physiotherapy_rehab", "dermatology"],
                              why="AYUSH / alternative medicine; allopathic procedures unexpected"),
    "home_care_or_ambulance": spec("unexpected", ["ambulance"],
                                   ["physiotherapy_rehab", "outpatient_opd", "icu_critical_care",
                                    "ventilator_support", "dialysis", "trauma_emergency"],
                                   why="home care / ambulance (home ICU, home dialysis plausible)"),
    "general_clinic_polyclinic": spec("unexpected", ["outpatient_opd"],
                                      ["ultrasound", "pediatrics", "dermatology", "ent", "ophthalmology_medical",
                                       "dental", "mental_health", "physiotherapy_rehab", "obstetrics_maternity",
                                       "gastroenterology", "pulmonology_respiratory", "nephrology_renal",
                                       "infertility_ivf", "cosmetic_aesthetic", "gi_endoscopy", "general_surgery"],
                                      why="clinic / polyclinic: consultations, minor procedures"),
    "other_single_specialty": spec("plausible",
                                   unexpected=["transplant", "cardiac_surgery", "nicu", "picu", "caesarean_section",
                                               "normal_delivery", "high_risk_obstetrics", "blood_bank",
                                               "radiotherapy", "neurosurgery"],
                                   why="specialty unknown (diabetes, pulmonology, colorectal, burns...); "
                                       "major-hospital services unexpected"),
    "not_a_care_facility": spec("unexpected", why="not a care facility"),
    "unknown": spec("plausible", why="type unknown"),
}
# pa_single_v2 labels: added to the per-type lists above (types not listed keep their default:
# plausible for broad hospitals / other_single_specialty / unknown, unexpected elsewhere).
NEW_FIT_V2 = {'multispecialty_hospital': {'expected': ['urology', 'cardiac_diagnostics', 'gynecology', 'diabetes_endocrinology', 'neurology', 'health_checkup_vaccination']},
    'medical_college_teaching_hospital': {'expected': ['urology', 'cardiac_diagnostics', 'gynecology', 'diabetes_endocrinology', 'neurology', 'health_checkup_vaccination']},
    'government_health_centre': {'expected': ['health_checkup_vaccination']},
    'general_hospital_or_nursing_home': {'expected': ['gynecology']},
    'maternity_women_children': {'expected': ['gynecology'], 'plausible': ['health_checkup_vaccination', 'diabetes_endocrinology']},
    'pediatric': {'expected': ['health_checkup_vaccination'], 'plausible': ['neurology', 'diabetes_endocrinology', 'urology', 'cardiac_diagnostics']},
    'ent_hearing_speech': {'plausible': ['neurology']},
    'skin_hair_cosmetic': {'plausible': ['ayush_traditional_medicine']},
    'ivf_fertility': {'expected': ['gynecology'], 'plausible': ['urology', 'diabetes_endocrinology']},
    'kidney_urology': {'expected': ['urology'], 'plausible': ['diabetes_endocrinology']},
    'cardiac': {'expected': ['cardiac_diagnostics'], 'plausible': ['diabetes_endocrinology', 'health_checkup_vaccination', 'neurology']},
    'cancer': {'plausible': ['gynecology', 'urology', 'pain_management', 'health_checkup_vaccination', 'proctology']},
    'orthopedic_spine': {'expected': ['pain_management'], 'plausible': ['neurology']},
    'neuro': {'expected': ['neurology'], 'plausible': ['pain_management']},
    'gastro_liver': {'expected': ['proctology'], 'plausible': ['health_checkup_vaccination', 'diabetes_endocrinology']},
    'diagnostic_imaging_lab': {'expected': ['health_checkup_vaccination'], 'plausible': ['cardiac_diagnostics']},
    'physiotherapy_rehab': {'expected': ['pain_management'], 'plausible': ['neurology', 'ayush_traditional_medicine']},
    'mental_health_deaddiction': {'plausible': ['neurology']},
    'ayush_alternative': {'expected': ['ayush_traditional_medicine'], 'plausible': ['pain_management']},
    'home_care_or_ambulance': {'plausible': ['health_checkup_vaccination', 'pain_management']},
    'general_clinic_polyclinic': {'plausible': ['diabetes_endocrinology', 'health_checkup_vaccination', 'cardiac_diagnostics', 'gynecology', 'neurology', 'urology', 'proctology', 'pain_management', 'ayush_traditional_medicine']}}
# vascular_surgery and cardiology_medical (added after the first 9)
for _ft, _lvl, _svc in [
        ("multispecialty_hospital", "expected", "cardiology_medical"),
        ("medical_college_teaching_hospital", "expected", "cardiology_medical"),
        ("medical_college_teaching_hospital", "expected", "vascular_surgery"),
        ("cardiac", "expected", "cardiology_medical"), ("cardiac", "plausible", "vascular_surgery"),
        ("pediatric", "plausible", "cardiology_medical"),
        ("general_clinic_polyclinic", "plausible", "cardiology_medical"),
        ("kidney_urology", "plausible", "vascular_surgery"),
        ("home_care_or_ambulance", "plausible", "cardiology_medical")]:
    NEW_FIT_V2.setdefault(_ft, {}).setdefault(_lvl, []).append(_svc)
for _ft, _lv in NEW_FIT_V2.items():
    for _lvl, _svcs in _lv.items():
        TYPE_SPEC[_ft][_lvl] |= set(_svcs)
NON_CARE = {"not_a_care_facility", "unknown"}
XRAY_NOT_FLOOR = {"eye", "skin_hair_cosmetic", "ivf_fertility", "mental_health_deaddiction",
                  "home_care_or_ambulance", "physiotherapy_rehab", "ayush_alternative", "ent_hearing_speech"}

EXCEPTIONS = {
    ("xray", "dental"): ("plausible", "dental x-ray / OPG is routine in dental clinics"),
    ("cancer_surgery", "dental"): ("unexpected", "oral cancer surgery belongs to hospitals, not dental clinics"),
    ("blood_bank", "diagnostic_imaging_lab"): ("unexpected", "labs collect samples; blood banks are licensed separately"),
    ("dialysis", "general_clinic_polyclinic"): ("unexpected", "standalone dialysis centres are typed kidney_urology"),
    ("burn_care", "other_single_specialty"): ("plausible", "burn centres have no own type and fall here"),
    ("mri", "government_health_centre"): ("plausible", "district hospitals sometimes have MRI (often PPP)"),
    ("dialysis", "cardiac"): ("plausible", "changed after blind validation: cardiac hospitals often run dialysis units"),
}


# ---------------------------------------------------------------------------
# service_terms source: words / phrases that NAME each service (wave 1, proposal 1).
# Used by scripts/build_facility_service.py: Confirmed needs >= 1 non-demoted explicit/stated
# claim sentence matching the service's terms (whole word / phrase, case-insensitive).
# Plain strings are phrases (spaces match space or hyphen); "re:" strings are Java regex.
# HAND_CHECKED: the 8 headline services, reviewed for synonyms and false friends.
# Others: built from the label, readable name and obvious synonyms (source 'generated');
# SYNONYM_PASS: synonyms added after reading demoted quotes (source 'synonym_pass').
# other_clinical has no terms (catch-all) and is never gated.
# ---------------------------------------------------------------------------
HAND_CHECKED = {"dialysis", "caesarean_section", "nicu", "cardiac_surgery", "mri", "burn_care",
                "icu_critical_care", "trauma_emergency"}
TERMS = {
    # --- hand-checked ---
    "cardiac_surgery": [r"re:cardiac\s+surg\w*", r"re:heart\s+surg\w*", "cabg", "ctvs",
                        r"re:coronary\s+(artery\s+)?bypass\w*", r"re:bypass\s+(surg\w*|graft\w*)",
                        r"re:valve\s+(replacement|repair|surg\w*)\w*", r"re:open[\s-]+heart",
                        r"re:cardio[\s-]?thoracic\w*", r"re:cardiothoracic\w*"],
    # not "cardiology" alone, not "operating theatre", not TAVI (interventional)
    "caesarean_section": [r"re:ca?esa?r(e|i|ea|ia)n\w*", r"re:caeserean\w*", r"re:c[\s-]?sections?",
                          "lscs", r"re:operative\s+deliver\w*"],
    "nicu": [r"re:n\.?i\.?c\.?u\.?s?", r"re:neo[\s-]?natal\s+(intensive|icu|critical)\w*",
             r"re:new[\s-]?born\s+(intensive|icu|critical)\w*",
             r"re:level[\s-]*(ii|iii|2|3)\s+(neonatal|nicu|newborn)\w*"],
    # SNCU (special newborn care unit) is NOT NICU and does not match
    "dialysis": [r"re:dialy[sz]\w*", r"re:ha?emo[\s-]?dialy\w*", "capd", "crrt",
                 "renal replacement therapy"],
    "mri": ["mri", r"re:m\.r\.i\.?", "magnetic resonance", "tesla", r"re:\d(\.\d)?\s?t(?![a-z])",
            r"re:mr\s+(scan|imaging|angiograph)\w*"],
    "burn_care": [r"re:burns?\s+(unit|ward|care|cent(re|er)|icu|treatment|management|surg\w*|injur\w*)",
                  r"re:(treat\w*|management|care)\s+of\s+burns?", r"re:burns?\s+(and|&)\s+(plastic|trauma)\w*"],
    "icu_critical_care": [r"re:i\.?c\.?u\.?s?", "iccu", "ccu", "micu", "sicu", "cticu",
                          "intensive care", "critical care"],
    "trauma_emergency": [r"re:emergency\s+(department|room|ward|unit|services?|care|medicine|cent(re|er))",
                         "casualty", r"re:trauma\s+(care|cent(re|er)|unit|ward|team|services?|surg\w*|icu)",
                         r"re:accident\s+(and|&)\s+emergency", r"re:a\s?&\s?e", "polytrauma"],
    # --- generated (not hand-checked) ---
    "normal_delivery": [r"re:normal\s+deliver\w*", r"re:vaginal\s+(deliver\w*|birth)", r"re:deliver(y|ies)",
                        "labour room", "labor room", "childbirth", "painless delivery", "birthing"],
    "high_risk_obstetrics": [r"re:high[\s-]+risk\s+(pregnan\w*|obstetric\w*|deliver\w*)",
                             r"re:maternal[\s-]+fetal\s+medicine", "perinatology", "fetal medicine"],
    "picu": [r"re:p\.?i\.?c\.?u\.?s?", r"re:(paediatric|pediatric)\s+(intensive|icu|critical)\w*"],
    "ventilator_support": [r"re:ventilat\w*", "bipap", "cpap", "life support"],
    "interventional_cardiology": [r"re:angioplast\w*", r"re:angiograph\w*", r"re:angiogram\w*",
                                  r"re:cath(eterisation|eterization)?\s*lab\w*", "cathlab", "ptca",
                                  r"re:stent\w*", r"re:pacemaker\w*", "tavi", "tavr",
                                  "interventional cardiology"],
    "stroke_care": [r"re:strokes?", "thrombolysis", "thrombectomy", "brain attack"],
    "ct_scan": [r"re:ct(\s+scan\w*)?", r"re:c\.t\.", "cat scan", "computed tomography",
                r"re:\d+[\s-]*slice\w*"],
    "xray": [r"re:x[\s-]?rays?", r"re:radiograph\w*", "dr system", "cr system"],
    "ultrasound": [r"re:ultrasound\w*", r"re:ultrasonograph\w*", "usg", r"re:sonograph\w*",
                   r"re:doppler\w*", r"re:color\s+doppler"],
    "mammography": [r"re:mammogra\w*"],
    "nuclear_imaging": [r"re:pet[\s-]?ct", r"re:pet\s+scan\w*", "spect", r"re:nuclear\s+(medicine|imaging|scan\w*)",
                        "gamma camera", r"re:bone\s+scan\w*"],
    "ambulance": [r"re:ambulances?"],
    "chemotherapy": [r"re:chemo\w*", "immunotherapy", "targeted therapy", "day care oncology"],
    "radiotherapy": [r"re:radiotherap\w*", r"re:radiation\s+(therap\w*|oncology)", r"re:linac\w*",
                     r"re:linear\s+accelerator\w*", r"re:brachytherap\w*", "cyberknife", "tomotherapy",
                     "gamma knife"],
    "cancer_surgery": [r"re:(cancer|onco|tumou?r)\w*\s+surg\w*", r"re:surgical\s+oncolog\w*", "oncosurgery",
                       r"re:mastectom\w*", r"re:\w*ectomy\s+for\s+cancer", r"re:tumou?r\s+(removal|excision|resection)"],
    "orthopedic_surgery": [r"re:orthop(a)?edic\w*", r"re:fractur\w*", r"re:arthroscop\w*", r"re:bone\s+surg\w*",
                           "ligament reconstruction", "acl reconstruction"],
    "joint_replacement": [r"re:(joint|knee|hip|shoulder)\s+replacement\w*", r"re:arthroplast\w*", "tkr", "thr"],
    "neurosurgery": [r"re:neuro[\s-]?surg\w*", r"re:brain\s+(surg\w*|tumou?r)", r"re:cranio\w*",
                     "skull base"],
    "spine_surgery": [r"re:spin(e|al)\s+surg\w*", r"re:discectom\w*", r"re:laminectom\w*",
                      r"re:spinal\s+fusion", r"re:endoscopic\s+spine"],
    "general_surgery": [r"re:general\s+surg\w*", r"re:laparoscop\w*", r"re:hernia\w*", r"re:appendic\w*",
                        r"re:gall\s?bladder", r"re:cholecystectom\w*", r"re:surger(y|ies)"],
    "gi_endoscopy": [r"re:endoscop\w*", r"re:colonoscop\w*", r"re:gastroscop\w*", "ercp", "eus",
                     r"re:sigmoidoscop\w*"],
    "pulmonology_respiratory": [r"re:pulmon\w*", r"re:respirat\w*", r"re:chest\s+(medicine|physician|diseases?)",
                                r"re:bronchoscop\w*", "spirometry", "pft", "lung function",
                                r"re:sleep\s+(study|studies|lab)", r"re:asthma\w*", "copd", "tuberculosis"],
    "transplant": [r"re:transplant\w*"],
    "blood_bank": [r"re:blood\s+(bank|storage|transfusion)\w*", "blood component"],
    "laboratory_pathology": [r"re:laborator(y|ies)", r"re:lab(s)?", r"re:patholog\w*", r"re:blood\s+tests?",
                             r"re:diagnostic\s+tests?", "biochemistry", "microbiology", "haematology",
                             "hematology", r"re:sample\s+collection"],
    "dental": [r"re:dent\w*", r"re:teeth", r"re:tooth", r"re:orthodont\w*", "root canal", r"re:implants?",
               r"re:oral\s+(surg\w*|care)", "braces", "aligners", "maxillofacial"],
    "eye_surgery": [r"re:cataract\w*", "lasik", r"re:retina\w*", r"re:vitreo\w*", r"re:glaucoma\w*",
                    r"re:cornea\w*", r"re:phaco\w*", r"re:eye\s+surg\w*", "squint", r"re:oculoplast\w*"],
    "physiotherapy_rehab": [r"re:physio\w*", r"re:rehabilit\w*", "occupational therapy",
                            r"re:physical\s+(therapy|medicine)"],
    "mental_health": [r"re:psychiatr\w*", r"re:psycholog\w*", r"re:mental\s+health", r"re:de[\s-]?addiction",
                      "counselling", "counseling", r"re:depression", "anxiety"],
    "outpatient_opd": [r"re:opd\w*", r"re:out[\s-]?patient\w*", r"re:consult\w*", r"re:general\s+(medicine|physician)",
                       r"re:internal\s+medicine", r"re:fever\w*", "family medicine", r"re:clinic\w*"],
    "infertility_ivf": [r"re:ivf", r"re:infertil\w*", r"re:fertility", r"re:iui", r"re:icsi",
                        r"re:test[\s-]?tube", "embryo", r"re:assisted\s+reproduct\w*"],
    "dermatology": [r"re:dermat\w*", r"re:skin\w*", r"re:acne", "psoriasis", "vitiligo", r"re:hair\s+(loss|fall)"],
    "ent": [r"re:\bent\b", r"re:e\.n\.t", r"re:otolaryng\w*", r"re:otorhino\w*", r"re:ear,?\s+nose",
            r"re:sinus\w*", r"re:tonsil\w*", r"re:cochlear\w*", r"re:audiolog\w*", r"re:hearing"],
    "ophthalmology_medical": [r"re:ophthalm\w*", r"re:eye\s+(care|check\w*|clinic|exam\w*|test\w*)",
                              r"re:optometr\w*", r"re:spectacle\w*", r"re:vision", r"re:contact\s+lens\w*"],
    "pediatrics": [r"re:p(a)?ediatric\w*", r"re:child(ren)?('s)?\s+(care|health|specialist|clinic|hospital)",
                   r"re:new[\s-]?born\s+care", r"re:neonat\w*", "child care", r"re:p(a)?ediatrician\w*"],
    "obstetrics_maternity": [r"re:obstetric\w*", r"re:maternity", r"re:antenatal", r"re:prenatal",
                             r"re:postnatal", r"re:pregnan\w*", r"re:labou?r\s+room", r"re:deliver(y|ies)",
                             r"re:gyn(a)?ecolog\w*"],
    "cosmetic_aesthetic": [r"re:cosmet\w*", r"re:aesthet\w*", r"re:esthet\w*", r"re:liposuc\w*",
                           r"re:rhinoplast\w*", "botox", r"re:hair\s+transplant\w*", r"re:fillers?",
                           r"re:plastic\s+surg\w*", r"re:laser\s+hair"],
    "prosthetics_orthotics": [r"re:prosthe\w*", r"re:orthotic\w*", r"re:artificial\s+limb\w*", r"re:calipers?",
                              r"re:braces"],
    "nephrology_renal": [r"re:nephrolog\w*", r"re:kidney\w*", r"re:renal"],
    "gastroenterology": [r"re:gastro\w*", r"re:hepatolog\w*", r"re:liver", r"re:digestive"],
    "urology": [r"re:urolog\w*", r"re:kidney\s+stones?", r"re:stones?", r"re:prostat\w*", r"re:lithotrip\w*",
                "turp", "pcnl", "urs", "rirs", r"re:cystoscop\w*", r"re:uroflow\w*", r"re:circumcis\w*",
                r"re:androlog\w*", "holep"],
    "diabetes_endocrinology": [r"re:diabet\w*", r"re:thyroid\w*", r"re:endocrin\w*", r"re:insulin\w*",
                               r"re:hormon\w*"],
    "health_checkup_vaccination": [r"re:check[\s-]?ups?", r"re:health\s+(check\w*|packages?|screening)",
                                   r"re:vaccin\w*", r"re:immuni[sz]\w*", r"re:screening", r"re:medical\s+exam\w*"],
    "cardiac_diagnostics": [r"re:echo\w*", r"re:2d\s+echo", "tmt", r"re:treadmill\w*", r"re:stress\s+test\w*",
                            r"re:holter\w*", "eecp", r"re:doppler\w*", "ambulatory blood pressure"],
    "neurology": [r"re:neurolog\w*", r"re:epilep\w*", r"re:migraine\w*", r"re:headache\w*", r"re:vertigo",
                  r"re:parkinson\w*", r"re:dementia", r"re:neuropath\w*", "eeg", "emg", "ncv",
                  r"re:nerve\s+conduction"],
    "ayush_traditional_medicine": [r"re:ayurved\w*", r"re:homo?eopath\w*", r"re:panchakarma\w*", r"re:unani",
                                   r"re:siddha", r"re:naturopath\w*", r"re:yoga\w*", r"re:shirodhara",
                                   r"re:kshar\w*", "ayush"],
    "gynecology": [r"re:gyn(a)?ecolog\w*", r"re:hysteroscop\w*", r"re:colposcop\w*", r"re:pap\s+smear\w*",
                   r"re:pco[sd]", r"re:fibroid\w*", r"re:endometrio\w*", r"re:hysterectom\w*",
                   r"re:menopaus\w*", r"re:contracept\w*", r"re:tubectom\w*", "mtp"],
    "proctology": [r"re:piles", r"re:fistula\w*", r"re:fissure\w*", r"re:ha?emorrhoid\w*", r"re:proctolog\w*",
                   r"re:pilonidal", r"re:ano[\s-]?rectal"],
    "pain_management": [r"re:pain\w*", r"re:nerve\s+blocks?"],
    "vascular_surgery": [r"re:vascular\w*", r"re:varicose\w*", r"re:av\s+fistula\w*", r"re:sclerotherap\w*",
                         r"re:endovenous\w*", r"re:vein\w*", r"re:arter\w*"],
    "cardiology_medical": [r"re:cardiolog\w*", r"re:heart\w*", r"re:cardiac\w*"],
    "other_clinical": [],
}
# Filled after reading the demoted non-theatre quotes of services whose Confirmed rows would lose
# > 30% under the term gate (neurosurgery, obstetrics_maternity, orthopedic_surgery). Most demoted
# quotes were operating-theatre lines, which are not synonyms and were not added.
SYNONYM_PASS = {
    "neurosurgery": [r"re:spin(e|al)\s+surg\w*", r"re:(vp\s+)?shunt\w*", r"re:ventriculo\w*",
                     r"re:hydrocephal\w*", r"re:deep\s+brain\s+stimulation", r"re:omm?aya\w*",
                     r"re:neuro[\s-]?navigation"],
    "obstetrics_maternity": [r"re:mother\s*(and|&)?\s*(child|baby|newborn)\w*", r"re:labou?r\s+wards?",
                             r"re:women('s)?\s+(wing|wards?)", r"re:perineal"],
    "orthopedic_surgery": [r"re:(knee|hip|joint|shoulder)\s+(replacement|reconstruction|preservation)\w*",
                           r"re:arthroplast\w*", r"re:sports\s+injur\w*", r"re:limb\s+lengthening",
                           r"re:deformity\s+correction", r"re:hand\s+surg\w*"],
}
# services kept on the old Confirmed rule (term gate off) because the term list would still
# demote > 30% of their Confirmed rows after the synonym pass
TERM_GATE_OFF = {"neurosurgery", "obstetrics_maternity"}  # 41.6% and 33.7% after the synonym pass

# Dedicated-facility terms (wave 1 fix): a sentence naming one of these counts as "detailed"
# (explicit) even without a count. Hand-written for the 8 hand-checked services; every other
# service uses its term list plus a facility noun (FACILITY_NOUN_RE in build_facility_service.py).
# A bare service name or a department name is not detailed.
DEDICATED = {
    "dialysis": [r"re:(ha?emo[\s-]?)?dialy\w*\s+(unit|units|cent(re|er)s?|machines?|stations?|wards?|rooms?|beds?|chairs?|facilit(y|ies)|wing)",
                 r"re:(machines?|units?|stations?)\s+for\s+(ha?emo)?dialy\w*", "crrt",
                 # dialysis-specific infrastructure: RO / water treatment plant, isolation (HBV/HCV) machines or rooms
                 r"re:dialy\w*\s+(water|ro\s+plant|reverse\s+osmosis)",
                 r"re:isolat\w*\s+(\w+\s+){0,8}?(ha?emo)?dialy\w*"],
    "caesarean_section": [r"re:(obstetric\w*|maternity|labou?r|lscs|ca?esa?r(e|i|ea|ia)n\w*)\s+(section\s+)?(ot|ots|operation\s+theat\w*|operating\s+theat\w*|theat\w*|suites?)"],
    "nicu": [r"re:n\.?i\.?c\.?u\.?s?", r"re:neo[\s-]?natal\s+(intensive\s+care\s+units?|icu)",
             r"re:new[\s-]?born\s+(intensive\s+care\s+units?|icu)", r"re:level[\s-]*(ii|iii|2|3)\s+(neonatal|nicu|newborn)\w*"],
    "cardiac_surgery": [r"re:(cardiac|heart|cardio[\s-]?thoracic|cardiothoracic|ctvs)\s+(surg\w*\s+)?(ot|ots|operation\s+theat\w*|operating\s+theat\w*|theat\w*|icu|units?|suites?)",
                        r"re:heart[\s-]+lung\s+machines?"],
    "mri": [r"re:mri\s+(scanners?|machines?|units?|suites?|cent(re|er)s?|facilit(y|ies))", "tesla",
            r"re:\d(\.\d)?\s?t(?![a-z])", r"re:magnetic\s+resonance\s+(scanners?|machines?|units?)"],
    "burn_care": [r"re:burns?\s+(units?|wards?|icu|cent(re|er)s?|care\s+units?|beds?)"],
    "icu_critical_care": [r"re:i\.?c\.?u\.?s?", "iccu", "ccu", "micu", "sicu", "cticu",
                          r"re:intensive\s+care\s+units?", r"re:critical\s+care\s+units?"],
    "trauma_emergency": [r"re:trauma\s+(cent(re|er)s?|units?|wards?|icu|bays?|theat\w*)",
                         r"re:emergency\s+(department|room|ward|unit)s?",
                         r"re:casualty\s+(wards?|units?|department|block)",
                         r"re:accident\s+(and|&)\s+emergency\s+(department|unit)s?"],
}

# Procedure services: a sentence that names a specific procedure (or gives a count of procedures,
# COUNT_RE in build_facility_service.py) counts as "detailed". A bare service name, a department
# name or a generic operating-theatre line does not. Procedure terms also satisfy the term gate.
# Hand-checked: caesarean_section, cardiac_surgery. The other 9 are not hand-checked.
PROCEDURE_HAND_CHECKED = {"caesarean_section", "cardiac_surgery"}
PROCEDURE = {
    "caesarean_section": ["lscs", r"re:lower\s+segment\s+ca?esa?r\w*", r"re:ca?esa?r(e|i|ea|ia)n\w*\s+(section|delivery|deliveries|operation|surger\w*)s?",
                          r"re:c[\s-]?sections?", r"re:caeserean\w*\s+(section|delivery)s?", r"re:elective\s+ca?esa?r\w*",
                          r"re:emergency\s+ca?esa?r\w*"],
    "cardiac_surgery": ["cabg", "opcab", "mics", r"re:coronary\s+(artery\s+)?bypass\w*", r"re:bypass\s+(surg\w*|graft\w*)",
                        r"re:valve\s+(replacement|repair)\w*", r"re:(mitral|aortic|tricuspid)\s+valve\s+\w+", "mvr", "avr", "dvr",
                        r"re:open[\s-]+heart\s+surg\w*", r"re:beating[\s-]+heart\s+surg\w*", r"re:bentall\w*",
                        r"re:(asd|vsd)\s+(closure|repair)", r"re:tof\s+repair", r"re:aortic\s+aneurysm\s+repair",
                        r"re:minimally\s+invasive\s+cardiac\s+surg\w*"],
    "normal_delivery": [r"re:normal\s+deliver\w*", r"re:vaginal\s+(deliver\w*|birth\w*)", "vbac", r"re:painless\s+(deliver\w*|labou?r)",
                        r"re:water\s+birth\w*", r"re:epidural\s+(analgesia|for\s+labou?r)"],
    "joint_replacement": [r"re:(total\s+)?(knee|hip|shoulder|elbow|ankle)\s+replacement\w*", "tkr", "thr", "tka", "tha",
                          r"re:arthroplast\w*", r"re:revision\s+(knee|hip)\w*"],
    "neurosurgery": [r"re:craniotom\w*", r"re:(vp\s+)?shunt\w*", "dbs", r"re:deep\s+brain\s+stimulation",
                     r"re:aneurysm\s+(clipping|coiling)", r"re:brain\s+tumou?r\s+(surg\w*|excision|removal|resection)",
                     r"re:trans[\s-]?sphenoidal\w*", r"re:skull\s+base\s+surg\w*"],
    "spine_surgery": [r"re:spinal\s+fusion", r"re:(micro)?discectom\w*", r"re:laminectom\w*", r"re:kyphoplast\w*",
                      r"re:vertebroplast\w*", "tlif", "plif", r"re:scoliosis\s+(correction|surg\w*)", r"re:endoscopic\s+spine\s+surg\w*"],
    "transplant": [r"re:(kidney|renal|liver|heart|lung|pancrea\w*|bone\s+marrow|corneal?|stem\s+cell|cornea)\s+transplant\w*",
                   "bmt", r"re:keratoplast\w*"],
    "cancer_surgery": [r"re:mastectom\w*", r"re:lumpectom\w*", r"re:whipple\w*", r"re:o?esophagectom\w*", r"re:gastrectom\w*",
                       r"re:(hemi)?colectom\w*", r"re:radical\s+\w+ectom\w*", r"re:neck\s+dissection", "hipec",
                       r"re:tumou?r\s+(resection|excision|removal)", r"re:limb\s+salvage", r"re:wertheim\w*"],
    "eye_surgery": [r"re:cataract\s+(surg\w*|operation\w*|extraction)", r"re:phaco\w*", "sics", "lasik", "smile", "prk",
                    r"re:vitrectom\w*", r"re:keratoplast\w*", r"re:trabeculectom\w*", r"re:squint\s+surg\w*",
                    r"re:retinal\s+detachment\s+surg\w*", r"re:iol\s+implant\w*", "dsek", "dmek"],
    "interventional_cardiology": [r"re:angioplast\w*", "ptca", r"re:stent(ing)?", r"re:angiograph\w*", r"re:angiogram\w*",
                                  r"re:pacemaker\s+implant\w*", "icd", "crt", "tavi", "tavr", r"re:balloon\s+valvotom\w*", "bmv",
                                  r"re:device\s+closure", r"re:ep\s+stud\w*", r"re:(rf\s+)?ablation", r"re:rotablat\w*", "ivus"],
    "gi_endoscopy": ["ercp", "eus", r"re:colonoscop\w*", r"re:gastroscop\w*", r"re:upper\s+gi\s+endoscop\w*", r"re:sigmoidoscop\w*",
                     r"re:polypectom\w*", r"re:variceal\s+(banding|ligation)", r"re:capsule\s+endoscop\w*", "peg", "poem",
                     "esd", "emr"],
}


def procedure_regex(service: str) -> str:
    import re as _re
    parts = [t[3:] if t.startswith("re:") else r"[\s-]+".join(_re.escape(w) for w in t.split())
             for t in PROCEDURE.get(service, [])]
    return (r"(?i)(?<![a-z0-9])(?:" + "|".join(parts) + r")(?![a-z0-9])") if parts else ""


def dedicated_regex(service: str) -> str:
    """Combined whole-word regex of a hand-checked service's dedicated-facility terms; '' otherwise."""
    import re as _re
    parts = [t[3:] if t.startswith("re:") else r"[\s-]+".join(_re.escape(w) for w in t.split())
             for t in DEDICATED.get(service, [])]
    return (r"(?i)(?<![a-z0-9])(?:" + "|".join(parts) + r")(?![a-z0-9])") if parts else ""


def build_terms():
    rows = []
    for sv in SERVICES:
        for src, lst in (("hand" if sv in HAND_CHECKED else "generated", TERMS[sv]),
                         ("synonym_pass", SYNONYM_PASS.get(sv, []))):
            for t in lst:
                is_re = t.startswith("re:")
                rows.append((sv, t[3:] if is_re else t, is_re, sv in HAND_CHECKED, src))
    return rows


def term_regex(service: str) -> str:
    """Combined case-insensitive whole-word Java regex for a service; '' if no terms."""
    parts = []
    for sv, t, is_re, _, _ in build_terms():
        if sv != service:
            continue
        if is_re:
            parts.append(t)
        else:
            import re as _re
            parts.append(r"[\s-]+".join(_re.escape(w) for w in t.split()))
    if not parts:
        return ""
    return r"(?i)(?<![a-z0-9])(?:" + "|".join(parts) + r")(?![a-z0-9])"


def build_fit():
    rows = []
    for ft in FACILITY_TYPES:
        s = TYPE_SPEC[ft]
        for sv in SERVICES:
            fit, note = s["default"], f"type default ({s['why']})"
            for lvl in ("expected", "plausible", "unexpected"):
                if sv in s[lvl]:
                    fit, note = lvl, f"listed {lvl} for {ft}"
            if ft not in NON_CARE and fit == "unexpected":
                if sv in ("outpatient_opd", "laboratory_pathology", "other_clinical"):
                    fit, note = "plausible", f"global floor: {sv} plausible wherever care is given"
                elif sv == "xray" and ft not in XRAY_NOT_FLOOR:
                    fit, note = "plausible", "global floor: xray plausible wherever care is given"
            if (sv, ft) in EXCEPTIONS:
                fit, why = EXCEPTIONS[(sv, ft)]
                note = f"exception: {why}"
            rows.append((sv, ft, fit, note))
    return rows


def build_map():
    rows = []
    for sv in SERVICES:
        n0 = len(rows)
        for v, st, note in SPEC[sv]:
            rows.append((sv, "specialty", v, STRENGTH[st], note or DEF_NOTE[st]))
        for v, st, note in IMPLIED[sv]:
            rows.append((sv, "implied_by_service", v, STRENGTH[st],
                         note or ("evidence for that service makes this one likely" if st == "S"
                                  else "evidence for that service makes this one possible")))
        if len(rows) == n0:
            rows.append((sv, "specialty", None, None,
                         "catch-all label: no specialty or service backs it specifically"))
    return rows


def lit(x):
    return "NULL" if x is None else "'" + str(x).replace("\\", "\\\\").replace("'", "\\'") + "'"


def lookups_main():
    assert set(SPEC) == ALL and set(IMPLIED) == ALL and set(TYPE_SPEC) == set(FACILITY_TYPES)
    for sv in SERVICES:
        for v, _, _ in IMPLIED[sv]:
            assert v in ALL and v != sv, (sv, v)
    for (sv, ft) in EXCEPTIONS:
        assert sv in ALL and ft in FACILITY_TYPES
    for s in TYPE_SPEC.values():
        for lvl in ("expected", "plausible", "unexpected"):
            assert s[lvl] <= ALL, s[lvl] - ALL

    con = connect()
    cur = con.cursor()
    cur.execute("""select s, count(distinct unique_id) from workspace.vf.facilities_clean
                   lateral view explode(specialties) t as s where not is_shifted group by s""")
    counts = dict(cur.fetchall())
    map_rows = build_map()
    bad = sorted({r[2] for r in map_rows if r[1] == "specialty" and r[2] is not None
                  and counts.get(r[2], 0) < MIN_FACILITIES})
    assert not bad, f"specialty values below {MIN_FACILITIES} facilities: {bad}"
    fit_rows = build_fit()
    assert len(fit_rows) == len(SERVICES) * len(FACILITY_TYPES) == 60 * 26

    cur.execute(f"""create or replace table workspace.vf.service_department_map (
        service string comment 'Service label (one of the 60 labels in scripts/build_lookups.py SERVICES: 49 from pa_single_v1 + 11 from pa_single_v2)',
        link_type string comment 'specialty = value is a facilities_clean.specialties term; implied_by_service = value is another service label whose evidence implies this service is possible',
        value string comment 'Specialty term (exact facilities_clean.specialties value, in >= {MIN_FACILITIES} non-shifted facilities) or a service label; NULL only for other_clinical and ayush_traditional_medicine',
        strength string comment 'strong = the service''s own department/subspecialty or a near-certain implication; supporting = related, raises likelihood only',
        note string comment 'Why this link exists or its caveat')
      comment 'Hand-written: which specialties and which other services back each of the 60 service labels (49 + 11 added with prompt pa_single_v2). Department-only specialties (internalMedicine, familyMedicine, generalMedicine, preventiveMedicine, anesthesia) are never strong. Source and rebuild: scripts/build_lookups.py'""")
    cur.execute("insert into workspace.vf.service_department_map values " +
                ",".join("(" + ",".join(lit(x) for x in r) + ")" for r in map_rows))
    cur.execute("""create or replace table workspace.vf.service_type_fit (
        service string comment 'Service label (one of the 60 labels in scripts/build_lookups.py SERVICES: 49 from pa_single_v1 + 11 from pa_single_v2)',
        facility_type string comment 'facility_type value from workspace.vf.facility_type (26 values)',
        fit string comment 'expected | plausible | unexpected: how normal it is for this facility type to offer this service',
        note string comment 'Which rule decided the fit: type default, listed for type, global floor, or exception with reason')
      comment 'Hand-written: 60 services x 26 facility types = 1,560 rows, one fit per pair. Method: type default -> per-type expected/plausible/unexpected lists -> global floors (outpatient_opd, laboratory_pathology, other_clinical, xray at least plausible in care-giving types) -> explicit exceptions. Source and rebuild: scripts/build_lookups.py'""")
    cur.execute("insert into workspace.vf.service_type_fit values " +
                ",".join("(" + ",".join(lit(x) for x in r) + ")" for r in fit_rows))
    assert set(TERMS) == ALL, ALL ^ set(TERMS)
    term_rows = build_terms()
    cur.execute("""create or replace table workspace.vf.service_terms (
        service string comment 'Service label (60 labels)',
        term string comment 'Word/phrase that names the service; regex when is_regex (Java, matched case-insensitive and whole-word on the original sentence)',
        is_regex boolean comment 'true = term is a regex; false = plain phrase (spaces match space or hyphen)',
        hand_checked boolean comment 'true for the 8 headline services reviewed by hand for synonyms and false friends',
        source string comment 'hand | generated (from label, readable name, obvious synonyms) | synonym_pass (added after reading demoted quotes)')
      comment 'Hand-written term lists per service (wave 1). scripts/build_facility_service.py requires >= 1 non-demoted explicit/stated claim sentence matching the list for Confirmed (term gate), except services in TERM_GATE_OFF (see workspace.vf.service_term_gate). Source and rebuild: scripts/build_lookups.py'""")
    cur.execute("insert into workspace.vf.service_terms values " +
                ",".join("(" + ",".join("true" if x is True else "false" if x is False else lit(x) for x in r) + ")"
                         for r in term_rows))
    for t in ("service_department_map", "service_type_fit", "service_terms"):
        cur.execute(f"select count(*), count(distinct service) from workspace.vf.{t}")
        print(t, cur.fetchone())
    cur.execute("select fit, count(*) from workspace.vf.service_type_fit group by 1 order by 1")
    print(cur.fetchall())
    cur.close()
    con.close()

In [0]:
if REBUILD:
    lookups_main()
fit = build_fit()
print(len(build_map()), "department-map rows,", len(fit), "type-fit rows,", len(build_terms()), "term rows (built in memory)")
print([r for r in fit if r[0] == "dialysis" and r[1] in ("eye", "kidney_urology", "cardiac", "dental")])

520 department-map rows, 1560 type-fit rows, 388 term rows (built in memory)
[('dialysis', 'eye', 'unexpected', 'type default (eye facility (B-scan, oculoplastics, ocular oncology plausible))'), ('dialysis', 'dental', 'unexpected', 'type default (dental facility)'), ('dialysis', 'kidney_urology', 'expected', 'listed expected for kidney_urology'), ('dialysis', 'cardiac', 'plausible', 'exception: changed after blind validation: cardiac hospitals often run dialysis units')]


In [0]:
# scripts/build_facility_service.py (the verdict: SQL only, INSERT OVERWRITE workspace.vf.facility_service)
import time

OUT = f"{VF}.facility_service"

# Possible tier: hospital-type facilities only.
HOSPITAL_TYPES = ["multispecialty_hospital", "general_hospital_or_nursing_home",
                  "medical_college_teaching_hospital", "government_health_centre",
                  "maternity_women_children", "pediatric", "cardiac", "cancer",
                  "orthopedic_spine", "neuro", "gastro_liver", "kidney_urology"]

# Operating-theatre sentence (caesarean_section Possible gate b). Spark rlike (Java regex) on the
# original sentence (not sentence_key). Phrases case-insensitive; bare "OT" only as a whole word
# in capitals (case-sensitive), so it never matches "physiotherapy" or "other".
OT_PATTERN = (r"(?i:operation theatre|operating theatre|operation theater|operating theater"
              r"|surgical theatre|modular ot)|(?<![A-Za-z])OT(?![A-Za-z])")

# Demote-to-weak patterns (Spark rlike / Java regex, case-insensitive, on the ORIGINAL sentence).
# A sentence that is hedged/future, extractor commentary, about another place, or negated near the
# service term counts as weak for that service whatever its tag says, and is never quoted as
# evidence for Confirmed or Likely (it may still be quoted for Unverified).
HEDGE_RE = (r"(?i)\b(planned|proposed|upcoming|will be|to be (started|commissioned|set up)"
            r"|under construction|coming soon)\b")
COMMENT_RE = (r"(?i)\([^)]*\b(not explicit|not listed|not mentioned|explicit|explicitly|mentioned"
              r"|listed|stated|specified|implied|unclear|inferred)\b")
ELSEWHERE_RE = (r"(?i)\b(oversees|supervises|across \d+ districts|at hub hospitals"
                r"|in other districts|network of|partner hospitals)\b")
# Negation within 4 words of the service term, either side: NEG_PRE + term + NEG_PRE_END and
# NEG_POST + term + NEG_POST_END. "No." (number) is not a negation; a bare "not available"
# elsewhere in the sentence does not demote.
NEG_PRE = r"(?i)\b(?:not|unavailable|no)\b(?!\.)(?:\W+\w+){0,4}?\W+(?:"
NEG_PRE_END = r")\b"
NEG_POST = r"(?i)\b(?:"
NEG_POST_END = r")(?:\W+\w+){0,4}?\W+(?:not|unavailable|no)\b(?!\.)"
SERVICE_TERMS = {   # service term regex; services not listed use the label with spaces
    "dialysis": r"dialysis|haemodialysis|hemodialysis|capd",
    "caesarean_section": r"caesarean|cesarean|c-section|lscs",
    "nicu": r"nicu|neonatal|newborn",
    "picu": r"picu|paediatric icu|pediatric icu|paediatric intensive|pediatric intensive",
    "icu_critical_care": r"icu|iccu|intensive care|critical care",
    "cardiac_surgery": r"cardiac surgery|heart surgery|cabg|bypass|cardiothoracic",
    "interventional_cardiology": r"angioplasty|angiography|cath lab|cathlab|ptca|stent",
    "mri": r"mri|magnetic resonance",
    "ct_scan": r"ct|cat scan|computed tomography",
    "xray": r"x-ray|xray|radiograph",
    "ultrasound": r"ultrasound|usg|sonography",
    "mammography": r"mammogra\w*",
    "nuclear_imaging": r"pet|spect|nuclear|gamma camera",
    "burn_care": r"burns?",
    "trauma_emergency": r"trauma|emergency|casualty|accident",
    "ambulance": r"ambulances?",
    "blood_bank": r"blood bank|blood storage",
    "ventilator_support": r"ventilators?|ventilation",
    "transplant": r"transplant\w*",
    # pa_single_v2 labels
    "urology": r"urolog\w*|kidney stones?|stones?|prostate|lithotripsy|turp|pcnl",
    "diabetes_endocrinology": r"diabet\w*|thyroid|endocrin\w*",
    "health_checkup_vaccination": r"check-?ups?|vaccin\w*|immuni[sz]ation",
    "cardiac_diagnostics": r"echo\w*|tmt|treadmill|stress test|holter|eecp|doppler",
    "neurology": r"neurolog\w*|eeg|emg|epilepsy|migraine|headache",
    "ayush_traditional_medicine": r"ayurved\w*|homo?eopath\w*|panchakarma|unani|siddha|naturopathy|yoga",
    "gynecology": r"gyn\w*colog\w*|hysteroscop\w*|colposcop\w*|hysterectomy",
    "proctology": r"piles|fistula|fissure|ha?emorrhoids?|proctolog\w*",
    "pain_management": r"pain",
    "vascular_surgery": r"vascular|varicose|veins?|fistula",
    "cardiology_medical": r"cardiolog\w*|heart|cardiac",
}

# Wave 1 (proposal 2): "explicit" counts only when a number sits next to a unit word, after
# stripping time and marketing numbers. Spark rlike (Java regex), case-insensitive.
STRIP_RE = (r"(?i)24\s*[/x×*]\s*7|24\s*(hours?|hrs?)|round[\s-]the[\s-]clock"
            r"|\b\d+\s*\+?\s*(years?|yrs?)\b|\b\d+\s*\+\s*[a-z]+"
            r"|\b\d+\s*(services|specialt(y|ies)|specialit(y|ies)|doctors|specialists|departments"
            r"|consultants|surgeons|staff|nurses|patients|cities|centres|centers|hospitals|branches)\b"
            r"|\b(19|20)\d\d\b")
UNIT_RE = (r"(?i)\b\d+(\.\d+)?\s*(-\s*)?([a-z]+[\s-]+){0,2}?"
           r"(beds?|bedded|machines?|units?|stations?|theat(re|er)s?|ots|scanners?|tesla|t|slices?"
           r"|sessions?|treatments|ventilators?|cots?|incubators?|warmers?|chairs?)\b")
# Department-name or generic line (wave 1, proposal 1): a sentence with no term match that names a
# department / speciality or an operating theatre. Evidence made only of these -> Possible.
DEPT_RE = (r"(?i)\b(department|departments|dept|division|specialit(y|ies)|specialt(y|ies)"
           r"|(?!techno)[a-z]+olog(y|ies|ist|ists)|[a-z]*iatrics?|orthop(a)?edics|obstetrics|medicine)\b")
# Generic dedicated-facility test (services without a DEDICATED list): term match + facility noun.
FACILITY_NOUN_RE = (r"(?i)\b(units?|cent(re|er)s?|wards?|labs?|laborator(y|ies)|machines?|scanners?"
                    r"|theat(re|er)s?)\b")
# Procedure services: a count of procedures counts as detailed ("500 deliveries a year"); a 4-digit
# year is not a count.
COUNT_RE = (r"(?i)(?<![\d,.])(?!(19|20)\d\d\b)\d[\d,]*\+?\s+([a-z-]+\s+){0,3}?"
            r"(surger(y|ies)|procedures?|operations?|deliver(y|ies)|transplants?|replacements?"
            r"|angioplast(y|ies)|bypass\w*|cases|sections|c-sections|lscs|stents?|implants?)\b")
DEPT_WARNING = "Only a department or an operating theatre is listed for this service"


def term_values() -> str:
    """VALUES rows (service, term regex, gate) from scripts/build_lookups.py TERMS."""
    # build_lookups is the cell above
    rows = []
    for sv in SERVICES:
        rx = term_regex(sv)
        gate = bool(rx) and sv not in TERM_GATE_OFF
        drx = dedicated_regex(sv)
        prx = procedure_regex(sv)
        nul = "CAST(NULL AS STRING)"
        rows.append(f"({lit(sv)}, {lit(rx) if rx else nul}, {'true' if gate else 'false'}, "
                    f"{lit(drx) if drx else nul}, {lit(prx) if prx else nul})")
    return ", ".join(rows)


# Name-token rule, identical to scripts/contamination.py signal a (A_MIN_LEN, A_GENERIC,
# and tokens equal to any city name are dropped).
A_MIN_LEN = 4
A_GENERIC = """hospital hospitals clinic clinics eye dental dentist care centre center multispeciality
multispecialty multi speciality specialty super health healthcare medical nursing home institute
research diagnostic diagnostics the and of for dr doctor doctors children women maternity memorial
general city life new sri shri shree lab labs path pathology polyclinic trust foundation private
limited ltd pvt india indian unit road nagar""".split()


def lit(s: str) -> str:
    return "'" + s.replace("\\", "\\\\").replace("'", "\\'") + "'"


def arr(xs) -> str:
    return "array(" + ",".join(lit(x) for x in xs) + ")"


def conn():
    return connect()


def run(c, q, fetch=False):
    with c.cursor() as cur:
        cur.execute(q)
        if fetch:
            cols = [d[0] for d in cur.description]
            return [dict(zip(cols, r)) for r in cur.fetchall()]
    return None


def svc_label(x: str) -> str:
    """SQL expression: human label for a service code, capitalised."""
    return (f"CASE {x} WHEN 'mri' THEN 'MRI' WHEN 'nicu' THEN 'NICU' WHEN 'picu' THEN 'PICU' "
            f"WHEN 'icu_critical_care' THEN 'ICU / critical care' WHEN 'ct_scan' THEN 'CT scan' "
            f"ELSE concat(upper(substr(replace({x}, '_', ' '), 1, 1)), "
            f"substr(replace({x}, '_', ' '), 2)) END")


def svc_label_lc(x: str) -> str:
    """SQL expression: human label for a service code, mid-sentence."""
    return (f"CASE {x} WHEN 'mri' THEN 'MRI' WHEN 'nicu' THEN 'NICU' WHEN 'picu' THEN 'PICU' "
            f"WHEN 'icu_critical_care' THEN 'ICU / critical care' WHEN 'ct_scan' THEN 'CT scan' "
            f"ELSE replace({x}, '_', ' ') END")


def _readable_names() -> dict:
    """Readable service names from the app's config (single source), mid-sentence case."""
    names = dict(SERVICE_NAME)   # app/config.py, shared-names cell (all 60 labels)
    return {k: (v if v[:2].isupper() else v[:1].lower() + v[1:]) for k, v in names.items()}


def readable_lc(x: str) -> str:
    """SQL expression: readable service name for label x (falls back to label with spaces)."""
    whens = " ".join(f"WHEN {lit(k)} THEN {lit(v)}" for k, v in _readable_names().items())
    return f"CASE {x} {whens} ELSE replace({x}, '_', ' ') END"


def type_readable(x: str) -> str:
    """SQL expression: readable facility type with article, from app/config.py FACILITY_TYPE_NAME."""
    names = FACILITY_TYPE_NAME   # app/config.py, shared-names cell
    whens = " ".join(f"WHEN {lit(k)} THEN {lit(v)}" for k, v in names.items())
    return (f"CASE {x} {whens} ELSE concat('a ', replace({x}, '_', ' '), ' facility') END")


def build_sql(has_audit: bool, has_flag: bool) -> str:
    foreign_anti = (f"LEFT ANTI JOIN (SELECT DISTINCT unique_id, source_column, position "
                    f"FROM {VF}.sentence_foreign_flag WHERE is_foreign) ff "
                    f"USING (unique_id, source_column, position)") if has_flag else ""
    audit_cte = (f"""audit AS (
  SELECT unique_id, verdict,
         CASE WHEN n_usable > 0 THEN n_flagged / n_usable END flagged_share
  FROM {VF}.record_audit WHERE NOT coalesce(failed, false)
  QUALIFY row_number() OVER (PARTITION BY unique_id ORDER BY audited_at DESC) = 1)"""
                 if has_audit else
                 "audit AS (SELECT CAST(NULL AS STRING) unique_id, CAST(NULL AS STRING) verdict, "
                 "CAST(NULL AS DOUBLE) flagged_share "
                 "WHERE false)")
    n_audited = (f"(SELECT count(DISTINCT unique_id) FROM {VF}.record_audit "
                 f"WHERE NOT coalesce(failed, false))") if has_audit else "CAST(NULL AS BIGINT)"
    terms_values = ", ".join(f"({lit(k)}, {lit(v)})" for k, v in SERVICE_TERMS.items())
    return f"""
INSERT OVERWRITE {OUT}
WITH
cities AS (SELECT collect_set(lower(trim(city))) cities FROM {VF}.facilities_clean
           WHERE city IS NOT NULL),
fac AS (
  SELECT c.unique_id, lower(trim(c.city)) city_l, c.specialties,
         coalesce(ft.facility_type, 'unknown') facility_type,
         array_distinct(filter(split(lower(coalesce(c.name, '')), '[^a-z0-9]+'),
           x -> length(x) >= {A_MIN_LEN} AND NOT array_contains({arr(A_GENERIC)}, x)
                AND NOT array_contains(cities.cities, x))) toks
  FROM {VF}.facilities_clean c CROSS JOIN cities
  LEFT JOIN {VF}.facility_type ft USING (unique_id)
  WHERE NOT coalesce(c.is_shifted, false)),
tags AS (
  SELECT d.sentence_key,
         CASE WHEN t.prompt_version = 'rule_dental_only' THEN true ELSE t.is_claim END is_claim,
         CASE WHEN t.prompt_version = 'rule_dental_only' THEN 'stated' ELSE t.strength END strength,
         CASE WHEN t.prompt_version = 'rule_dental_only' THEN filter(t.services, x -> x = 'dental')
              ELSE t.services END services
  FROM {VF}.sentence_tags t JOIN {VF}.distinct_sentences d USING (sentence_id)
  WHERE NOT coalesce(t.failed, false)
  QUALIFY row_number() OVER (PARTITION BY t.sentence_id ORDER BY t.tagged_at DESC) = 1),
sent AS (
  SELECT s.unique_id, s.source_column, s.position, s.sentence, s.sentence_key
  FROM {VF}.facility_sentences s
  JOIN fac USING (unique_id)
  {foreign_anti}
  WHERE NOT s.is_placeholder AND NOT s.is_repeat),
terms AS (SELECT * FROM VALUES {terms_values} AS v(service, term)),
svc_terms AS (SELECT * FROM VALUES {term_values()} AS v(service, rx, gate, drx, prx)),
claim0 AS (
  SELECT s.unique_id, svc AS service, s.sentence_key, s.sentence, s.source_column, s.position,
         t.strength tag_strength,
         CASE WHEN exists(f.toks, x -> instr(lower(s.sentence), x) > 0)
                OR (f.city_l IS NOT NULL AND instr(lower(s.sentence), f.city_l) > 0)
              THEN 0 ELSE 1 END nrank
  FROM sent s JOIN tags t USING (sentence_key) JOIN fac f USING (unique_id)
  LATERAL VIEW explode(t.services) e AS svc
  WHERE t.is_claim AND t.strength IN ('explicit', 'stated', 'weak')
  QUALIFY row_number() OVER (PARTITION BY s.unique_id, svc, s.sentence_key
                             ORDER BY s.source_column, s.position) = 1),
claim AS (
  SELECT c.*,
    CASE WHEN demoted OR tag_strength = 'weak' THEN 2
         -- explicit needs a number next to a unit word, after stripping time/marketing numbers
         -- explicit = a number next to a unit word (after stripping time/marketing numbers), or a
         -- dedicated unit/machine/facility for the service (DEDICATED list, else term + facility noun)
         WHEN tag_strength = 'explicit'
              AND (regexp_replace(sentence, {lit(STRIP_RE)}, ' ') RLIKE {lit(UNIT_RE)} OR dedicated) THEN 0
         -- procedure services: a named specific procedure or a count of procedures (explicit or stated tag)
         WHEN procedure_hit THEN 0
         ELSE 1 END srank,
    -- term-list match first in the evidence order
    CASE WHEN tmatch THEN 0 ELSE 1 END tmiss,
    -- evidence order after strength: sentences with a digit first, then longer first
    CASE WHEN sentence RLIKE '[0-9]' THEN 0 ELSE 1 END nodigit,
    -length(sentence) neg_len,
    -- near-duplicate key: lowercase, punctuation (incl. thousands separators) removed
    trim(regexp_replace(regexp_replace(lower(sentence), '[^a-z0-9 ]', ''), ' +', ' ')) norm
  FROM (
    SELECT c0.*,
      coalesce(c0.sentence RLIKE st.rx, false) OR coalesce(c0.sentence RLIKE st.prx, false) tmatch,
      coalesce(st.gate, false) gate,
      (st.prx IS NOT NULL AND c0.tag_strength IN ('explicit', 'stated')
       AND (c0.sentence RLIKE st.prx OR c0.sentence RLIKE {lit(COUNT_RE)})) procedure_hit,
      coalesce(CASE WHEN st.drx IS NOT NULL THEN c0.sentence RLIKE st.drx
                    ELSE c0.sentence RLIKE st.rx AND c0.sentence RLIKE {lit(FACILITY_NOUN_RE)} END, false) dedicated,
      (NOT coalesce(c0.sentence RLIKE st.rx, false) AND NOT coalesce(c0.sentence RLIKE st.prx, false)
       AND (c0.sentence RLIKE {lit(OT_PATTERN)} OR c0.sentence RLIKE {lit(DEPT_RE)})) dept_ot,
      (c0.sentence RLIKE {lit(HEDGE_RE)} OR c0.sentence RLIKE {lit(COMMENT_RE)}
       OR c0.sentence RLIKE {lit(ELSEWHERE_RE)}
       OR c0.sentence RLIKE concat({lit(NEG_PRE)}, coalesce(tm.term, replace(c0.service, '_', ' ')),
                                   {lit(NEG_PRE_END)})
       OR c0.sentence RLIKE concat({lit(NEG_POST)}, coalesce(tm.term, replace(c0.service, '_', ' ')),
                                   {lit(NEG_POST_END)})) demoted
    FROM claim0 c0 LEFT JOIN terms tm ON tm.service = c0.service
    LEFT JOIN svc_terms st ON st.service = c0.service) c),
direct AS (
  SELECT unique_id, service,
    count(DISTINCT CASE WHEN srank = 0 THEN sentence_key END) n_explicit,
    count(DISTINCT CASE WHEN srank = 1 THEN sentence_key END) n_stated,
    count(DISTINCT CASE WHEN srank = 2 THEN sentence_key END) n_weak,
    count(DISTINCT CASE WHEN srank <= 1 AND tmatch THEN sentence_key END) n_term,
    count(DISTINCT CASE WHEN srank <= 1 AND NOT dept_ot THEN sentence_key END) n_nondept,
    max(gate) gate,
    array_sort(collect_list(struct(tmiss, srank, nodigit, neg_len, source_column, position, sentence,
            demoted, norm))) ev_all
  FROM claim GROUP BY unique_id, service),
backing AS (
  SELECT f.unique_id, m.service, array_sort(collect_set(m.value)) specs
  FROM (SELECT unique_id, explode(specialties) sp FROM fac) f
  JOIN {VF}.service_department_map m
    ON m.link_type = 'specialty' AND m.strength = 'strong' AND m.value = sp
  GROUP BY f.unique_id, m.service),
possible AS (
  SELECT b.unique_id, m.service,
    array_sort(collect_set(c.service)) related,
    array_distinct(array_sort(collect_list(struct(c.tmiss, c.srank, c.nodigit, c.neg_len, c.source_column,
            c.position, c.sentence, c.demoted, c.norm)))) ev_all
  FROM backing b
  JOIN fac f ON f.unique_id = b.unique_id
              AND array_contains({arr(HOSPITAL_TYPES)}, f.facility_type)
  JOIN {VF}.service_department_map m
    ON m.link_type = 'implied_by_service' AND m.strength = 'strong' AND m.service = b.service
  JOIN claim c ON c.unique_id = b.unique_id AND c.service = m.value AND c.srank <= 1
  LEFT ANTI JOIN direct d ON d.unique_id = b.unique_id AND d.service = m.service
  WHERE m.service <> 'caesarean_section'
     OR (b.unique_id IN (SELECT unique_id FROM claim
                         WHERE service IN ('normal_delivery', 'high_risk_obstetrics'))
         AND (b.unique_id IN (SELECT unique_id FROM claim WHERE service = 'general_surgery')
              OR b.unique_id IN (SELECT unique_id FROM sent WHERE sentence RLIKE {lit(OT_PATTERN)})))
  GROUP BY b.unique_id, m.service),
base AS (
  SELECT unique_id, service, n_explicit, n_stated, n_weak, ev_all,
         CAST(NULL AS ARRAY<STRING>) related, true has_claim, n_term, n_nondept, gate FROM direct
  UNION ALL
  SELECT unique_id, service, 0, 0, 0, ev_all, related, false, 0, 0, false FROM possible),
{audit_cte},
j AS (
  SELECT b.*, f.facility_type, coalesce(bk.specs, array()) specs,
         size(coalesce(bk.specs, array())) > 0 backed,
         tf.fit type_fit, a.verdict audit_verdict, a.flagged_share,
         -- mixed record: mostly_foreign, or partly_foreign with >= 50% of usable sentences flagged
         (a.verdict = 'mostly_foreign'
          OR (a.verdict = 'partly_foreign' AND a.flagged_share >= 0.5)) mixed_unverified,
         -- warn when mostly_foreign or >= 25% flagged; unaudited facilities get no warning
         coalesce(a.verdict = 'mostly_foreign' OR a.flagged_share >= 0.25, false) mixed_warn
  FROM base b JOIN fac f USING (unique_id)
  LEFT JOIN backing bk ON bk.unique_id = b.unique_id AND bk.service = b.service
  LEFT JOIN {VF}.service_type_fit tf
    ON tf.service = b.service AND tf.facility_type = f.facility_type
  LEFT JOIN audit a ON a.unique_id = b.unique_id),
t AS (
  SELECT j.*,
    CASE WHEN coalesce(mixed_unverified, false) OR type_fit = 'unexpected'
              OR (has_claim AND n_explicit + n_stated = 0) THEN 'Unverified'
         -- term gate: evidence made only of department names / operating-theatre lines
         WHEN gate AND has_claim AND n_nondept = 0 THEN 'Possible'
         WHEN n_explicit >= 1 AND (NOT gate OR n_term >= 1) AND (backed OR n_explicit + n_stated >= 2
              OR type_fit = 'expected') THEN 'Confirmed'
         WHEN n_explicit + n_stated >= 1 THEN 'Likely'
         ELSE 'Possible' END tier,
    replace(facility_type, '_', ' ') type_l
  FROM j),
t2 AS (
  SELECT t.*, (gate AND has_claim AND tier = 'Possible') dept_only,
         CASE WHEN tier IN ('Confirmed', 'Likely') OR (has_claim AND tier = 'Possible')
              THEN filter(ev_all, x -> NOT x.demoted)
                   ELSE ev_all END ev_f
  FROM t),
stamp AS (
  SELECT (SELECT count(*) FROM {VF}.sentence_tags WHERE NOT coalesce(failed, false)) n_tagged,
         (SELECT count(DISTINCT sentence_id) FROM {VF}.sentence_tags) n_attempted,
         {n_audited} n_audited)
SELECT unique_id, service, tier,
  CAST(n_explicit AS INT), CAST(n_stated AS INT), CAST(n_weak AS INT),
  CAST(backed AS STRING) department_backing, type_fit, audit_verdict,
  -- demoted (hedged / commentary / elsewhere / negated) sentences are never quoted for Confirmed
  -- or Likely; near-duplicates (same norm) are quoted once, keeping the first in evidence order
  slice(transform(aggregate(ev_f, filter(ev_f, x -> false),
                            (acc, x) -> CASE WHEN exists(acc, y -> y.norm = x.norm) THEN acc
                                             ELSE concat(acc, array(x)) END),
                  x -> named_struct('sentence', x.sentence, 'source_column', x.source_column)),
        1, 3) evidence,
  array_compact(array(
    CASE WHEN mixed_warn
         THEN 'Parts of this record may describe a different hospital. Call to confirm.' END,
    CASE WHEN type_fit = 'unexpected'
         THEN concat(upper(substr({readable_lc('service')}, 1, 1)), substr({readable_lc('service')}, 2),
                     ' is unexpected for ', {type_readable('facility_type')}) END,
    CASE WHEN has_claim AND n_explicit + n_stated = 0 THEN 'Only weak evidence' END,
    CASE WHEN dept_only THEN {lit(DEPT_WARNING)} END,
    CASE WHEN NOT has_claim
         THEN concat('No direct claim; inferred from the ',
                     array_join(transform(specs, s -> lower(regexp_replace(s, '([A-Z])', ' $1'))), ', '),
                     CASE WHEN size(specs) > 1 THEN ' departments and ' ELSE ' department and ' END,
                     CASE WHEN size(related) > 1 THEN '' ELSE 'a ' END,
                     array_join(transform(related, r -> {readable_lc('r')}), ', '),
                     CASE WHEN size(related) > 1 THEN ' claims' ELSE ' claim' END) END
  )) warnings,
  current_timestamp() build_ts,
  stamp.n_tagged n_sentences_tagged_at_build,
  stamp.n_audited n_records_audited_at_build,
  stamp.n_attempted n_sentences_attempted_at_build,
  facility_type, flagged_share, CAST(n_term AS INT) n_term_sentences
FROM t2 CROSS JOIN stamp
"""


TABLE_COMMENT = (
    "Verdict per (unique_id, service), built by scripts/build_facility_service.py with SQL only "
    "(no LLM calls); rerun as tagging and the audit progress (INSERT OVERWRITE). Rows exist only "
    "for pairs with >= 1 claim sentence, or that qualify for Possible. Claims = sentence_tags rows "
    "with is_claim true (failed rows ignored) on usable sentences (not placeholder, not repeat, "
    "facility not shifted), minus sentences flagged foreign in sentence_foreign_flag (joined on "
    "unique_id, source_column, position) when that table has rows. Rows tagged by prompt_version "
    "'rule_dental_only' (is_claim NULL, services [dental], strength NULL) are treated as is_claim "
    "true with strength 'stated', for the dental service only. An explicit-tagged sentence counts as explicit "
    "only when a number sits next to a unit word (UNIT_RE: beds, machines, units, stations, theatres, scanners, Tesla, "
    "slice, sessions, ventilators, cots, incubators...) after stripping time and marketing numbers (STRIP_RE: 24/7, "
    "N years, N+ services, years); otherwise it counts as stated. And a sentence that is hedged/future, extractor commentary, "
    "about another place, or negated within 4 words of the service term counts as weak (patterns in the "
    "script) and is never quoted for Confirmed or Likely; both applied before the tier rules, "
    "sentence_tags unchanged. "
    "Tier, first match wins: "
    "Unverified if audit mostly_foreign (or partly_foreign with flagged_share = n_flagged/n_usable >= 0.5), type_fit unexpected, or only weak sentences; "
    "term gate (services with applied = true in workspace.vf.service_term_gate; term lists in workspace.vf.service_terms): "
    "Possible with warning 'Only a department or an operating theatre is listed for this service' if every explicit/stated "
    "sentence lacks a term match and is a department-name (DEPT_RE) or operating-theatre (OT_PATTERN) line; Confirmed if "
    ">= 1 explicit, >= 1 explicit/stated sentence matching the term list (gated services only) and (department_backing, or >= 2 explicit+stated, or type_fit expected); Likely "
    "if >= 1 explicit or stated; Possible if no claim sentence for the service but department_backing "
    "and an explicit/stated claim for a related service that implies it (service_department_map "
    "implied_by_service, STRONG links only), hospital facility types only; evidence quotes only the "
    "triggering related-service sentences. caesarean_section Possible additionally needs a claim for "
    "normal_delivery or high_risk_obstetrics AND (a general_surgery claim OR a usable non-foreign "
    "sentence matching the operating-theatre pattern OT_PATTERN in the script). Coverage at build time is in "
    "build_ts, n_sentences_tagged_at_build, n_records_audited_at_build: tiers change as coverage "
    "grows. Read by the vf-locator Databricks App.")

COL_COMMENTS = {
    "unique_id": "Facility key; joins to workspace.vf.facilities_clean.unique_id",
    "service": "Service label (one of the 60 labels in scripts/build_lookups.py SERVICES: 49 from pa_single_v1 + 11 from pa_single_v2; readable names in app/config.py and scripts/service_names_extra.py), e.g. dialysis, caesarean_section, nicu",
    "tier": "Confirmed | Likely | Possible | Unverified (rules in the table comment). Possible = no direct claim, inferred from department + related-service claim",
    "n_explicit": "Distinct sentences (distinct sentence_key per facility) tagged explicit for this service in which a number sits next to a unit word after stripping time/marketing numbers (UNIT_RE, STRIP_RE in the script); 0 for inferred Possible",
    "n_term_sentences": "Distinct non-demoted explicit/stated sentences for this service that match its term list (workspace.vf.service_terms, whole word/phrase); Confirmed needs >= 1 for gated services; 0 for inferred Possible",
    "n_stated": "Distinct sentences tagged stated for this service, plus explicit-tagged sentences without a number next to a unit word (rule_dental_only rows count here for dental); 0 for Possible",
    "n_weak": "Distinct sentences tagged weak for this service; 0 for Possible",
    "department_backing": "'true' if facilities_clean.specialties contains a specialty linked to the service with strength STRONG in service_department_map (supporting links are not used); 'false' otherwise, including services with no strong specialty link",
    "type_fit": "service_type_fit.fit for the facility's facility_type (expected / plausible / unexpected); NULL if no fit row",
    "audit_verdict": "record_audit.verdict (latest non-failed); NULL if not audited, audit failed, or record_audit absent at build",
    "evidence": "Up to 3 verbatim sentences with source_column: term-list matches first, then explicit before stated before weak, then sentences with a digit, then longer first; near-duplicates (same text after lowercasing and removing punctuation/thousands separators) quoted once. For Possible: only explicit/stated sentences of the related service(s) whose strong implied_by_service link triggered the row",
    "facility_type": "Added by build_facility_service.py: workspace.vf.facility_type label used for type_fit ('unknown' if none)",
    "flagged_share": "Added by build_facility_service.py: record_audit n_flagged / n_usable (share of the record's usable sentences flagged as describing another organisation); NULL if not audited",
    "warnings": "Plain-text reasons a row is Unverified or Possible. Mixed-record warning 'Parts of this record may describe a different hospital. Call to confirm.' only when audit is mostly_foreign or record_audit n_flagged/n_usable >= 0.25 (none for unaudited). Location caveats are added by the app",
    "build_ts": "Added by build_facility_service.py: when this build ran",
    "n_sentences_tagged_at_build": "Added by build_facility_service.py: non-failed rows in sentence_tags (LLM + rule_dental_only) at build time; compare to 271,262 distinct sentences",
    "n_sentences_attempted_at_build": "Added by build_facility_service.py: distinct sentence_id in sentence_tags (tagged or failed) at build time; tagging is complete when this reaches 271,262",
    "n_records_audited_at_build": "Added by build_facility_service.py: distinct unique_id with a non-failed record_audit row at build time; NULL if record_audit absent",
}


STATS = f"{VF}.build_stats"


def build_stats(c, has_audit):
    """Funnel numbers for the app's "How this works" panel, as of this build."""
    run(c, f"""CREATE TABLE IF NOT EXISTS {STATS} (metric STRING, value BIGINT, build_ts TIMESTAMP)
        COMMENT 'Funnel numbers for the vf-locator How this works panel, rewritten by scripts/build_facility_service.py on every rebuild of facility_service. build_ts = facility_service.build_ts.'""")
    audited = (f"(SELECT count(*) FROM {VF}.record_audit WHERE NOT coalesce(failed, false))"
               if has_audit else "CAST(NULL AS BIGINT)")
    mixed = (f"""(SELECT count(*) FROM {VF}.record_audit WHERE NOT coalesce(failed, false)
                 AND (verdict = 'mostly_foreign' OR (n_usable > 0 AND n_flagged / n_usable >= 0.25)))"""
             if has_audit else "CAST(NULL AS BIGINT)")
    metrics = [
        ("facilities", f"SELECT count(*) FROM {VF}.facilities_clean WHERE NOT coalesce(is_shifted, false)"),
        ("sentences_read", f"""SELECT count(*) FROM {VF}.facility_sentences s JOIN {VF}.facilities_clean f
            USING (unique_id) WHERE NOT coalesce(f.is_shifted, false)
            AND NOT s.is_placeholder AND NOT s.is_repeat"""),
        ("distinct_sentences", f"SELECT count(*) FROM {VF}.distinct_sentences"),
        ("distinct_sentences_tagged", f"""SELECT count(DISTINCT sentence_id) FROM {VF}.sentence_tags
            WHERE NOT coalesce(failed, false)"""),
        ("service_claim_sentences", f"""SELECT count(DISTINCT sentence_id) FROM {VF}.sentence_tags
            WHERE NOT coalesce(failed, false) AND (is_claim OR prompt_version = 'rule_dental_only')"""),
        ("facilities_audited", f"SELECT {audited}"),
        ("records_mixed", f"SELECT {mixed}"),
        ("rows_confirmed", f"SELECT count_if(tier = 'Confirmed') FROM {OUT}"),
        ("rows_likely", f"SELECT count_if(tier = 'Likely') FROM {OUT}"),
        ("rows_possible", f"SELECT count_if(tier = 'Possible') FROM {OUT}"),
        ("rows_unverified", f"SELECT count_if(tier = 'Unverified') FROM {OUT}"),
    ]
    union = "\nUNION ALL ".join(f"SELECT {lit(m)} AS metric, CAST(({q}) AS BIGINT) AS value"
                                 for m, q in metrics)
    run(c, f"""INSERT OVERWRITE {STATS}
SELECT metric, value, (SELECT max(build_ts) FROM {OUT}) FROM ({union})""")
    print("build_stats:", run(c, f"SELECT metric, value FROM {STATS} ORDER BY metric", True))


def facility_service_main():
    c = conn()
    present = {r["table_name"] for r in run(c, f"""SELECT table_name FROM workspace.information_schema.tables
        WHERE table_schema = 'vf' AND table_name IN ('record_audit', 'sentence_foreign_flag',
        'service_department_map', 'service_type_fit', 'facility_type', 'sentence_tags')""", True)}
    for req in ("service_department_map", "service_type_fit", "facility_type", "sentence_tags"):
        if req not in present:
            raise SystemExit(f"required table workspace.vf.{req} missing")
    has_audit = "record_audit" in present
    has_flag = "sentence_foreign_flag" in present
    print(f"record_audit present={has_audit} sentence_foreign_flag present={has_flag}")

    cols = {r["col_name"] for r in run(c, f"DESCRIBE TABLE {OUT}", True)}
    add = [(n, t) for n, t in [("build_ts", "TIMESTAMP"), ("n_sentences_tagged_at_build", "BIGINT"),
                               ("n_records_audited_at_build", "BIGINT"),
                               ("n_sentences_attempted_at_build", "BIGINT"),
                               ("facility_type", "STRING"), ("flagged_share", "DOUBLE"),
                               ("n_term_sentences", "INT")]
           if n not in cols]
    if add:
        run(c, f"ALTER TABLE {OUT} ADD COLUMNS (" + ", ".join(f"{n} {t}" for n, t in add) + ")")
        print("added columns:", [n for n, _ in add])

    t0 = time.time()
    q = build_sql(has_audit, has_flag)
    run(c, q)
    print(f"build secs={time.time() - t0:.1f}")
    # how many (facility, sentence, service) claim pairs the demote-to-weak patterns hit
    ctes = q[q.index("WITH"):q.index(",\ndirect AS (")]
    print("demoted pairs:", run(c, ctes + """
SELECT count(*) n_pairs, count_if(demoted AND tag_strength <> 'weak') n_demoted_from_explicit_or_stated,
       count_if(demoted) n_demoted, count_if(demoted AND tag_strength <> 'weak' AND service = 'dialysis') n_dialysis
FROM claim""", True))

    build_stats(c, has_audit)
    run(c, f"COMMENT ON TABLE {OUT} IS {lit(TABLE_COMMENT)}")
    for col, cm in COL_COMMENTS.items():
        run(c, f"ALTER TABLE {OUT} ALTER COLUMN {col} COMMENT {lit(cm)}")

    for r in run(c, f"""SELECT tier, count(*) n FROM {OUT} GROUP BY tier ORDER BY tier""", True):
        print(r)
    print(run(c, f"""SELECT max(n_sentences_tagged_at_build) tagged, max(n_records_audited_at_build) audited,
        max(build_ts) ts, count(*) n_rows FROM {OUT}""", True))

In [0]:
if REBUILD:
    facility_service_main()
q_tier = build_sql(True, True)
print(q_tier[q_tier.index("\nt AS ("):q_tier.index("\nt2 AS (")])   # the tier rule exactly as the build runs it


t AS (
  SELECT j.*,
    CASE WHEN coalesce(mixed_unverified, false) OR type_fit = 'unexpected'
              OR (has_claim AND n_explicit + n_stated = 0) THEN 'Unverified'
         -- term gate: evidence made only of department names / operating-theatre lines
         WHEN gate AND has_claim AND n_nondept = 0 THEN 'Possible'
         WHEN n_explicit >= 1 AND (NOT gate OR n_term >= 1) AND (backed OR n_explicit + n_stated >= 2
              OR type_fit = 'expected') THEN 'Confirmed'
         WHEN n_explicit + n_stated >= 1 THEN 'Likely'
         ELSE 'Possible' END tier,
    replace(facility_type, '_', ' ') type_l
  FROM j),


In [0]:
%sql
SELECT service,
       count_if(tier = 'Confirmed') AS confirmed, count_if(tier = 'Likely') AS likely,
       count_if(tier = 'Possible') AS possible, count_if(tier = 'Unverified') AS unverified
FROM workspace.vf.facility_service
WHERE service IN ('dialysis', 'caesarean_section', 'nicu', 'cardiac_surgery', 'mri', 'burn_care', 'icu_critical_care', 'trauma_emergency')
GROUP BY 1 ORDER BY confirmed DESC

service,confirmed,likely,possible,unverified
icu_critical_care,987,1178,130,461
mri,583,451,330,428
dialysis,406,485,294,146
nicu,388,473,135,150
caesarean_section,365,16,396,96
cardiac_surgery,309,647,606,494
trauma_emergency,123,2705,25,1232
burn_care,16,120,3,41


In [0]:
%sql
-- The demo records: Aravind Eye Hospital stays Unverified for dialysis; Sudar Hospital, Thanjavur is Confirmed
SELECT c.name, c.city, fs.service, fs.tier, fs.n_explicit, fs.n_stated, fs.n_weak, fs.department_backing,
       fs.type_fit, fs.audit_verdict, fs.evidence, fs.warnings
FROM workspace.vf.facility_service fs JOIN workspace.vf.facilities_clean c USING (unique_id)
WHERE fs.service = 'dialysis' AND (fs.unique_id = 'b8a5401f-42f1-422a-8cd9-686a15b4cb76' OR (c.name ILIKE 'Sudar%' AND c.city ILIKE 'Thanjavur%'))

name,city,service,tier,n_explicit,n_stated,n_weak,department_backing,type_fit,audit_verdict,evidence,warnings
Aravind Eye Hospital,Hyderabad,dialysis,Unverified,2,3,0,false,unexpected,mostly_foreign,"List(List(Dialysis treatment is provided via a 20-bed Dialysis Unit with 10 machines as part of the Pradhan Mantri National Dialysis Programme, procedure), List(Dialysis unit of 20 beds under Pradhan Mantri National Dialysis Programme, capability), List(A dedicated Dialysis Operating Theatre is commissioned in the OT Complex, procedure))","List(Parts of this record may describe a different hospital. Call to confirm., Dialysis is unexpected for an eye hospital)"
Sudar Hospital and Research Institute,Thanjavur,dialysis,Confirmed,1,1,0,true,expected,consistent,"List(List(Dialysis unit, capability), List(Dialysis procedures are provided, procedure))",List()


In [0]:
%sql
SELECT metric, value, build_ts FROM workspace.vf.build_stats ORDER BY metric

metric,value,build_ts
distinct_sentences,271262,2026-11-22T04:29:37.497Z
distinct_sentences_tagged,269731,2026-11-22T04:29:37.497Z
facilities,9989,2026-11-22T04:29:37.497Z
facilities_audited,9575,2026-11-22T04:29:37.497Z
records_mixed,814,2026-11-22T04:29:37.497Z
rows_confirmed,12128,2026-11-22T04:29:37.497Z
rows_likely,55352,2026-11-22T04:29:37.497Z
rows_possible,8005,2026-11-22T04:29:37.497Z
rows_unverified,22760,2026-11-22T04:29:37.497Z
sentences_read,423706,2026-11-22T04:29:37.497Z


## Step 7 — District health context (NFHS-5)

**What.** Links each facility and each PIN code to its NFHS-5 district (2019-21 survey) through the post-office directory, with a hand-written alias table for renamed districts and districts created after the survey. Maps six headline services to 0-3 survey indicators (dialysis -> high blood sugar and blood pressure; NICU -> births in a facility, skilled birth attendance, anaemia), ranked 1..706 with 1 = worst.

**Why.** A planner wants to know whether an area with no nearby Confirmed provider also has high need. This is shown as **context only**: it never changes a tier or a rank. MRI and burn care deliberately get no indicator, because no survey figure measures their need.

**Reads** share `nfhs_5_district_health_indicators`, `india_post_pincode_directory`, `facilities_clean`. **Writes** `nfhs_district` (706), `district_alias` (110), `pin_district` (20,824), `facility_district` (9,989; 9,689 linked), `service_indicator_map` (13). Source: `scripts/build_nfhs.py`.

In [0]:
# scripts/build_nfhs.py
NFHS = f"{SHARE}.nfhs_5_district_health_indicators"
PIN = f"{SHARE}.india_post_pincode_directory"

# Normalisation, identical on both sides: lowercase, '&' -> 'and', drop non-letters.
def norm_sql(col):
    return f"regexp_replace(replace(lower(trim({col})), '&', 'and'), '[^a-z]', '')"

# State fixes (normalised form -> canonical normalised form)
STATE_FIX = {
    "maharastra": "maharashtra",
    "nctofdelhi": "delhi",
    "thedadraandnagarhavelianddamananddiu": "dadraandnagarhavelianddamananddiu",
}

def state_sql(col):
    expr = norm_sql(col)
    cases = " ".join(f"WHEN '{k}' THEN '{v}'" for k, v in STATE_FIX.items())
    return f"(CASE {expr} {cases} ELSE {expr} END)"

# ---------------------------------------------------------------------------------------------
# Alias table: (state, pin-directory district, NFHS district, note), all as normalised keys.
# 'rename' = same territory, new/other spelling. 'new' = carved out after NFHS-5 (2019 boundaries),
# mapped to the parent district that held most of it; the parent's value is shown, and must be
# labelled as such. Ordered roughly by number of facilities we hold in the PIN district.
ALIASES = [
    ("karnataka", "bengaluruurban", "bangalore", "rename"),
    ("haryana", "gurugram", "gurgaon", "rename"),
    ("westbengal", "paraganasnorth", "northtwentyfourpargana", "rename"),
    ("andhrapradesh", "ntr", "krishna", "new 2022, carved from Krishna (Vijayawada)"),
    ("uttarpradesh", "prayagraj", "allahabad", "rename"),
    ("maharashtra", "raigad", "raigarh", "spelling (NFHS lists Maharashtra's Raigad as Raigarh)"),
    ("andhrapradesh", "visakhapatanam", "visakhapatnam", "spelling"),
    ("assam", "kamrupmetro", "kamrupmetropolitan", "spelling"),
    ("westbengal", "darjeeling", "darjiling", "spelling"),
    ("punjab", "sasnagar", "sahibzadaajitsinghnagar", "abbreviation (SAS Nagar / Mohali)"),
    ("tamilnadu", "chengalpattu", "kancheepuram", "new 2019, carved from Kancheepuram"),
    ("westbengal", "howrah", "haora", "spelling"),
    ("westbengal", "paschimbardhaman", "paschimbarddhaman", "spelling"),
    ("jharkhand", "eastsinghbum", "purbisinghbhum", "rename (East Singhbhum = Purbi Singhbhum)"),
    ("karnataka", "belagavi", "belgaum", "rename"),
    ("maharashtra", "ahmednagar", "ahmadnagar", "spelling"),
    ("westbengal", "hooghly", "hugli", "spelling"),
    ("uttarakhand", "haridwar", "hardwar", "spelling"),
    ("karnataka", "mysuru", "mysore", "rename"),
    ("karnataka", "shivamogga", "shimoga", "rename"),
    ("telangana", "hanumakonda", "warangalurban", "rename 2021 (Warangal Urban -> Hanumakonda)"),
    ("westbengal", "paraganassouth", "southtwentyfourpargana", "rename"),
    ("karnataka", "tumakuru", "tumkur", "rename"),
    ("westbengal", "medinipurwest", "paschimmedinipur", "rename"),
    ("puducherry", "pondicherry", "puducherry", "rename"),
    ("andhrapradesh", "spsrnellore", "sripottisriramulunello", "abbreviation (SPSR Nellore)"),
    ("uttarakhand", "udamsinghnagar", "udhamsinghnagar", "spelling"),
    ("karnataka", "davangere", "davanagere", "spelling"),
    ("karnataka", "ballari", "bellary", "rename"),
    ("karnataka", "bengalururural", "bangalorerural", "rename"),
    ("bihar", "purbichamparan", "purbachamparan", "spelling"),
    ("tamilnadu", "tuticorin", "thoothukkudi", "rename"),
    ("andhrapradesh", "tirupati", "chittoor", "new 2022, carved from Chittoor"),
    ("rajasthan", "jhunjhunu", "jhunjhunun", "spelling"),
    ("punjab", "srimuktsarsahib", "muktsar", "rename"),
    ("tamilnadu", "tenkasi", "tirunelveli", "new 2019, carved from Tirunelveli"),
    ("uttarpradesh", "ayodhya", "faizabad", "rename"),
    ("maharashtra", "buldhana", "buldana", "spelling"),
    ("karnataka", "kalaburagi", "gulbarga", "rename"),
    ("maharashtra", "beed", "bid", "spelling"),
    ("andhrapradesh", "eluru", "westgodavari", "new 2022, carved from West Godavari"),
    ("gujarat", "arvalli", "aravali", "spelling"),
    ("tamilnadu", "villupuram", "viluppuram", "spelling"),
    ("punjab", "malerkotla", "sangrur", "new 2021, carved from Sangrur"),
    ("westbengal", "medinipureast", "purbamedinipur", "rename"),
    ("andhrapradesh", "annamayya", "ysr", "new 2022, carved mainly from YSR Kadapa"),
    ("karnataka", "chikkamagaluru", "chikmagalur", "rename"),
    ("tamilnadu", "ranipet", "vellore", "new 2019, carved from Vellore"),
    ("punjab", "firozepur", "firozpur", "spelling"),
    ("westbengal", "coochbehar", "kochbihar", "spelling"),
    ("andhrapradesh", "bapatla", "guntur", "new 2022, carved mainly from Guntur"),
    ("tamilnadu", "kanchipuram", "kancheepuram", "spelling"),
    ("westbengal", "purulia", "puruliya", "spelling"),
    ("andhrapradesh", "palnadu", "guntur", "new 2022, carved from Guntur"),
    ("haryana", "charkidadri", "charkhidadri", "spelling"),
    ("andhrapradesh", "kakinada", "eastgodavari", "new 2022, carved from East Godavari"),
    ("jammuandkashmir", "budgam", "badgam", "spelling"),
    ("uttarpradesh", "bhadohi", "santravidasnagarbhadohi", "rename"),
    ("rajasthan", "jalore", "jalor", "spelling"),
    ("madhyapradesh", "khargone", "khargonewestnimar", "rename"),
    ("maharashtra", "gondia", "gondiya", "spelling"),
    ("tamilnadu", "kallakurichi", "viluppuram", "new 2019, carved from Viluppuram"),
    ("andhrapradesh", "srisathyasai", "anantapur", "new 2022, carved from Anantapur"),
    ("karnataka", "vijayapura", "bijapur", "rename"),
    ("andhrapradesh", "nandyal", "kurnool", "new 2022, carved from Kurnool"),
    ("madhyapradesh", "eastnimar", "khandwaeastnimar", "rename"),
    ("westbengal", "dinajpuruttar", "uttardinajpur", "rename"),
    ("tamilnadu", "tirupathur", "vellore", "new 2019, carved from Vellore"),
    ("andhrapradesh", "anakapalli", "visakhapatnam", "new 2022, carved from Visakhapatnam"),
    ("uttarpradesh", "kasganj", "kanshiramnagar", "rename"),
    ("andamanandnicobarislands", "southandamans", "southandaman", "spelling"),
    ("jammuandkashmir", "baramulla", "baramula", "spelling"),
    ("chhattisgarh", "kabirdham", "kabeerdham", "spelling"),
    ("uttarpradesh", "amroha", "jyotibaphulenagar", "rename"),
    ("rajasthan", "chittorgarh", "chittaurgarh", "spelling"),
    ("uttarpradesh", "maharajganj", "mahrajganj", "spelling"),
    ("tamilnadu", "mayiladuthurai", "nagapattinam", "new 2020, carved from Nagapattinam"),
    ("andhrapradesh", "konaseema", "eastgodavari", "new 2022, carved from East Godavari"),
    ("jharkhand", "koderma", "kodarma", "spelling"),
    ("karnataka", "vijaynagar", "bellary", "new 2020 (Vijayanagara), carved from Bellary"),
    ("odisha", "nabarangpur", "nabarangapur", "spelling"),
    ("gujarat", "chhotaudepur", "chhotaudaipur", "spelling"),
    ("uttarpradesh", "santkabeernagar", "santkabirnagar", "spelling"),
    ("westbengal", "kalimpong", "darjiling", "new 2017, carved from Darjiling"),
    ("chhattisgarh", "korea", "koriya", "spelling"),
    ("andhrapradesh", "parvathipurammanyam", "vizianagaram", "new 2022, carved mainly from Vizianagaram"),
    ("assam", "marigaon", "morigaon", "spelling"),
    ("madhyapradesh", "narsinghpur", "narsimhapur", "spelling"),
    ("uttarpradesh", "hathras", "mahamayanagar", "rename"),
    ("karnataka", "chamarajanagara", "chamarajanagar", "spelling"),
    # NFHS districts with no facility today but a PIN-side spelling difference
    ("haryana", "nuh", "mewat", "rename"),
    ("jammuandkashmir", "poonch", "punch", "spelling"),
    ("jammuandkashmir", "shopian", "shupiyan", "spelling"),
    ("jammuandkashmir", "bandipora", "bandipore", "spelling"),
    ("jharkhand", "sahebganj", "sahibganj", "spelling"),
    ("jharkhand", "westsinghbhum", "pashchimisinghbhum", "rename"),
    ("gujarat", "dangs", "thedangs", "spelling"),
    ("chhattisgarh", "kondagaon", "kodagaon", "spelling"),
    ("chhattisgarh", "kanker", "uttarbastarkanker", "rename"),
    ("odisha", "boudh", "baudh", "spelling"),
    ("odisha", "deogarh", "debagarh", "spelling"),
    ("odisha", "sonepur", "subarnapur", "rename"),
    ("rajasthan", "dholpur", "dhaulpur", "spelling"),
    ("telangana", "kumurambheemasifabad", "komarambheemasifabad", "spelling"),
    ("telangana", "warangal", "warangalrural", "rename 2021 (Warangal Rural -> Warangal)"),
    ("uttarakhand", "paurigarhwal", "garhwal", "rename"),
    ("uttarpradesh", "shravasti", "shrawasti", "spelling"),
    ("westbengal", "dinajpurdakshin", "dakshindinajpur", "rename"),
    ("westbengal", "alipurduar", "jalpaiguri", "new 2014, carved from Jalpaiguri"),
    ("westbengal", "jhargram", "paschimmedinipur", "new 2017, carved from Paschim Medinipur"),
    ("meghalaya", "eastjaintiahills", "eastjantiahills", "spelling"),
]
# Not aliased: West Bengal 'Purba Bardhaman' (absent from this 706-row NFHS table), and
# Mizoram 'Chandel' (a duplicate Manipur row in the NFHS table).

# ---------------------------------------------------------------------------------------------
# Service -> indicator map. direction: 'higher_is_worse' (rank 1 = highest value) or
# 'lower_is_worse' (rank 1 = lowest value). indicator_column NULL = deliberately no indicator.
SERVICE_MAP = [
    ("dialysis", "m15_plus_with_high_or_very_high_gt_140_mg_dl_blood_sugar_or_pct",
     "Men 15+ with high blood sugar or on medication for it", "higher_is_worse",
     "Diabetes is the leading cause of kidney failure; a risk factor years upstream, not dialysis demand."),
    ("dialysis", "w15_plus_with_high_or_very_high_gt_140_mg_dl_blood_sugar_or_pct",
     "Women 15+ with high blood sugar or on medication for it", "higher_is_worse",
     "Same diabetes risk factor for women; risk factor for kidney disease, not a patient count."),
    ("dialysis", "m15_plus_with_high_bp_sys_gte_140_mmhg_and_or_dia_gte_90_mm_pct",
     "Men 15+ with high blood pressure or on medication for it", "higher_is_worse",
     "Hypertension is the second main cause of kidney failure; risk factor only."),
    ("caesarean_section", "births_delivered_by_csection_5y_pct",
     "Births delivered by caesarean section (last 5 years)", "lower_is_worse",
     "Direct measure of C-section use; very low (<5%) suggests unmet need, very high suggests over-use, so show the number without a good/bad label. Rank 1 = lowest rate."),
    ("caesarean_section", "institutional_birth_5y_pct",
     "Births in a health facility (last 5 years)", "lower_is_worse",
     "A C-section needs a facility birth; a low share means many births happen where surgery is not available."),
    ("nicu", "institutional_birth_5y_pct",
     "Births in a health facility (last 5 years)", "lower_is_worse",
     "Newborns born at home cannot reach neonatal care quickly; risk context, not NICU demand."),
    ("nicu", "births_attended_by_skilled_hp_5y_10_pct",
     "Births attended by a skilled health worker (last 5 years)", "lower_is_worse",
     "Unattended births raise the risk of newborn complications going unrecognised; risk context only."),
    ("nicu", "all_w15_49_who_are_anaemic_pct",
     "Women 15-49 who are anaemic", "higher_is_worse",
     "Maternal anaemia is linked to preterm birth and low birth weight, the main reasons for NICU care; risk factor only."),
    ("cardiac_surgery", "m15_plus_with_high_bp_sys_gte_140_mmhg_and_or_dia_gte_90_mm_pct",
     "Men 15+ with high blood pressure or on medication for it", "higher_is_worse",
     "Hypertension is a major risk factor for heart disease; years upstream of surgery."),
    ("cardiac_surgery", "m15_plus_with_high_or_very_high_gt_140_mg_dl_blood_sugar_or_pct",
     "Men 15+ with high blood sugar or on medication for it", "higher_is_worse",
     "Diabetes is a major risk factor for coronary artery disease; risk factor only."),
    ("cardiac_surgery", "m15_plus_who_use_any_kind_of_tobacco_pct",
     "Men 15+ who use any kind of tobacco", "higher_is_worse",
     "Tobacco use is a major risk factor for coronary artery disease; risk factor only."),
    ("mri", None, None, None,
     "No defensible population indicator: MRI need is not measured by NFHS-5. Show no district context."),
    ("burn_care", None, None, None,
     "No defensible population indicator. Unclean cooking fuel is a plausible burn-risk proxy, but too indirect to show; show no district context."),
]

UNITS = {}  # every mapped indicator is a percentage; set in SQL as '%'


def q(cur, stmt):
    cur.execute(stmt)
    try:
        return cur.fetchall()
    except Exception:
        return None


def esc(s):
    return "NULL" if s is None else "'" + s.replace("'", "''") + "'"


def nfhs_main():
    con = connect()
    cur = con.cursor()

    # ---- nfhs_district --------------------------------------------------------------------
    cols = q(cur, f"""SELECT column_name, data_type FROM {SHARE.split('.')[0]}.information_schema.columns
                     WHERE table_schema='{SHARE.split('.')[1]}' AND table_name='nfhs_5_district_health_indicators'
                     ORDER BY ordinal_position""")
    dbl = [c for c, t in cols if t == "DOUBLE"]
    assert len(dbl) == 58, len(dbl)
    mapped = sorted({(r[1], r[3]) for r in SERVICE_MAP if r[1]})
    for c, _ in mapped:
        assert c in dbl, c
    rank_cols = []
    for c, d in mapped:
        order = "DESC" if d == "higher_is_worse" else "ASC"
        rank_cols.append(f"CAST(rank() OVER (ORDER BY {c} {order}) AS INT) AS rank_{c}")
        rank_cols.append(f"CAST(ntile(4) OVER (ORDER BY {c} {order}) AS INT) AS quartile_{c}")
    q(cur, f"""CREATE OR REPLACE TABLE workspace.vf.nfhs_district AS
      SELECT concat({state_sql('state_ut')}, '|', {norm_sql('district_name')}) AS nfhs_key,
             trim(district_name) AS district, trim(state_ut) AS state,
             {', '.join(dbl)},
             {', '.join(rank_cols)}
      FROM {NFHS}""")

    # ---- district_alias -------------------------------------------------------------------
    vals = ",\n".join(f"({esc(s)},{esc(p)},{esc(n)},{esc(note)})" for s, p, n, note in ALIASES)
    q(cur, f"""CREATE OR REPLACE TEMPORARY VIEW alias_raw AS
      SELECT * FROM VALUES {vals} AS t(state_key, pin_key, nfhs_district_key, note)""")
    # resolve keys to display names; refuse aliases whose keys do not exist on both sides
    q(cur, f"""CREATE OR REPLACE TEMPORARY VIEW pin_names AS
      SELECT {state_sql('statename')} AS state_key, {norm_sql('district')} AS pin_key,
             max(trim(district)) AS pin_district_name, max(trim(statename)) AS pin_state
      FROM {PIN} GROUP BY 1, 2""")
    bad = q(cur, """SELECT a.state_key, a.pin_key, a.nfhs_district_key,
                          p.pin_key IS NULL AS pin_missing, n.nfhs_key IS NULL AS nfhs_missing,
                          e.nfhs_key IS NOT NULL AS pin_already_exact
                   FROM alias_raw a
                   LEFT JOIN pin_names p USING (state_key, pin_key)
                   LEFT JOIN workspace.vf.nfhs_district n ON n.nfhs_key = a.state_key || '|' || a.nfhs_district_key
                   LEFT JOIN workspace.vf.nfhs_district e ON e.nfhs_key = a.state_key || '|' || a.pin_key
                   WHERE p.pin_key IS NULL OR n.nfhs_key IS NULL OR e.nfhs_key IS NOT NULL""")
    for b in bad:
        print("alias dropped:", b)
    q(cur, """CREATE OR REPLACE TABLE workspace.vf.district_alias AS
      SELECT p.pin_district_name, p.pin_state, n.district AS nfhs_district, n.state AS nfhs_state,
             n.nfhs_key, a.state_key || '|' || a.pin_key AS pin_key, a.note
      FROM alias_raw a
      JOIN pin_names p USING (state_key, pin_key)
      JOIN workspace.vf.nfhs_district n ON n.nfhs_key = a.state_key || '|' || a.nfhs_district_key
      LEFT JOIN workspace.vf.nfhs_district e ON e.nfhs_key = a.state_key || '|' || a.pin_key
      WHERE e.nfhs_key IS NULL""")

    # ---- pin_district ---------------------------------------------------------------------
    q(cur, f"""CREATE OR REPLACE TABLE workspace.vf.pin_district AS
      WITH po AS (
        SELECT lpad(CAST(pincode AS STRING), 6, '0') AS pin, trim(district) AS district, trim(statename) AS state,
               concat({state_sql('statename')}, '|', {norm_sql('district')}) AS pin_key
        FROM {PIN} WHERE upper(trim(statename)) <> 'NA' AND pincode IS NOT NULL),
      g AS (
        SELECT pin, pin_key, max(district) AS district, max(state) AS state, count(*) AS n_offices
        FROM po GROUP BY pin, pin_key),
      s AS (
        SELECT *, n_offices / sum(n_offices) OVER (PARTITION BY pin) AS pin_share,
               row_number() OVER (PARTITION BY pin ORDER BY n_offices DESC, pin_key) AS share_rank
        FROM g)
      SELECT s.pin, s.district, s.state, s.pin_key, CAST(s.n_offices AS INT) AS n_offices,
             ROUND(s.pin_share, 4) AS pin_share, s.share_rank = 1 AS is_primary,
             coalesce(e.nfhs_key, a.nfhs_key) AS nfhs_key,
             CASE WHEN e.nfhs_key IS NOT NULL THEN 'exact' WHEN a.nfhs_key IS NOT NULL THEN 'alias' END AS match_method
      FROM s
      LEFT JOIN workspace.vf.nfhs_district e ON e.nfhs_key = s.pin_key
      LEFT JOIN workspace.vf.district_alias a ON a.pin_key = s.pin_key""")

    # ---- facility_district ----------------------------------------------------------------
    q(cur, """CREATE OR REPLACE TABLE workspace.vf.facility_district AS
      SELECT f.unique_id, f.postcode AS pin, p.district, p.state, p.pin_key, p.pin_share,
             p.nfhs_key, p.match_method
      FROM workspace.vf.facilities_clean f
      LEFT JOIN workspace.vf.pin_district p ON p.pin = f.postcode AND p.is_primary AND f.postcode_valid
      WHERE NOT f.is_shifted""")

    # ---- service_indicator_map ------------------------------------------------------------
    vals = ",\n".join(
        f"({esc(s)},{esc(c)},{esc(l)},{esc(d)},{esc(r)},{esc('%' if c else None)})"
        for s, c, l, d, r in SERVICE_MAP)
    q(cur, f"""CREATE OR REPLACE TABLE workspace.vf.service_indicator_map AS
      SELECT * FROM VALUES {vals} AS t(service, indicator_column, label_for_users, direction, reason, unit)""")

    comments(cur, dbl, mapped)
    report(cur)


def comments(cur, dbl, mapped):
    c = lambda t, s: q(cur, f"COMMENT ON TABLE workspace.vf.{t} IS {esc(s)}")
    col = lambda t, k, s: q(cur, f"ALTER TABLE workspace.vf.{t} ALTER COLUMN {k} COMMENT {esc(s)}")
    c("nfhs_district", "NFHS-5 (2019-21) district indicators, one row per NFHS row (706). The 58 DOUBLE columns of the "
      "share table copied as-is (no suppressed values). rank_<col>: 1..706, 1 = highest burden (ties share a rank); "
      "quartile_<col>: 1 = highest-burden quarter. Direction per indicator is in service_indicator_map.direction: "
      "higher_is_worse ranks the highest value 1; lower_is_worse ranks the lowest value 1. Context about population "
      "need only; never use for facility tier or ranking. Built by scripts/build_nfhs.py.")
    col("nfhs_district", "nfhs_key", "state|district, both lowercased, & -> and, non-letters dropped; state spellings fixed (Maharastra, NCT of Delhi). Key is never district alone (8 names occur in two states).")
    col("nfhs_district", "district", "NFHS district name, trimmed")
    col("nfhs_district", "state", "NFHS state/UT name as in the source (incl. its spelling 'Maharastra')")
    for k, d in mapped:
        w = "highest value" if d == "higher_is_worse" else "lowest value"
        col("nfhs_district", f"rank_{k}", f"Rank of {k} out of 706, 1 = {w} ({d}). Ties share a rank.")
        col("nfhs_district", f"quartile_{k}", f"Quartile of {k}, 1 = quarter with the {w} ({d}).")
    c("district_alias", "Hand-written aliases from India Post PIN-directory district names to NFHS-5 districts: renames, "
      "spellings, and districts created after the survey mapped to their parent (note says 'new YYYY, carved from X'; "
      "show the parent's value and say so). Only for PIN districts with no exact NFHS match. Built by scripts/build_nfhs.py.")
    col("district_alias", "pin_district_name", "District name as in the PIN directory")
    col("district_alias", "pin_state", "State name as in the PIN directory")
    col("district_alias", "nfhs_district", "NFHS district it maps to")
    col("district_alias", "nfhs_state", "NFHS state")
    col("district_alias", "nfhs_key", "Join key to nfhs_district")
    col("district_alias", "pin_key", "Normalised state|district of the PIN-directory name (join key to pin_district.pin_key)")
    col("district_alias", "note", "rename / spelling / new district with its parent")
    c("pin_district", "One row per (PIN, district) from the India Post directory (grain there is post office). pin_share = "
      "share of the PIN's post offices in that district; is_primary marks the majority district per PIN. nfhs_key via "
      "exact normalised name or district_alias. Built by scripts/build_nfhs.py.")
    col("pin_district", "pin", "6-digit PIN code as string")
    col("pin_district", "district", "PIN-directory district name")
    col("pin_district", "state", "PIN-directory state name")
    col("pin_district", "pin_key", "Normalised state|district")
    col("pin_district", "n_offices", "Post offices of this PIN in this district")
    col("pin_district", "pin_share", "n_offices / all post offices of the PIN")
    col("pin_district", "is_primary", "True for the district with most of the PIN's post offices (one per PIN)")
    col("pin_district", "nfhs_key", "Join key to nfhs_district; NULL if no exact or alias match")
    col("pin_district", "match_method", "'exact' | 'alias' | NULL")
    c("facility_district", "Non-shifted facilities (facilities_clean) -> district of their postcode (primary district of "
      "the PIN). NULL district when the postcode is invalid or not in the PIN directory. Built by scripts/build_nfhs.py.")
    col("facility_district", "unique_id", "Joins facilities_clean")
    col("facility_district", "pin", "facilities_clean.postcode")
    col("facility_district", "pin_share", "Share of the PIN's post offices in this district (below 1 = PIN spans districts)")
    col("facility_district", "nfhs_key", "Join key to nfhs_district; NULL if not linked")
    col("facility_district", "match_method", "'exact' | 'alias' | NULL")
    c("service_indicator_map", "Hand mapping of six headline services to 0-3 clean NFHS-5 indicators with a reason. A row "
      "with NULL indicator_column means deliberately no indicator (show no district context). Indicators are risk or "
      "use context, never service demand. Built by scripts/build_nfhs.py.")
    col("service_indicator_map", "indicator_column", "Column in nfhs_district (rank_/quartile_ prefixed columns hold its rank)")
    col("service_indicator_map", "direction", "higher_is_worse (rank 1 = highest value) or lower_is_worse (rank 1 = lowest value)")
    col("service_indicator_map", "unit", "Unit of the value")


def report(cur):
    for t in ["nfhs_district", "district_alias", "pin_district", "facility_district", "service_indicator_map"]:
        print(t, q(cur, f"SELECT count(*) FROM workspace.vf.{t}")[0][0])
    print(q(cur, """SELECT count(*) n, count(district) with_district, count(nfhs_key) linked,
                         count_if(match_method='exact') exact, count_if(match_method='alias') alias,
                         count(DISTINCT nfhs_key) nfhs_districts
                  FROM workspace.vf.facility_district"""))
    print("top unlinked:", q(cur, """SELECT state, district, count(*) n FROM workspace.vf.facility_district
                  WHERE district IS NOT NULL AND nfhs_key IS NULL GROUP BY 1,2 ORDER BY n DESC LIMIT 10"""))

In [0]:
if REBUILD:
    nfhs_main()            # builds the five tables, comments, then prints the report
else:
    with connect() as con:
        report(con.cursor())   # read-only: row counts and linkage

nfhs_district 706
district_alias 110
pin_district 20824
facility_district 9989
service_indicator_map 13
[(9989, 9700, 9689, 8242, 1447, 532)]
top unlinked: [('WEST BENGAL', 'PURBA BARDHAMAN', 11)]


In [0]:
%sql
-- Rayagada (PIN 765019), the NICU demo: the district's figures for the NICU indicators
SELECT p.pin, n.district, n.state, m.label_for_users, m.indicator_column,
       CASE m.indicator_column
         WHEN 'institutional_birth_5y_pct' THEN n.institutional_birth_5y_pct
         WHEN 'births_attended_by_skilled_hp_5y_10_pct' THEN n.births_attended_by_skilled_hp_5y_10_pct
         WHEN 'all_w15_49_who_are_anaemic_pct' THEN n.all_w15_49_who_are_anaemic_pct END AS value_pct,
       CASE m.indicator_column
         WHEN 'institutional_birth_5y_pct' THEN n.rank_institutional_birth_5y_pct
         WHEN 'births_attended_by_skilled_hp_5y_10_pct' THEN n.rank_births_attended_by_skilled_hp_5y_10_pct
         WHEN 'all_w15_49_who_are_anaemic_pct' THEN n.rank_all_w15_49_who_are_anaemic_pct END AS rank_of_706
FROM workspace.vf.pin_district p
JOIN workspace.vf.nfhs_district n ON n.nfhs_key = p.nfhs_key
JOIN workspace.vf.service_indicator_map m ON m.service = 'nicu'
WHERE p.pin = '765019' AND p.is_primary

pin,district,state,label_for_users,indicator_column,value_pct,rank_of_706
765019,Rayagada,Odisha,Births in a health facility (last 5 years),institutional_birth_5y_pct,68.9,41
765019,Rayagada,Odisha,Births attended by a skilled health worker (last 5 years),births_attended_by_skilled_hp_5y_10_pct,72.1,36
765019,Rayagada,Odisha,Women 15-49 who are anaemic,all_w15_49_who_are_anaemic_pct,69.1,87


## Step 8 — Blind validation

**What.** There is no ground truth, so we sampled facilities from each tier and checked each one against its own website or one web search, without seeing our tier. A verdict is `yes` (offers the service), `no`, or `cannot tell` (no website, or the site does not say). Three services: dialysis (80 sampled), caesarean section (31), cardiac surgery (40).

**Result.** Every Confirmed facility that could be checked offers the service: 30 of 30 across the three services. For cardiac surgery the share falls tier by tier, which is what the tiers are meant to show. The dialysis Unverified facilities that did not offer it were eye hospitals and diagnostic centres, the cases the checks are built to catch.

**Cautions.** After the dialysis check two rules were relaxed (heart hospitals can run dialysis; a dedicated dialysis unit counts as detailed evidence), so that check is no longer independent; the C-section and cardiac checks were scored against rules fixed beforehand. About 40% of sampled facilities could not be checked at all, mostly small ones with no website.

**Reads** `validation_dialysis` (tier at sampling time) and `validation_wave3` (tier now). The query is the one the app uses for its validation line (`app/queries.py`, `WEB_CHECK_SQL`).

In [0]:
%sql
SELECT service, tier, count_if(verdict IN ('yes', 'no')) AS checked,
       count_if(verdict = 'yes') AS yes, count_if(verdict = 'no') AS no,
       count_if(verdict = 'cannot tell') AS cannot_tell
FROM (SELECT 'dialysis' AS service, tier_at_sample AS tier, verdict FROM workspace.vf.validation_dialysis
      UNION ALL
      SELECT service, tier_now AS tier, verdict FROM workspace.vf.validation_wave3)
GROUP BY service, tier
ORDER BY service, CASE tier WHEN 'Confirmed' THEN 0 WHEN 'Likely' THEN 1 WHEN 'Possible' THEN 2 ELSE 3 END

service,tier,checked,yes,no,cannot_tell
caesarean_section,Confirmed,7,7,0,4
caesarean_section,Possible,4,4,0,6
caesarean_section,Unverified,0,0,0,10
cardiac_surgery,Confirmed,8,8,0,3
cardiac_surgery,Likely,5,4,1,4
cardiac_surgery,Possible,6,1,5,4
cardiac_surgery,Unverified,4,0,4,6
dialysis,Confirmed,15,15,0,5
dialysis,Likely,9,9,0,11
dialysis,Possible,11,11,0,9


In [0]:
%sql
-- The 'no' answers, with the evidence the checker wrote down
SELECT 'dialysis' AS service, name, city, tier_at_sample AS tier, verdict, evidence FROM workspace.vf.validation_dialysis WHERE verdict = 'no'
UNION ALL
SELECT service, name, city, tier_now, verdict, evidence FROM workspace.vf.validation_wave3 WHERE verdict = 'no'
ORDER BY service, tier

service,name,city,tier,verdict,evidence
cardiac_surgery,East Delhi Medical Centre,New Delhi,Likely,no,Heart menu lists only 'Cardiology: Non-Invasive'; specialities list omits cardiac surgery
cardiac_surgery,Vaishali Hospital & Surgical Research Center Jaipur,Jaipur,Possible,no,"Specialties: Laparoscopic, Plastic, Open Surgery, Nephrology, Anorectal, Gynae, Urology, Orthopaedic; no cardiac"
cardiac_surgery,Q1 Hospitals,Visakhapatnam,Possible,no,"Surgical list: joint replacement, spine, neuro, plastic, vascular, urology, OBG, ENT; only Cardiology, no cardiac surgery"
cardiac_surgery,Christian Hospital Mungeli,Mungeli,Possible,no,"Services: General Surgery, Emergency, Obstetrics, Dentistry, Eye, Burn Unit; no cardiac surgery listed"
cardiac_surgery,"Matra Drishti Medi Care, Jankipuram Extension",Lucknow,Possible,no,"Own site: Women's Healthcare, Eye Care, General Surgery/Medicine, Cardiology OPD; no cardiac surgery"
cardiac_surgery,B&B Memorial Hospital,Cochin,Possible,no,"Specialties: Anaesthesia, Ortho, Gen Med, ENT, OBG, Radiology, Cardiology, Nephrology, Emergency; no cardiac surgery"
cardiac_surgery,Niraamaya Diagnostics,Varanasi,Unverified,no,"Best Diagnostic Centre in Varanasi: MRI, CT, X-ray, ECG, 2-D Echo; no surgery"
cardiac_surgery,Kushwaha Eye Hospital,Muzaffarnagar,Unverified,no,Kushwaha eye hospital is the leading eye hospital in the region; all services eye-related
cardiac_surgery,Ultra Care Clinic,Pune,Unverified,no,"Treatments focus on laparoscopic surgeries (hernia, gallstone, appendix) and anorectal laser; no cardiac surgery"
cardiac_surgery,Jai Kamal Eye Hospital,Amritsar,Unverified,no,a single speciality 15 bedded ... Hospital for Ophthalmic conditions; no cardiac surgery


## Step 9 — The app's core logic: rank by tier, then location, then distance

At query time nothing is re-read by a model. The user's location becomes a point (a clicked map point, or a PIN code -> `pin_point`), and one SQL query returns the facilities that have a verdict for the service: **best tier first**; inside a tier, facilities with real checked coordinates before those placed only at a city or PIN centre; then nearest by `haversine_km`; up to 20 per tier. Weak locations add a warning.

Below: the ranking query copied from `app/queries.py`, run read-only for **dialysis near PIN 613001 (Thanjavur)**, then the plain-language matcher from `app/matcher.py` (its prompt and the label validation that every model answer passes through).

In [0]:
# app/queries.py: the ranking query and the search function (read-only)
FALLBACK_SOURCES = ("city_point", "pin_point")   # facility point is a centroid, not an address
CITY_BLOCK_KM = 25   # centre-placed facilities this close get their own block per tier


def _rows(cn, sql, params=None):
    with cn.cursor() as c:
        c.execute(sql, params or [])
        cols = [d[0] for d in c.description]
        return [dict(zip(cols, r)) for r in c.fetchall()]


def is_fallback(location_source: str) -> bool:
    return location_source in FALLBACK_SOURCES


PIN_SQL = f"""
SELECT pin, lat, lon, n_offices, spread_km
FROM {T_PIN_POINT} WHERE pin = ?
"""

# Within a tier, facilities with real coordinates come before those placed only at a
# city or PIN centroid; then distance. Up to {per_tier} rows per tier.
SERVICE_SQL = """
SELECT * FROM (
  SELECT fs.unique_id, c.name, c.city, c.state, c.postcode, l.address,
         c.official_domain, c.official_phone, c.specialties, c.source_domains,
         l.final_lat, l.final_lon,
         {hav}(?, ?, l.final_lat, l.final_lon) AS distance_km,
         l.location_source, l.location_confidence, l.location_note,
         fs.tier, fs.n_explicit, fs.n_stated, fs.n_weak,
         fs.department_backing, fs.type_fit, fs.audit_verdict,
         {extra}
         to_json(fs.evidence) AS evidence_json,
         to_json(fs.warnings) AS warnings_json,
         row_number() OVER (
           PARTITION BY fs.tier
           ORDER BY CASE WHEN l.location_source IN ('city_point', 'pin_point') THEN 1 ELSE 0 END,
                    {hav}(?, ?, l.final_lat, l.final_lon), c.name) AS tier_rank
  FROM {tbl} fs
  JOIN {fc} c ON c.unique_id = fs.unique_id
  JOIN {loc} l ON l.facility_id = fs.unique_id
  WHERE fs.service = ?
    AND NOT coalesce(c.is_shifted, false)
    AND l.final_lat IS NOT NULL AND l.final_lon IS NOT NULL
    AND coalesce(l.location_confidence, 'none') <> 'none')
WHERE tier_rank <= ?
   -- city/PIN-centre facilities near the user are never cut by the per-tier cap
   OR (location_source IN ('city_point', 'pin_point') AND distance_km <= {city_km})
ORDER BY CASE tier WHEN 'Confirmed' THEN 0 WHEN 'Likely' THEN 1
                   WHEN 'Possible' THEN 2 WHEN 'Unverified' THEN 3 ELSE 4 END, tier_rank
"""


def service_search(cn, table: str, service: str, user_lat: float,
                   user_lon: float, per_tier: int = 20) -> list[dict]:
    """Facilities with a verdict for `service`, up to `per_tier` per tier: best tier
    first; within a tier real coordinates before centroid-placed facilities, then
    nearest. tier_rank 1 is the tier's nearest. Location warnings are appended when
    the facility's point is weak."""
    extra = ("fs.facility_type, fs.flagged_share," if table == T_SERVICE
             else "CAST(NULL AS STRING) AS facility_type, CAST(NULL AS DOUBLE) AS flagged_share,")
    sql = SERVICE_SQL.format(hav=F_HAVERSINE, tbl=table, fc=T_FAC_CLEAN, loc=T_FAC_LOC,
                             extra=extra, city_km=CITY_BLOCK_KM)
    rows = _rows(cn, sql, [float(user_lat), float(user_lon), float(user_lat), float(user_lon),
                           service, int(per_tier)])
    for r in rows:
        r["evidence"] = json.loads(r.pop("evidence_json") or "[]")
        warns = json.loads(r.pop("warnings_json") or "[]")
        if r["location_confidence"] in WEAK_LOCATION and not is_fallback(r["location_source"]):
            warns.append(f"Location {r['location_confidence']}: "
                         f"{r['location_note'] or 'position not verified'}")
        r["warnings"] = warns
        r["specialties"] = list(r["specialties"]) if r["specialties"] is not None else []
        r["source_domains"] = list(r["source_domains"]) if r.get("source_domains") is not None else []
    return rows

In [0]:
cn = connect()
pin = _rows(cn, PIN_SQL, ["613001"])[0]
print("PIN 613001 ->", pin)
rows = service_search(cn, T_SERVICE, "dialysis", pin["lat"], pin["lon"], per_tier=3)
for r in rows:
    print(f"{r['tier']:<10} {r['distance_km']:7.2f} km  {r['name']} ({r['city']})  location={r['location_confidence']}")
    for e in r["evidence"][:2]:
        print(f"{'':12}\"{e['sentence']}\"  [{e['source_column']}]")
    for w in r["warnings"]:
        print(f"{'':12}warning: {w}")

PIN 613001 -> {'pin': '613001', 'lat': 10.78658335, 'lon': 79.14, 'n_offices': 10, 'spread_km': 2.143240837176342}
Confirmed     0.95 km  Sudar Hospital and Research Institute (Thanjavur)  location=high
            "Dialysis unit"  [capability]
            "Dialysis procedures are provided"  [procedure]
Confirmed    48.78 km  GVN Hospital (Tiruchirappalli)  location=high
            "ICU with dialysis facilities"  [equipment]
            "ICU with dialysis capability"  [equipment]
Confirmed   116.21 km  MM Hospital Namakkal (Namakkal)  location=high
            "Dialysis machines"  [equipment]
            "Offers Orthopedics & Joint Replacement, Obstetrics & Gynecology, General Surgery, Spine Surgery, Plastic Surgery, Dialysis"  [capability]
Likely        1.33 km  Anu Specialty Hospital (Thanjavur)  location=good
            "Peritoneal dialysis"  [procedure]
            "Hemodialysis"  [procedure]
Likely        1.43 km  Vinodhagan Memorial Hospital (Thanjavur)  location=good
         

In [0]:
# app/matcher.py: map free text to service labels with ONE LLM call; every label is validated
ENDPOINT = os.getenv("SERVING_ENDPOINT", "databricks-meta-llama-3-3-70b-instruct")
MATCH_ALLOWED = {s: n for g in SERVICE_GROUPS.values() for s, n in g if s not in HIDDEN_SERVICES}

PROMPT = """You map a person's words to hospital service labels. Do not give medical advice.
Labels (label: meaning):
{labels}

Text: "{text}"

Step 1. Classify the text:
- "service": it names a service or procedure (e.g. dialysis, MRI, C-section, knee replacement).
- "symptom": it describes a problem or symptom (e.g. "I can't jump", "chest pain", "my child has fever").
- "unclear": it is not a health need, or cannot be understood (gibberish, random words). Then return no
  options and confidence "low"; do not guess.
Step 2. Choose up to 3 labels from the list, each with a plain reason of at most 10 words
(e.g. "Joint or leg problem -> orthopaedic clinic").
For "symptom": order options least invasive first: outpatient_opd (clinic), physiotherapy_rehab,
diagnostics (imaging, lab) before any surgery label. A surgery label may only come after a non-surgical
one, unless the text explicitly asks for surgery. Confidence may not be "high" for a symptom.
Exactly one option has "fit": "best": the label that most specifically matches the text. Prefer a specific
medical specialty when the text points to one (e.g. arm numbness with breathlessness -> cardiology_medical,
sugar problem -> diabetes_endocrinology) over the generic outpatient_opd. List the best option first, then the
others; the others are "possible".
"emergency" is true only if the text suggests an urgent, possibly life-threatening situation. When it is
true, do not offer outpatient_opd or health_checkup_vaccination; include trauma_emergency (emergency
department) or icu_critical_care among the options, plus the matching specialty.

Return ONLY JSON, no other text:
{{"kind": "service" | "symptom" | "unclear", "options": [{{"service": "label", "reason": "...", "fit": "best" | "possible"}}], "confidence": "high" | "medium" | "low", "emergency": true | false}}"""

EMERGENCY_LABELS = {"trauma_emergency", "icu_critical_care", "ambulance"}
NOT_FOR_EMERGENCY = {"outpatient_opd", "health_checkup_vaccination"}
SURGICAL = {"general_surgery", "cardiac_surgery", "interventional_cardiology", "neurosurgery",
            "orthopedic_surgery", "joint_replacement", "spine_surgery", "eye_surgery", "transplant",
            "cosmetic_aesthetic", "cancer_surgery", "caesarean_section", "vascular_surgery"}
ASKS_SURGERY = re.compile(r"surg|operat|replace|transplant|c-?section|caesar|cesar|bypass|angioplast", re.I)


def _call(text: str) -> str:
    from databricks.sdk import WorkspaceClient
    from databricks.sdk.service.serving import ChatMessage, ChatMessageRole
    labels = "\n".join(f"{s}: {n}" for s, n in MATCH_ALLOWED.items())
    r = WorkspaceClient().serving_endpoints.query(
        name=ENDPOINT, temperature=0, max_tokens=350,
        messages=[ChatMessage(role=ChatMessageRole.USER,
                              content=PROMPT.format(labels=labels, text=text[:500]))])
    return r.choices[0].message.content


def _clean(d: dict, text: str) -> dict:
    kind = d.get("kind") if d.get("kind") in ("service", "symptom", "unclear") else "symptom"
    conf = d.get("confidence") if d.get("confidence") in ("high", "medium", "low") else "low"
    emergency = d.get("emergency") is True
    if kind == "unclear":
        # not understood: no guess at all (the page asks the person to describe the care they need)
        return {"kind": "unclear", "options": [], "services": [], "confidence": "low",
                "emergency": emergency, "error": None}
    opts, seen = [], set()
    for o in d.get("options") or []:
        if not isinstance(o, dict):
            continue
        s = o.get("service")
        if isinstance(s, str) and s in MATCH_ALLOWED and s not in seen:
            seen.add(s)
            reason = " ".join(str(o.get("reason") or "").split()[:12])
            opts.append({"service": s, "reason": reason,
                         "fit": "best" if o.get("fit") == "best" else "possible"})
    if kind == "symptom":
        if conf == "high":
            conf = "medium"
        if not ASKS_SURGERY.search(text):
            # least invasive first: no surgery label before a non-surgical one
            non = [o for o in opts if o["service"] not in SURGICAL]
            sur = [o for o in opts if o["service"] in SURGICAL]
            if sur and not non:
                non = [{"service": "outpatient_opd", "reason": "Start with a clinic visit", "fit": "possible"}]
            opts = non + sur
            # a surgery label cannot be the best guess for a symptom unless surgery was asked for
            if any(o["fit"] == "best" and o["service"] in SURGICAL for o in opts) and non:
                for o in opts:
                    o["fit"] = "possible"
                non[0]["fit"] = "best"
    if emergency:
        # an emergency never gets a clinic or check-up option, and always gets emergency care
        had_best = any(o["fit"] == "best" and o["service"] in NOT_FOR_EMERGENCY for o in opts)
        opts = [o for o in opts if o["service"] not in NOT_FOR_EMERGENCY]
        if had_best:
            for o in opts:
                o["fit"] = "possible"
        if not any(o["service"] in EMERGENCY_LABELS for o in opts):
            opts.insert(1 if opts else 0, {"service": "trauma_emergency",
                                           "reason": "Urgent symptoms -> emergency department",
                                           "fit": "possible" if opts else "best"})
    elif not opts and kind == "symptom":
        opts = [{"service": "outpatient_opd", "reason": "Start with a clinic visit", "fit": "best"}]
    opts = opts[:3]
    if opts and not any(o["fit"] == "best" for o in opts):
        opts[0]["fit"] = "best"
    for o in opts[1:]:
        if o["fit"] == "best" and opts[0]["fit"] == "best":
            o["fit"] = "possible"
    return {"kind": kind, "options": opts, "services": [o["service"] for o in opts],
            "confidence": conf, "emergency": emergency, "error": None}


# The validation, without a model call: a made-up answer with one invented label and an out-of-order surgery.
fake = {"kind": "symptom", "confidence": "high", "emergency": False,
        "options": [{"service": "cardiac_surgery", "reason": "heart", "fit": "best"},
                    {"service": "kidney_machine", "reason": "invented label", "fit": "possible"},
                    {"service": "cardiology_medical", "reason": "heart care", "fit": "possible"}]}
print(json.dumps(_clean(fake, "my chest hurts when I climb stairs"), indent=1))

if RUN_LLM:
    text = "my father needs a kidney machine"
    raw = _call(text)
    print(json.dumps(_clean(json.loads(re.search(r"\{.*\}", raw, re.S).group(0)), text), indent=1))

{
 "kind": "symptom",
 "options": [
  {
   "service": "cardiology_medical",
   "reason": "heart care",
   "fit": "best"
  },
  {
   "service": "cardiac_surgery",
   "reason": "heart",
   "fit": "possible"
  }
 ],
 "services": [
  "cardiology_medical",
  "cardiac_surgery"
 ],
 "confidence": "medium",
 "emergency": false,
 "error": null
}


### District hint: using NFHS to point to nearby care when the text is silent

**Problem.** Search for a C-section near **Thanjavur (PIN 613001)** and the nearest Confirmed provider is about 108 km away. Yet NFHS-5 says 51% of births in Thanjavur district are by caesarean, so C-sections clearly happen nearby; the nearby hospitals' text simply never mentions them.

**Rule.** The app adds a separate hint block under the results when all of these hold:
- the service has a **supply-signal** NFHS indicator (C-section -> births by caesarean; delivery and maternity -> births in a facility). Need indicators such as blood sugar or anaemia are never used here;
- the district is in the **top half** of all 706 NFHS-5 districts for that indicator (value at or above the median);
- **no facility is Confirmed** for the searched service within **30 km**;
- at least one facility within 30 km is **Confirmed or Likely for a related maternity service** (from `service_department_map`, limited to maternity labels). Up to **3** are listed, nearest first.

Each listed facility must have its own quote: a sentence tagged for a maternity service, not flagged as foreign, that names maternity care in words (department lists rank last, then explicit > stated > weak). A facility with no such sentence is skipped and the search continues further out within 30 km; if no facility has a quote, the hint is not shown.

**It never changes tiers or ranking.** The hint is a suggestion to call and check, shown apart from the tiered list. Below: the logic copied from `app/config.py`, `app/queries.py` and `app/main.py`, then a read-only run for C-section at 613001.

In [0]:
# app/config.py + app/queries.py + app/main.py: the district hint (read-only; suggestion block only)
HINT_SUPPLY_INDICATOR = {
    "caesarean_section":    "births_delivered_by_csection_5y_pct",
    "normal_delivery":      "institutional_birth_5y_pct",
    "obstetrics_maternity": "institutional_birth_5y_pct",
}
HINT_RADIUS_KM = 30.0   # hint shows only if no Confirmed for the service within this distance
HINT_MAX = 3            # nearby related facilities listed under the hint
HINT_RELATED_FALLBACK = {
    "caesarean_section":    ["obstetrics_maternity", "high_risk_obstetrics", "normal_delivery"],
    "normal_delivery":      ["obstetrics_maternity", "caesarean_section", "high_risk_obstetrics"],
    "obstetrics_maternity": ["normal_delivery", "caesarean_section", "high_risk_obstetrics"],
}
HINT_RELATED_ALLOWED = {"obstetrics_maternity", "high_risk_obstetrics", "normal_delivery", "caesarean_section"}
HINT_QUOTE_TERMS = (r"(?i)obstetric|maternity|deliver(y|ies)|\blabou?r\b|gyn(a)?ec|antenatal|ante-natal|"
                    r"prenatal|birth|caesarean|cesarean|c-section|\blscs\b")
HINT_DEPT_LIST = (r"(?i)^(offers )?(multiple )?(medical )?(departments?|specialt(y|ies)|ancillaries|includes)\b|"
                  r"departments? (include|of)|specialties include")
HINT_STRENGTH_RANK = {"explicit": 0, "stated": 1, "weak": 2}

RELATED_SQL = f"""
SELECT value FROM {TGT}.service_department_map
WHERE service = ? AND link_type = 'implied_by_service' AND value IS NOT NULL
"""

HINT_DISTRICT_SQL = """
SELECT n.{col} AS value,
       (SELECT percentile_cont(0.5) WITHIN GROUP (ORDER BY {col}) FROM {tgt}.nfhs_district) AS median,
       (SELECT count(*) FROM {svc} fs
          JOIN {fc} c ON c.unique_id = fs.unique_id AND NOT coalesce(c.is_shifted, false)
          JOIN {loc} l ON l.facility_id = fs.unique_id
        WHERE fs.service = ? AND fs.tier = 'Confirmed'
          AND l.final_lat IS NOT NULL AND coalesce(l.location_confidence, 'none') <> 'none'
          AND {hav}(?, ?, l.final_lat, l.final_lon) <= ?) AS n_confirmed_near
FROM {tgt}.nfhs_district n WHERE n.nfhs_key = ?
"""

HINT_FACILITIES_SQL = """
SELECT * FROM (
  SELECT fs.unique_id, c.name, c.city, fs.service, fs.tier,
         l.location_source, l.location_confidence,
         {hav}(?, ?, l.final_lat, l.final_lon) AS distance_km,
         row_number() OVER (PARTITION BY fs.unique_id
                            ORDER BY CASE fs.tier WHEN 'Confirmed' THEN 0 ELSE 1 END, fs.service) AS rk
  FROM {svc} fs
  JOIN {fc} c ON c.unique_id = fs.unique_id AND NOT coalesce(c.is_shifted, false)
  JOIN {loc} l ON l.facility_id = fs.unique_id
  WHERE fs.service IN ({in_list}) AND fs.tier IN ('Confirmed', 'Likely')
    AND l.final_lat IS NOT NULL AND coalesce(l.location_confidence, 'none') <> 'none')
WHERE rk = 1 AND distance_km <= ?
ORDER BY distance_km, name
LIMIT ?
"""

HINT_QUOTES_SQL = """
SELECT DISTINCT s.unique_id, s.sentence, t.strength, t.services
FROM {sent} s
JOIN {dist} d ON d.sentence_key = s.sentence_key
JOIN {tags} t ON t.sentence_id = d.sentence_id
LEFT JOIN {ff} ff ON ff.unique_id = s.unique_id AND ff.source_column = s.source_column
                 AND ff.position = s.position
WHERE s.unique_id IN ({ids}) AND NOT coalesce(s.is_placeholder, false)
  AND NOT coalesce(ff.is_foreign, false)
  AND arrays_overlap(t.services, array({in_list}))
"""


def _in_list(labels):
    return ", ".join("'" + s.replace("'", "") + "'" for s in labels)


def hint_district(cn, service, col, nfhs_key, lat, lon, radius_km):
    sql = HINT_DISTRICT_SQL.format(col=col, tgt=TGT, svc=T_SERVICE, fc=T_FAC_CLEAN, loc=T_FAC_LOC,
                                   hav=F_HAVERSINE)
    r = _rows(cn, sql, [service, float(lat), float(lon), float(radius_km), nfhs_key])
    return r[0] if r else None


def hint_facilities(cn, related, lat, lon, radius_km, limit):
    sql = HINT_FACILITIES_SQL.format(svc=T_SERVICE, fc=T_FAC_CLEAN, loc=T_FAC_LOC, hav=F_HAVERSINE,
                                     in_list=_in_list(related))
    return _rows(cn, sql, [float(lat), float(lon), float(radius_km), int(limit)])


def hint_quotes(cn, ids, related):
    if not ids:
        return []
    sql = HINT_QUOTES_SQL.format(sent=f"{TGT}.facility_sentences", dist=f"{TGT}.distinct_sentences",
                                 tags=f"{TGT}.sentence_tags", ff=f"{TGT}.sentence_foreign_flag",
                                 ids=", ".join("?" for _ in ids), in_list=_in_list(related))
    return _rows(cn, sql, list(ids))


def hint_best_quotes(rows, facs):
    """Keep sentences naming maternity care; rank non-department-lists first, then explicit > stated
    > weak, then tagged with the facility's own related service, then shorter. None if no candidate."""
    own = {f["unique_id"]: f["service"] for f in facs}
    best = {}
    for r in rows:
        sent = (r.get("sentence") or "").strip()
        if not sent or not re.search(HINT_QUOTE_TERMS, sent):
            continue
        svcs = list(r.get("services") or [])
        key = (bool(re.search(HINT_DEPT_LIST, sent)), HINT_STRENGTH_RANK.get(r.get("strength"), 3),
               own.get(r["unique_id"]) not in svcs, len(sent))
        if r["unique_id"] not in best or key < best[r["unique_id"]][0]:
            best[r["unique_id"]] = (key, sent)
    return {k: v[1] for k, v in best.items()}


def district_hint(cn, service, lat, lon, nfhs_key):
    """Returns the hint dict or None. Never touches tiers or ranking."""
    col = HINT_SUPPLY_INDICATOR.get(service)
    if not col or lat is None or not nfhs_key:
        return None
    d = hint_district(cn, service, col, nfhs_key, lat, lon, HINT_RADIUS_KM)
    if not d or d["value"] is None or d["median"] is None:
        return None
    if d["value"] < d["median"] or d["n_confirmed_near"]:
        return None                                   # not top half, or a Confirmed is within 30 km
    related = [r["value"] for r in _rows(cn, RELATED_SQL, [service])] or HINT_RELATED_FALLBACK.get(service, [])
    related = [s for s in related if s in HINT_RELATED_ALLOWED and s != service]
    if not related:
        return None
    facs = hint_facilities(cn, related, lat, lon, HINT_RADIUS_KM, HINT_MAX)
    if not facs:
        return None
    quotes = hint_best_quotes(hint_quotes(cn, [f["unique_id"] for f in facs], sorted(HINT_RELATED_ALLOWED)), facs)
    return {"indicator": col, "value": float(d["value"]), "median_706": float(d["median"]),
            "facilities": [{**f, "sentence": quotes.get(f["unique_id"])} for f in facs]}

In [0]:
# Read-only example: C-section near PIN 613001 (Thanjavur)
cn = connect()
pin = _rows(cn, PIN_SQL, ["613001"])[0]
nk = _rows(cn, f"SELECT nfhs_key FROM {TGT}.pin_district WHERE pin = ? AND is_primary", ["613001"])[0]["nfhs_key"]
near = _rows(cn, f"""
SELECT min({F_HAVERSINE}(?, ?, l.final_lat, l.final_lon)) AS km FROM {T_SERVICE} fs
JOIN {T_FAC_CLEAN} c ON c.unique_id = fs.unique_id AND NOT coalesce(c.is_shifted, false)
JOIN {T_FAC_LOC} l ON l.facility_id = fs.unique_id
WHERE fs.service = 'caesarean_section' AND fs.tier = 'Confirmed' AND l.final_lat IS NOT NULL
  AND coalesce(l.location_confidence, 'none') <> 'none'""", [pin["lat"], pin["lon"]])[0]["km"]
print(f"district {nk}; nearest Confirmed C-section {near:.0f} km")
h = district_hint(cn, "caesarean_section", pin["lat"], pin["lon"], nk)
print(f"{h['indicator']} = {h['value']:.1f}% (median of 706 districts {h['median_706']:.1f}%)")
for f in h["facilities"]:
    print(f"  {f['distance_km']:5.1f} km  {f['name']} ({f['city']})  {f['service']} {f['tier']}")
    quote = '"' + f["sentence"] + '"' if f["sentence"] else "(none naming maternity care)"
    print(f"           quote: {quote}")

district tamilnadu|thanjavur; nearest Confirmed C-section 108 km
births_delivered_by_csection_5y_pct = 51.4% (median of 706 districts 18.6%)
    1.2 km  Meenakshi Hospital (Thanjavur)  obstetrics_maternity Likely
           quote: "Offers obstetric and gynecologic procedures"
    1.4 km  Vinodhagan Memorial Hospital (Thanjavur)  obstetrics_maternity Likely
           quote: "Departments include Pulmonology, Gynecology, and Neurology"
    3.9 km  Srikamatchi Medical Centre (Thanjavur)  obstetrics_maternity Likely
           quote: (none naming maternity care)


The full app is deployed as the Databricks App **vf-locator** (FastAPI + React, Leaflet map), with its source kept outside this notebook; it runs exactly the ranking query and matcher above against the tables this notebook builds.